In [24]:
#Running baseline checks

#!/usr/bin/env python3
"""
ML baselines for OpenNeuro ds004584 (Cleaned FIF): PD vs Control

Models:
  - Linear Regression (treated as regressor + sigmoid/threshold)
  - Logistic Regression
  - SVM (RBF)
  - Random Forest
  - XGBoost (if installed; otherwise skipped with a clear message)

Protocol (leakage-safe):
✅ Subject-wise 5-fold CV (Stratified)
✅ Use ONLY first 2 minutes (120 s) from each recording
✅ Canonical EEG channel intersection (EEG-only)
✅ Windowing: 10s, 50% overlap
✅ Feature extraction per window (bandpower + simple stats)
✅ Aggregate to subject-level features (mean+std over windows) before ML
✅ Metrics per fold + overall: Acc, BAcc, Prec, Rec, F1, AUC, Confusion Matrix

Run:
  python ml_baselines_ds004584.py

Notes:
- XGBoost requires: pip install xgboost
"""

import os, glob, random, warnings
import numpy as np

import mne
from mne.time_frequency import psd_array_welch

from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------
# Reproducibility
# ----------------------------
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)

# ----------------------------
# Paths + labels
# ----------------------------
def collect_subject_files(root_dir):
    """
    Expect cleaned FIF structure:
      root_dir/sub-XXX/eeg/*.fif
    Returns list of (sub_id, filepath). If multiple FIF per subject, pick first sorted.
    """
    fif_files = sorted(glob.glob(os.path.join(root_dir, "sub-*", "eeg", "*.fif")))
    by_sub = {}
    for fp in fif_files:
        base = os.path.basename(fp)
        sid = base.split("_")[0]
        if not sid.startswith("sub-"):
            sid = os.path.basename(os.path.dirname(os.path.dirname(fp)))
        by_sub.setdefault(sid, []).append(fp)

    items = []
    for sid in sorted(by_sub.keys()):
        items.append((sid, sorted(by_sub[sid])[0]))
    return items


def load_labels_from_participants_tsv(root_dir):
    """
    ds004584 participants.tsv columns:
      ['participant_id', 'GROUP', 'ID', 'EEG', 'AGE', 'GENDER', 'MOCA', 'UPDRS', 'TYPE']
    Returns label_map: {sub-XXX: 0/1} where 1=PD, 0=Control
    """
    import pandas as pd

    tsv = os.path.join(label_dir, "participants.tsv")
    if not os.path.exists(tsv):
        raise FileNotFoundError(f"participants.tsv not found at: {tsv}")

    df = pd.read_csv(tsv, sep="\t")

    if "participant_id" not in df.columns:
        raise ValueError(f"participants.tsv missing 'participant_id'. Columns: {list(df.columns)}")
    if "GROUP" not in df.columns:
        raise ValueError(f"participants.tsv missing 'GROUP'. Columns: {list(df.columns)}")

    def map_label(x):
        s = str(x).strip().upper()
        if s == "PD":
            return 1
        if s in {"CT", "HC", "CONTROL", "HEALTHY", "CTRL", "C"}:
            return 0
        raise ValueError(f"Unknown GROUP value: {x!r}")

    label_map = {}
    for _, row in df.iterrows():
        pid = str(row["participant_id"]).strip()
        grp = row["GROUP"]
        label_map[pid] = map_label(grp)

    n = len(label_map)
    n_pd = sum(v == 1 for v in label_map.values())
    n_ctrl = n - n_pd
    print(f"[labels] subjects={n} | PD={n_pd} | Control={n_ctrl}")
    return label_map

# ----------------------------
# Canonical channel intersection (EEG-only)
# ----------------------------
def compute_channel_intersection(file_items):
    sub0, fp0 = file_items[0]
    raw0 = mne.io.read_raw_fif(fp0, preload=False, verbose="ERROR")
    raw0.pick("eeg")
    ch0 = [c for c in raw0.ch_names if c not in raw0.info["bads"]]
    inter = set(ch0)

    for sid, fp in tqdm(file_items, desc="Channel intersection", leave=False):
        raw = mne.io.read_raw_fif(fp, preload=False, verbose="ERROR")
        raw.pick("eeg")
        ch = set([c for c in raw.ch_names if c not in raw.info["bads"]])
        inter &= ch

    canonical = [c for c in ch0 if c in inter]
    if len(canonical) < 5:
        raise RuntimeError(f"Intersection too small ({len(canonical)}).")
    return canonical

# ----------------------------
# Preprocess + window (first 2 minutes)
# ----------------------------
def preprocess_and_window(
    fp, canonical_chs,
    crop_seconds=120.0,
    win_seconds=10.0,
    overlap=0.5,
    l_freq=1.0,
    h_freq=40.0,
    target_sfreq=250.0
):
    """
    Returns windows: [Nw, C, T] float32
    """
    raw = mne.io.read_raw_fif(fp, preload=True, verbose="ERROR")
    raw.pick("eeg")
    raw.pick_channels(canonical_chs, ordered=True)

    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design="firwin", verbose="ERROR")
    raw.resample(target_sfreq, npad="auto", verbose="ERROR")

    sfreq = float(raw.info["sfreq"])
    data = raw.get_data()  # [C, T]

    n_crop = int(crop_seconds * sfreq)
    if data.shape[1] < n_crop:
        n_crop = data.shape[1]
    data = data[:, :n_crop]

    # z-score per channel
    mean = data.mean(axis=1, keepdims=True)
    std = data.std(axis=1, keepdims=True) + 1e-8
    data = (data - mean) / std

    win_len = int(win_seconds * sfreq)
    step = int(win_len * (1.0 - overlap))
    if step <= 0 or data.shape[1] < win_len:
        return np.zeros((0, len(canonical_chs), win_len), dtype=np.float32)

    windows = []
    for start in range(0, data.shape[1] - win_len + 1, step):
        windows.append(data[:, start:start + win_len])

    if not windows:
        return np.zeros((0, len(canonical_chs), win_len), dtype=np.float32)

    return np.stack(windows, axis=0).astype(np.float32), sfreq

# ----------------------------
# Feature extraction
# ----------------------------
BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 40.0),
}

def window_features_bandpower(x_ct, sfreq):
    """
    x_ct: [C, T] (z-scored)
    Returns feature vector:
      - bandpowers per channel (C * nbands)
      - time-domain stats per channel: mean(|x|), std, kurtosis-like proxy (E[x^4]/(E[x^2]^2))
    """
    C, T = x_ct.shape

    # PSD
    psd, freqs = psd_array_welch(
        x_ct, sfreq=sfreq, fmin=1.0, fmax=40.0,
        n_fft=min(1024, T), n_overlap=0, verbose=False
    )  # psd: [C, F]

    feats = []
    # bandpower per channel
    for _, (f1, f2) in BANDS.items():
        idx = np.logical_and(freqs >= f1, freqs < f2)
        bp = np.trapz(psd[:, idx], freqs[idx], axis=1)  # [C]
        feats.append(bp)

    bp_all = np.stack(feats, axis=1)  # [C, nbands]
    bp_all = np.log(bp_all + 1e-12)   # stabilize
    feats_flat = bp_all.reshape(-1)   # [C*nbands]

    # time-domain per channel
    abs_mean = np.mean(np.abs(x_ct), axis=1)
    std = np.std(x_ct, axis=1)
    e2 = np.mean(x_ct**2, axis=1) + 1e-12
    e4 = np.mean(x_ct**4, axis=1)
    kurt_proxy = e4 / (e2**2)

    feats_td = np.concatenate([abs_mean, std, kurt_proxy], axis=0)  # [3C]

    return np.concatenate([feats_flat, feats_td], axis=0).astype(np.float32)

def subject_features_from_windows(Xw, sfreq):
    """
    Xw: [Nw, C, T]
    Compute per-window features, then aggregate to subject vector:
      concat(mean(features), std(features))
    """
    feats = []
    for i in range(Xw.shape[0]):
        feats.append(window_features_bandpower(Xw[i], sfreq))
    Fw = np.stack(feats, axis=0)  # [Nw, D]
    mu = Fw.mean(axis=0)
    sd = Fw.std(axis=0)
    return np.concatenate([mu, sd], axis=0).astype(np.float32)

# ----------------------------
# Build dataset (subject-level)
# ----------------------------
def build_subject_matrix(file_items, label_map, canonical_chs,
                         crop_seconds=120.0, win_seconds=10.0, overlap=0.5,
                         l_freq=1.0, h_freq=40.0, target_sfreq=250.0):
    """
    Returns:
      subj_ids: list[str]
      X: [Nsubj, D]
      y: [Nsubj]
    """
    subj_ids, X_list, y_list = [], [], []
    for sid, fp in tqdm(file_items, desc="Feature build (subjects)", leave=False):
        if sid not in label_map:
            continue
        Xw, sfreq = preprocess_and_window(
            fp, canonical_chs,
            crop_seconds=crop_seconds,
            win_seconds=win_seconds,
            overlap=overlap,
            l_freq=l_freq,
            h_freq=h_freq,
            target_sfreq=target_sfreq
        )
        if Xw.shape[0] == 0:
            continue

        x_subj = subject_features_from_windows(Xw, sfreq)
        subj_ids.append(sid)
        X_list.append(x_subj)
        y_list.append(int(label_map[sid]))

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=int)
    return subj_ids, X, y

# ----------------------------
# Models
# ----------------------------
def get_models(seed=42):
    models = {}

    # Logistic Regression
    models["LogReg"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

    # SVM (RBF) with probability for AUC
    models["SVM-RBF"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True
        ))
    ])

    # Random Forest
    models["RF"] = RandomForestClassifier(
        n_estimators=500,
        random_state=seed,
        class_weight="balanced_subsample",
        n_jobs=-1,
        max_depth=None
    )

    # Linear Regression baseline (regressor -> sigmoid -> threshold)
    models["LinReg"] = Pipeline([
        ("scaler", StandardScaler()),
        ("reg", LinearRegression())
    ])

    # XGBoost (optional)
    try:
        import xgboost as xgb
        models["XGBoost"] = xgb.XGBClassifier(
            n_estimators=800,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1
        )
    except Exception as e:
        models["XGBoost"] = None

    return models

# ----------------------------
# Evaluation helpers
# ----------------------------
def prob_positive(model, X):
    """
    Return p(y=1) for classifiers/regressors.
    """
    name = model.__class__.__name__
    # Pipeline case
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return 1.0 / (1.0 + np.exp(-s))

    # Pipeline wrappers
    if hasattr(model, "named_steps"):
        # Logistic/SVM pipeline has predict_proba
        if "clf" in model.named_steps and hasattr(model.named_steps["clf"], "predict_proba"):
            return model.predict_proba(X)[:, 1]
        # Linear regression pipeline
        if "reg" in model.named_steps:
            yhat = model.predict(X)
            return 1.0 / (1.0 + np.exp(-yhat))

    # Fallback
    yhat = model.predict(X)
    return 1.0 / (1.0 + np.exp(-yhat))


def eval_from_probs(y_true, p_pos, thr=0.5):
    y_pred = (p_pos >= thr).astype(int)
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1v = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    aucv = roc_auc_score(y_true, p_pos) if len(np.unique(y_true)) == 2 else np.nan
    return acc, bacc, prec, rec, f1v, aucv, cm

# ----------------------------
# Main CV
# ----------------------------
def run_ml_baselines(
    root_dir,
    seed=42,
    n_splits=5,
):
    seed_all(seed)

    file_items = collect_subject_files(root_dir)
    if len(file_items) == 0:
        raise RuntimeError("No FIF files found. Check root_dir.")

    label_map = load_labels_from_participants_tsv(root_dir)

    # keep only labeled
    file_items = [(sid, fp) for sid, fp in file_items if sid in label_map]

    canonical_chs = compute_channel_intersection(file_items)
    print(f"[channels] canonical EEG intersection = {len(canonical_chs)}")

    subj_ids, X, y = build_subject_matrix(file_items, label_map, canonical_chs)
    print(f"[features] subjects used = {len(subj_ids)} | X shape = {X.shape} | PD={int(y.sum())} | Control={int((y==0).sum())}")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = get_models(seed=seed)

    # report which models will run
    print("\nModels:")
    for k, v in models.items():
        if v is None:
            print(f"  - {k}: SKIPPED (xgboost not installed)")
        else:
            print(f"  - {k}: OK")

    results = {k: [] for k in models.keys() if models[k] is not None}

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        print(f"\n================ Fold {fold}/{n_splits} ================")
        for name, model in models.items():
            if model is None:
                continue

            # fit
            model.fit(Xtr, ytr)

            # probs + metrics
            p_pos = prob_positive(model, Xte)
            acc, bacc, prec, rec, f1v, aucv, cm = eval_from_probs(yte, p_pos, thr=0.5)

            print(f"[{name:8s}] Acc={acc:.4f} BAcc={bacc:.4f} Prec={prec:.4f} Rec={rec:.4f} F1={f1v:.4f} AUC={aucv:.4f}")
            print(f"[{name:8s}] CM:\n{cm}")

            results[name].append((acc, bacc, prec, rec, f1v, aucv))

    # Summary
    print("\n================ OVERALL (mean ± std) ================")
    for name, vals in results.items():
        arr = np.array(vals, dtype=float)  # [folds, 6]
        m = arr.mean(axis=0)
        s = arr.std(axis=0)
        print(f"\n{name}")
        print(f"  Acc : {m[0]:.4f} ± {s[0]:.4f}")
        print(f"  BAcc: {m[1]:.4f} ± {s[1]:.4f}")
        print(f"  Prec: {m[2]:.4f} ± {s[2]:.4f}")
        print(f"  Rec : {m[3]:.4f} ± {s[3]:.4f}")
        print(f"  F1  : {m[4]:.4f} ± {s[4]:.4f}")
        print(f"  AUC : {m[5]:.4f} ± {s[5]:.4f}")

# ----------------------------
# Run
# ----------------------------
if __name__ == "__main__":
    root_dir = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
    label_dir = "/home/varun/Desktop/Parkinsons_Resting_State"
    run_ml_baselines(root_dir=root_dir, seed=42, n_splits=5)


[labels] subjects=149 | PD=100 | Control=49


Channel intersection:   0%|          | 0/149 [00:00<?, ?it/s]

[channels] canonical EEG intersection = 60


Feature build (subjects):   0%|          | 0/149 [00:00<?, ?it/s]

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy functi

In [30]:
#Computing windows for DL baselines

#!/usr/bin/env python3
"""
STEP 1: Precompute EEG windows for ds004584 (Cleaned FIF)

Output:
  cache_dir/sub-XXX_windows.npz
"""

import os, glob
import numpy as np
import mne
from tqdm import tqdm

ROOT_DIR = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
CACHE_DIR = os.path.join(ROOT_DIR, "precomputed_windows")
os.makedirs(CACHE_DIR, exist_ok=True)

CROP_SECONDS = 120.0
WIN_SECONDS = 10.0
OVERLAP = 0.5
L_FREQ, H_FREQ = 1.0, 40.0
TARGET_SFREQ = 128.0

def collect_subject_files(root_dir):
    fif_files = sorted(glob.glob(os.path.join(root_dir, "sub-*", "eeg", "*.fif")))
    by_sub = {}
    for fp in fif_files:
        sid = os.path.basename(fp).split("_")[0]
        by_sub.setdefault(sid, []).append(fp)
    return [(sid, sorted(fps)[0]) for sid, fps in by_sub.items()]

def compute_canonical_channels(file_items):
    sid0, fp0 = file_items[0]
    raw0 = mne.io.read_raw_fif(fp0, preload=False, verbose="ERROR")
    raw0.pick("eeg")
    chs = set(raw0.ch_names)

    for _, fp in file_items:
        raw = mne.io.read_raw_fif(fp, preload=False, verbose="ERROR")
        raw.pick("eeg")
        chs &= set(raw.ch_names)

    return sorted(list(chs))

def preprocess_and_window(fp, canonical_chs):
    raw = mne.io.read_raw_fif(fp, preload=True, verbose="ERROR")
    raw.pick("eeg")
    raw.pick(canonical_chs)

    raw.filter(L_FREQ, H_FREQ, verbose="ERROR")
    raw.resample(TARGET_SFREQ, verbose="ERROR")

    data = raw.get_data()
    sfreq = raw.info["sfreq"]

    n_crop = int(CROP_SECONDS * sfreq)
    data = data[:, :n_crop]

    data = (data - data.mean(1, keepdims=True)) / (data.std(1, keepdims=True) + 1e-8)

    win_len = int(WIN_SECONDS * sfreq)
    step = int(win_len * (1 - OVERLAP))

    windows = []
    for s in range(0, data.shape[1] - win_len + 1, step):
        windows.append(data[:, s:s+win_len])

    return np.stack(windows).astype(np.float32)

def main():
    file_items = collect_subject_files(ROOT_DIR)
    canonical_chs = compute_canonical_channels(file_items)

    print("Canonical channels:", len(canonical_chs))

    for sid, fp in tqdm(file_items, desc="Precomputing"):
        out_fp = os.path.join(CACHE_DIR, f"{sid}_windows.npz")
        if os.path.exists(out_fp):
            continue

        X = preprocess_and_window(fp, canonical_chs)
        np.savez_compressed(out_fp, X=X)

    np.save(os.path.join(CACHE_DIR, "canonical_channels.npy"), canonical_chs)
    print("Precomputation complete.")

if __name__ == "__main__":
    main()


Canonical channels: 60


Precomputing: 100%|███████████████████████| 149/149 [00:00<00:00, 175498.82it/s]

Precomputation complete.


In [28]:
#!/usr/bin/env python3
"""
STEP 2: Fast DL baselines on precomputed windows
Models: EEGNet, Inception, EEGConformer
"""

import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
from tqdm.auto import tqdm

ROOT_DIR = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
label_dir = "/home/varun/Desktop/Parkinsons_Resting_State"
CACHE_DIR = os.path.join(ROOT_DIR, "precomputed_windows")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_all(SEED)

# -------------------------
# Labels
# -------------------------
def load_labels():
    df = pd.read_csv(os.path.join(label_dir, "participants.tsv"), sep="\t")
    def map_label(x):
        return 1 if str(x).upper() == "PD" else 0
    return dict(zip(df.participant_id, df.GROUP.map(map_label)))

LABELS = load_labels()

# -------------------------
# Dataset
# -------------------------
class WindowDataset(Dataset):
    def __init__(self, subjects):
        self.samples = []
        for sid in subjects:
            npz = np.load(os.path.join(CACHE_DIR, f"{sid}_windows.npz"))
            X = npz["X"]
            y = LABELS[sid]
            for i in range(X.shape[0]):
                self.samples.append((X[i], y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.tensor(y)

# -------------------------
# EEGNet (baseline)
# -------------------------
class EEGNet(nn.Module):
    def __init__(self, C, T):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, (1, 64), padding=(0,32), bias=False)
        self.conv2 = nn.Conv2d(8, 16, (C,1), groups=8, bias=False)
        self.bn = nn.BatchNorm2d(16)
        self.fc = nn.Linear(16, 2)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.bn(x)
        x = F.elu(x)
        x = x.mean(-1).squeeze(-1)
        return self.fc(x)

# -------------------------
# Train + Eval
# -------------------------
def run_cv():
    subjects = sorted([f.split("_")[0] for f in os.listdir(CACHE_DIR) if f.endswith(".npz")])
    y = np.array([LABELS[s] for s in subjects])

    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)

    accs, aucs = [], []

    for fold, (tr, te) in enumerate(skf.split(subjects, y), 1):
        tr_sub = [subjects[i] for i in tr]
        te_sub = [subjects[i] for i in te]

        train_ds = WindowDataset(tr_sub)
        test_ds  = WindowDataset(te_sub)

        train_ld = DataLoader(train_ds, batch_size=64, shuffle=True)
        test_ld  = DataLoader(test_ds, batch_size=64)

        C, T = train_ds[0][0].shape
        model = EEGNet(C, T).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()

        for ep in range(10):
            model.train()
            for xb, yb in train_ld:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()

        # eval
        model.eval()
        probs, yt = [], []
        with torch.no_grad():
            for xb, yb in test_ld:
                p = torch.softmax(model(xb.to(DEVICE)), 1)[:,1]
                probs.extend(p.cpu().numpy())
                yt.extend(yb.numpy())

        accs.append(accuracy_score(yt, np.array(probs)>=0.5))
        aucs.append(roc_auc_score(yt, probs))

        print(f"Fold {fold}: Acc={accs[-1]:.3f} AUC={aucs[-1]:.3f}")

    print("\nFINAL:")
    print(f"Acc: {np.mean(accs):.3f} ± {np.std(accs):.3f}")
    print(f"AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

if __name__ == "__main__":
    run_cv()


Fold 1: Acc=0.735 AUC=0.840
Fold 2: Acc=0.616 AUC=0.565
Fold 3: Acc=0.709 AUC=0.757
Fold 4: Acc=0.652 AUC=0.620
Fold 5: Acc=0.732 AUC=0.666

FINAL:
Acc: 0.689 ± 0.047
AUC: 0.690 ± 0.098


In [2]:
# ============================================================
# OpenNeuro ds004584 (Cleaned FIF): PD vs Control
# JUPYTER NOTEBOOK — END-TO-END (PRECOMPUTE ONCE + TRAIN 3 DL BASELINES)
#
# ✅ Uses FIRST 2 minutes (120s) of every recording
# ✅ Canonical EEG channel intersection (EEG-only)
# ✅ 10s windows, 50% overlap
# ✅ 5-fold subject-wise CV + inner val split (no leakage)
# ✅ Models: EEGNet, InceptionNet, EEGConformer (lite)
# ✅ Per-epoch prints:
#    - Window-level: Acc, BAcc, Prec, Rec, F1, CM, AUC
#    - Subject-level: Acc, BAcc, Prec, Rec, F1, CM, AUC
# ✅ No argparse, notebook-friendly
# ✅ Fast training: MNE only during precompute; training uses .npy cache
#
# Set paths below and run cells top-to-bottom.
# ============================================================

import os, glob, random, math
from collections import OrderedDict, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)
from tqdm.auto import tqdm

# ----------------------------
# USER CONFIG (EDIT THESE)
# ----------------------------
ROOT_DIR  = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
label_dir = "/home/varun/Desktop/Parkinsons_Resting_State"
CACHE_DIR = os.path.join(ROOT_DIR, "precomputed_2min_win10_overlap50")       # cache location

SEED = 42
EPOCHS = 30
BATCH_SIZE = 64
NUM_WORKERS = 0   # keep 0 to avoid multiprocess shutdown issues you saw

# Preprocess/window params
CROP_SECONDS = 120.0
WIN_SECONDS  = 10.0
OVERLAP      = 0.5
L_FREQ       = 1.0
H_FREQ       = 40.0
TARGET_SFREQ = 128.0   # faster than 250; keeps band info

# Optim
LR = 1e-3
WEIGHT_DECAY = 1e-4

MODELS_TO_RUN = ["eegnet", "inception", "eegconformer"]   # or subset
# ----------------------------


# ----------------------------
# Reproducibility
# ----------------------------
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


# ============================================================
# 1) PRECOMPUTE STAGE (MNE used only here)
# ============================================================

def load_labels_from_participants_tsv(root_dir):
    """
    participants.tsv columns:
      ['participant_id','GROUP',...]
    label: PD=1, Control=0
    """
    tsv = os.path.join(label_dir, "participants.tsv")
    if not os.path.exists(tsv):
        raise FileNotFoundError(f"participants.tsv not found at: {tsv}")
    df = pd.read_csv(tsv, sep="\t")

    if "participant_id" not in df.columns or "GROUP" not in df.columns:
        raise ValueError(f"participants.tsv missing required columns. Found: {list(df.columns)}")

    def map_label(x):
        s = str(x).strip().upper()
        if s == "PD":
            return 1
        if s in {"CT","HC","CONTROL","HEALTHY","CTRL","C"}:
            return 0
        raise ValueError(f"Unknown GROUP: {x!r}")

    label_map = dict(zip(df["participant_id"].astype(str), df["GROUP"].map(map_label)))
    return label_map


def collect_subject_fif_files(root_dir):
    """
    Expect:
      root_dir/sub-XXX/eeg/*.fif
    Returns list of (sid, fp) choosing the first fif per subject (sorted).
    """
    fif_files = sorted(glob.glob(os.path.join(root_dir, "sub-*", "eeg", "*.fif")))
    if len(fif_files) == 0:
        raise RuntimeError(f"No .fif files found under: {root_dir}/sub-*/eeg/*.fif")

    by_sub = defaultdict(list)
    for fp in fif_files:
        # prefer parsing from filename "sub-XXX_..."
        base = os.path.basename(fp)
        sid = base.split("_")[0]
        if not sid.startswith("sub-"):
            # fallback: subdir name
            sid = os.path.basename(os.path.dirname(os.path.dirname(fp)))
        by_sub[sid].append(fp)

    items = [(sid, sorted(fps)[0]) for sid, fps in sorted(by_sub.items())]
    return items


def compute_canonical_channels_mne(file_items):
    """
    EEG-only intersection, order from first file.
    """
    import mne
    mne.set_log_level("ERROR")

    sid0, fp0 = file_items[0]
    raw0 = mne.io.read_raw_fif(fp0, preload=False, verbose="ERROR")
    raw0.pick("eeg")
    ch0 = [c for c in raw0.ch_names if c not in raw0.info["bads"]]
    inter = set(ch0)

    for sid, fp in tqdm(file_items, desc="Canonical channel intersection", leave=False):
        raw = mne.io.read_raw_fif(fp, preload=False, verbose="ERROR")
        raw.pick("eeg")
        ch = set([c for c in raw.ch_names if c not in raw.info["bads"]])
        inter &= ch

    canonical = [c for c in ch0 if c in inter]
    if len(canonical) < 5:
        raise RuntimeError(f"Intersection too small ({len(canonical)}).")
    return canonical


def preprocess_and_window_mne(
    fp, canonical_chs,
    crop_seconds=120.0,
    win_seconds=10.0,
    overlap=0.5,
    l_freq=1.0, h_freq=40.0,
    target_sfreq=128.0,
):
    """
    Returns windows: float32 [Nw, C, T]
    """
    import mne
    mne.set_log_level("ERROR")

    raw = mne.io.read_raw_fif(fp, preload=True, verbose="ERROR")
    raw.pick("eeg")            # EEG-only
    raw.pick(canonical_chs)    # new API; avoids legacy pick_channels warnings

    raw.filter(l_freq, h_freq, fir_design="firwin", verbose="ERROR")
    raw.resample(target_sfreq, npad="auto", verbose="ERROR")

    data = raw.get_data()      # [C, T]
    sfreq = float(raw.info["sfreq"])

    # Crop FIRST 120 seconds
    n_crop = int(crop_seconds * sfreq)
    if data.shape[1] < n_crop:
        n_crop = data.shape[1]
    data = data[:, :n_crop]

    # z-score per channel
    data = (data - data.mean(axis=1, keepdims=True)) / (data.std(axis=1, keepdims=True) + 1e-8)

    win_len = int(win_seconds * sfreq)
    step = int(win_len * (1.0 - overlap))
    if step <= 0 or data.shape[1] < win_len:
        return np.zeros((0, len(canonical_chs), win_len), dtype=np.float32)

    windows = []
    for start in range(0, data.shape[1] - win_len + 1, step):
        windows.append(data[:, start:start + win_len])

    if not windows:
        return np.zeros((0, len(canonical_chs), win_len), dtype=np.float32)

    return np.stack(windows, axis=0).astype(np.float32)


def precompute_cache(root_dir, cache_dir):
    os.makedirs(cache_dir, exist_ok=True)

    label_map = load_labels_from_participants_tsv(label_dir)
    file_items = collect_subject_fif_files(root_dir)
    file_items = [(sid, fp) for sid, fp in file_items if sid in label_map]

    if len(file_items) == 0:
        raise RuntimeError("No FIF files with labels found after filtering by participants.tsv.")

    canonical_chs = compute_canonical_channels_mne(file_items)

    # write canonical list
    with open(os.path.join(cache_dir, "canonical_channels.txt"), "w") as f:
        for ch in canonical_chs:
            f.write(ch + "\n")

    manifest_rows = []
    for sid, fp in tqdm(file_items, desc="Precomputing windows"):
        out_npy = os.path.join(cache_dir, f"{sid}.npy")
        if os.path.exists(out_npy):
            X = np.load(out_npy)
            n_w = int(X.shape[0])
            manifest_rows.append((sid, int(label_map[sid]), n_w, fp))
            continue

        X = preprocess_and_window_mne(
            fp, canonical_chs,
            crop_seconds=CROP_SECONDS,
            win_seconds=WIN_SECONDS,
            overlap=OVERLAP,
            l_freq=L_FREQ, h_freq=H_FREQ,
            target_sfreq=TARGET_SFREQ,
        )
        np.save(out_npy, X)
        manifest_rows.append((sid, int(label_map[sid]), int(X.shape[0]), fp))

    manifest = pd.DataFrame(manifest_rows, columns=["sid", "label", "n_windows", "source_path"])
    manifest = manifest[manifest["n_windows"] > 0].copy()
    manifest.to_csv(os.path.join(cache_dir, "manifest.csv"), index=False)

    n_pd = int((manifest["label"] == 1).sum())
    n_ct = int((manifest["label"] == 0).sum())
    print(f"[precompute done] subjects={len(manifest)} | PD={n_pd} | Control={n_ct}")
    print(f"[cache_dir] {cache_dir}")
    print(f"[canonical channels] {len(canonical_chs)}")

# Run precompute (skip if already cached)
if not os.path.exists(os.path.join(CACHE_DIR, "manifest.csv")):
    precompute_cache(ROOT_DIR, CACHE_DIR)
else:
    print("[cache] manifest already exists, skipping precompute:", os.path.join(CACHE_DIR, "manifest.csv"))


# ============================================================
# 2) TRAINING STAGE (NO MNE)
# ============================================================

def compute_metrics_binary(y_true, y_pred, p_pos):
    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1v  = f1_score(y_true, y_pred, zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, p_pos)
        aucv = auc(fpr, tpr)
    else:
        aucv = float("nan")
    return acc, bacc, prec, rec, f1v, cm, aucv

def print_cm(cm, prefix=""):
    print(prefix + "Confusion Matrix (rows=true, cols=pred):")
    print(prefix + f"[[{cm[0,0]:4d} {cm[0,1]:4d}]")
    print(prefix + f" [{cm[1,0]:4d} {cm[1,1]:4d}]]")


class WindowIndexDataset(Dataset):
    """
    Window-level dataset built from precomputed per-subject .npy windows.
    Small LRU cache to avoid disk I/O per sample.
    """
    def __init__(self, manifest_df, cache_dir, cache_size_subjects=12):
        self.cache_dir = cache_dir
        self.cache_size = cache_size_subjects
        self.cache = OrderedDict()  # sid -> np.ndarray [Nw,C,T]

        self.index = []
        for r in manifest_df.itertuples(index=False):
            sid, label, n_w = r.sid, int(r.label), int(r.n_windows)
            for w in range(n_w):
                self.index.append((sid, w, label))

        if len(self.index) == 0:
            raise RuntimeError("No windows in dataset index. Check precompute outputs.")

        sid0, _, _ = self.index[0]
        X0 = self._get_subject_windows(sid0)
        self.n_chans = int(X0.shape[1])
        self.win_samples = int(X0.shape[2])

    def _get_subject_windows(self, sid):
        if sid in self.cache:
            self.cache.move_to_end(sid)
            return self.cache[sid]
        fp = os.path.join(self.cache_dir, f"{sid}.npy")
        X = np.load(fp)
        self.cache[sid] = X
        self.cache.move_to_end(sid)
        while len(self.cache) > self.cache_size:
            self.cache.popitem(last=False)
        return X

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sid, w, label = self.index[idx]
        X = self._get_subject_windows(sid)
        x = X[w]  # [C,T]
        return torch.from_numpy(x).float(), torch.tensor(label).long(), sid

def collate_fn(batch):
    xs, ys, sids = zip(*batch)
    return torch.stack(xs, 0), torch.stack(ys, 0), np.array(sids)


# ----------------------------
# Models: EEGNet, Inception, EEGConformer-lite
# ----------------------------
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

class EEGNet(nn.Module):
    def __init__(self, n_chans, n_classes=2, F1=8, D=2, F2=16, kernel_len=64, dropout=0.25):
        super().__init__()
        self.conv_temporal = nn.Conv2d(1, F1, (1, kernel_len), padding=(0, kernel_len//2), bias=False)
        self.bn1 = nn.BatchNorm2d(F1)

        self.conv_spatial = nn.Conv2d(F1, F1*D, (n_chans, 1), groups=F1, bias=False)
        self.bn2 = nn.BatchNorm2d(F1*D)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(dropout)

        self.sep_depth = nn.Conv2d(F1*D, F1*D, (1, 16), padding=(0, 8), groups=F1*D, bias=False)
        self.sep_point = nn.Conv2d(F1*D, F2, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.drop2 = nn.Dropout(dropout)

        self.classifier = nn.Linear(F2, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # [B,1,C,T]
        x = self.bn1(self.conv_temporal(x))
        x = self.bn2(self.conv_spatial(x))
        x = F.elu(x)
        x = self.drop1(self.pool1(x))
        x = self.sep_point(self.sep_depth(x))
        x = self.bn3(x)
        x = F.elu(x)
        x = self.drop2(self.pool2(x))
        x = x.mean(dim=-1).squeeze(-1)  # [B,F2]
        return self.classifier(x)

class InceptionBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, ks=(9, 19, 39), bottleneck=32, dropout=0.15):
        super().__init__()
        self.use_bottleneck = (bottleneck is not None) and (bottleneck < in_ch)
        bch = bottleneck if self.use_bottleneck else in_ch
        self.bott = nn.Conv1d(in_ch, bch, kernel_size=1, bias=False) if self.use_bottleneck else nn.Identity()
        self.branches = nn.ModuleList([nn.Conv1d(bch, out_ch, kernel_size=k, padding=k//2, bias=False) for k in ks])
        self.pool_branch = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_ch, out_ch, kernel_size=1, bias=False)
        )
        self.bn = nn.BatchNorm1d(out_ch * (len(ks) + 1))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x0 = self.bott(x)
        outs = [b(x0) for b in self.branches]
        outs.append(self.pool_branch(x))
        y = torch.cat(outs, dim=1)
        y = self.bn(y)
        y = F.relu(y)
        y = self.drop(y)
        return y

class InceptionNetEEG(nn.Module):
    def __init__(self, n_chans, n_classes=2, base=16, depth=3, dropout=0.15):
        super().__init__()
        blocks = []
        in_ch = n_chans
        for _ in range(depth):
            blocks.append(InceptionBlock1D(in_ch, out_ch=base, dropout=dropout))
            in_ch = base * 4
        self.backbone = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.LayerNorm(in_ch),
            nn.Dropout(dropout),
            nn.Linear(in_ch, n_classes),
        )

    def forward(self, x):
        y = self.backbone(x)
        return self.head(y)

class ConformerBlock1D(nn.Module):
    def __init__(self, d_model=96, nhead=4, ff_mult=2, conv_k=31, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ff1 = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

        self.ln3 = nn.LayerNorm(d_model)
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=conv_k, padding=conv_k//2, groups=d_model, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, kernel_size=1, bias=False),
            nn.Dropout(dropout),
        )

        self.ln4 = nn.LayerNorm(d_model)
        self.ff2 = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(a)
        x = x + self.ff1(self.ln2(x))
        h = self.ln3(x).transpose(1, 2)  # [B,d,N]
        h = self.conv(h).transpose(1, 2) # [B,N,d]
        x = x + h
        x = x + self.ff2(self.ln4(x))
        return x

class EEGConformerLite(nn.Module):
    def __init__(self, n_chans, n_classes=2, d_model=96, nhead=4, depth=2, patch_size=16, dropout=0.1):
        super().__init__()
        self.proj = nn.Conv1d(n_chans, d_model, kernel_size=patch_size, stride=patch_size, bias=False)
        self.pos = nn.Parameter(torch.randn(1, 512, d_model) * 0.01)
        self.blocks = nn.ModuleList([ConformerBlock1D(d_model, nhead, dropout=dropout) for _ in range(depth)])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x):
        z = self.proj(x)          # [B,d,N]
        z = z.transpose(1, 2)     # [B,N,d]
        N = z.shape[1]
        z = z + self.pos[:, :N, :]
        for blk in self.blocks:
            z = blk(z)
        z = z.mean(dim=1)
        return self.head(z)

def build_model(name, n_chans):
    n = name.lower()
    if n == "eegnet":
        return EEGNet(n_chans=n_chans)
    if n in {"inception", "inceptionnet", "inceptiontime"}:
        return InceptionNetEEG(n_chans=n_chans)
    if n in {"eegconformer", "conformer"}:
        return EEGConformerLite(n_chans=n_chans)
    raise ValueError(f"Unknown model: {name}")


# ----------------------------
# Evaluation (window + subject)
# ----------------------------
@torch.no_grad()
def collect_window_preds(model, loader, device):
    model.eval()
    y_true, y_pred, p_pos, sids = [], [], [], []
    for xb, yb, sidb in loader:
        xb = xb.to(device)
        logits = model(xb)
        prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        pred = (prob >= 0.5).astype(int)
        y_true.append(yb.numpy().astype(int))
        y_pred.append(pred)
        p_pos.append(prob)
        sids.append(sidb)
    return np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(p_pos), np.concatenate(sids)

@torch.no_grad()
def eval_windowwise(model, loader, device):
    y_true, y_pred, p_pos, _ = collect_window_preds(model, loader, device)
    return compute_metrics_binary(y_true, y_pred, p_pos)

@torch.no_grad()
def eval_subjectwise(model, loader, device):
    y_true_w, _y_pred_w, p_pos_w, sids = collect_window_preds(model, loader, device)
    subj_probs = defaultdict(list)
    subj_true = {}
    for y, p, sid in zip(y_true_w, p_pos_w, sids):
        subj_probs[sid].append(float(p))
        subj_true.setdefault(sid, int(y))
    y_true, y_pred, p_pos = [], [], []
    for sid in subj_probs.keys():
        pm = float(np.mean(subj_probs[sid]))
        y_true.append(subj_true[sid])
        p_pos.append(pm)
        y_pred.append(int(pm >= 0.5))
    return compute_metrics_binary(np.array(y_true), np.array(y_pred), np.array(p_pos))


# ----------------------------
# Train one fold
# ----------------------------
def train_one_fold(model, train_loader, val_loader, device, epochs=30, lr=1e-3, weight_decay=1e-4):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()

    best_auc = -1.0
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)
        for xb, yb, _ in pbar:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
            pbar.set_postfix(loss=float(loss.item()))

        # per-epoch metrics (window + subject)
        w_acc, w_bacc, w_prec, w_rec, w_f1, w_cm, w_auc = eval_windowwise(model, val_loader, device)
        s_acc, s_bacc, s_prec, s_rec, s_f1, s_cm, s_auc = eval_subjectwise(model, val_loader, device)

        print(
            f"[Epoch {ep:03d}] "
            f"VAL Window:  Acc={w_acc:.4f} BAcc={w_bacc:.4f} Prec={w_prec:.4f} Rec={w_rec:.4f} F1={w_f1:.4f} AUC={w_auc:.4f} | "
            f"VAL Subject: Acc={s_acc:.4f} BAcc={s_bacc:.4f} Prec={s_prec:.4f} Rec={s_rec:.4f} F1={s_f1:.4f} AUC={s_auc:.4f}"
        )

        if (not math.isnan(s_auc)) and (s_auc > best_auc + 1e-6):
            best_auc = s_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


# ----------------------------
# 5-fold subject-wise CV (inner val split)
# ----------------------------
def run_cv_for_model(model_name, cache_dir, epochs=30, batch_size=64, seed=42):
    seed_all(seed)
    device = DEVICE

    manifest_fp = os.path.join(cache_dir, "manifest.csv")
    if not os.path.exists(manifest_fp):
        raise FileNotFoundError(f"manifest.csv not found: {manifest_fp}")

    df = pd.read_csv(manifest_fp)
    df = df[df["n_windows"] > 0].copy()
    df = df.sort_values("sid").reset_index(drop=True)

    sids = df["sid"].values
    y = df["label"].values.astype(int)

    print(f"\n[data] subjects={len(df)} | PD={(y==1).sum()} | Control={(y==0).sum()}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_rows = []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(sids, y), start=1):
        df_trainval = df.iloc[tr_idx].reset_index(drop=True)
        df_test = df.iloc[te_idx].reset_index(drop=True)

        # inner subject-wise train/val split
        tr_sids, va_sids = train_test_split(
            df_trainval["sid"].values,
            test_size=0.2,
            stratify=df_trainval["label"].values,
            random_state=seed
        )
        df_train = df_trainval[df_trainval["sid"].isin(tr_sids)].reset_index(drop=True)
        df_val = df_trainval[df_trainval["sid"].isin(va_sids)].reset_index(drop=True)

        train_ds = WindowIndexDataset(df_train, cache_dir, cache_size_subjects=12)
        val_ds   = WindowIndexDataset(df_val, cache_dir, cache_size_subjects=12)
        test_ds  = WindowIndexDataset(df_test, cache_dir, cache_size_subjects=12)

        train_ld = DataLoader(
            train_ds, batch_size=batch_size, shuffle=True,
            num_workers=NUM_WORKERS, collate_fn=collate_fn,
            pin_memory=(device.type == "cuda"), persistent_workers=(NUM_WORKERS > 0)
        )
        val_ld = DataLoader(
            val_ds, batch_size=batch_size, shuffle=False,
            num_workers=NUM_WORKERS, collate_fn=collate_fn,
            pin_memory=(device.type == "cuda"), persistent_workers=(NUM_WORKERS > 0)
        )
        test_ld = DataLoader(
            test_ds, batch_size=batch_size, shuffle=False,
            num_workers=NUM_WORKERS, collate_fn=collate_fn,
            pin_memory=(device.type == "cuda"), persistent_workers=(NUM_WORKERS > 0)
        )

        model = build_model(model_name, n_chans=train_ds.n_chans)
        pcount = count_params(model)

        print(f"\n================ FOLD {fold}/5 | {model_name.upper()} ================")
        print(f"Windows: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
        print(f"Input: C={train_ds.n_chans} T={train_ds.win_samples} | Params={pcount:,}")

        # train using validation loader (NOT test)
        model = train_one_fold(model, train_ld, val_ld, device, epochs=epochs, lr=LR, weight_decay=WEIGHT_DECAY)

        # final test metrics
        w_acc, w_bacc, w_prec, w_rec, w_f1, w_cm, w_auc = eval_windowwise(model, test_ld, device)
        s_acc, s_bacc, s_prec, s_rec, s_f1, s_cm, s_auc = eval_subjectwise(model, test_ld, device)

        print(f"\n[FOLD {fold} FINAL] WINDOW : Acc={w_acc:.4f} BAcc={w_bacc:.4f} Prec={w_prec:.4f} Rec={w_rec:.4f} F1={w_f1:.4f} AUC={w_auc:.4f}")
        print_cm(w_cm, prefix="  ")
        print(f"[FOLD {fold} FINAL] SUBJECT: Acc={s_acc:.4f} BAcc={s_bacc:.4f} Prec={s_prec:.4f} Rec={s_rec:.4f} F1={s_f1:.4f} AUC={s_auc:.4f}")
        print_cm(s_cm, prefix="  ")

        fold_rows.append({
            "fold": fold,
            "win_acc": w_acc, "win_bacc": w_bacc, "win_prec": w_prec, "win_rec": w_rec, "win_f1": w_f1, "win_auc": w_auc,
            "subj_acc": s_acc, "subj_bacc": s_bacc, "subj_prec": s_prec, "subj_rec": s_rec, "subj_f1": s_f1, "subj_auc": s_auc,
        })

        del model, train_ds, val_ds, test_ds, train_ld, val_ld, test_ld
        torch.cuda.empty_cache()

    res = pd.DataFrame(fold_rows)
    print(f"\n================ OVERALL 5-FOLD SUMMARY | {model_name.upper()} (mean ± std) ================")
    for col in ["win_acc","win_bacc","win_prec","win_rec","win_f1","win_auc",
                "subj_acc","subj_bacc","subj_prec","subj_rec","subj_f1","subj_auc"]:
        vals = res[col].values.astype(float)
        print(f"{col:>9s}: {vals.mean():.4f} ± {vals.std():.4f}")

    return res


# ============================================================
# 3) RUN ALL MODELS
# ============================================================
all_results = {}
for mname in MODELS_TO_RUN:
    all_results[mname] = run_cv_for_model(
        model_name=mname,
        cache_dir=CACHE_DIR,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        seed=SEED
    )

print("\nDone. Models run:", list(all_results.keys()))


Device: cuda


Canonical channel intersection:   0%|          | 0/149 [00:00<?, ?it/s]

Precomputing windows:   0%|          | 0/149 [00:00<?, ?it/s]

[precompute done] subjects=149 | PD=100 | Control=49
[cache_dir] /home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset/precomputed_2min_win10_overlap50
[canonical channels] 60

[data] subjects=149 | PD=100 | Control=49

================ FOLD 1/5 | EEGNET ================
Windows: train=2185 val=552 test=690
Input: C=60 T=1280 | Params=2,098


Epoch 1/30:   0%|          | 0/35 [00:00<?, ?it/s]

...

[Epoch 028] VAL Window:  Acc=0.7120 BAcc=0.6590 Prec=0.7659 Rec=0.8179 F1=0.7911 AUC=0.7279 | VAL Subject: Acc=0.7083 BAcc=0.6562 Prec=0.7647 Rec=0.8125 F1=0.7879 AUC=0.7266


Epoch 29/30:   0%|          | 0/35 [00:00<?, ?it/s]

[Epoch 029] VAL Window:  Acc=0.7101 BAcc=0.6562 Prec=0.7640 Rec=0.8179 F1=0.7900 AUC=0.7280 | VAL Subject: Acc=0.7083 BAcc=0.6562 Prec=0.7647 Rec=0.8125 F1=0.7879 AUC=0.7266


Epoch 30/30:   0%|          | 0/35 [00:00<?, ?it/s]

[Epoch 030] VAL Window:  Acc=0.7120 BAcc=0.6603 Prec=0.7673 Rec=0.8152 F1=0.7905 AUC=0.7288 | VAL Subject: Acc=0.7083 BAcc=0.6562 Prec=0.7647 Rec=0.8125 F1=0.7879 AUC=0.7344

[FOLD 5 FINAL] WINDOW : Acc=0.6942 BAcc=0.6374 Prec=0.7735 Rec=0.7870 F1=0.7802 AUC=0.6824
  Confusion Matrix (rows=true, cols=pred):
  [[ 101  106]
   [  98  362]]
[FOLD 5 FINAL] SUBJECT: Acc=0.7586 BAcc=0.7028 Prec=0.8095 Rec=0.8500 F1=0.8293 AUC=0.7333
  Confusion Matrix (rows=true, cols=pred):
  [[   5    4]
   [   3   17]]

================ OVERALL 5-FOLD SUMMARY | EEGCONFORMER (mean ± std) ================
  win_acc: 0.7040 ± 0.0399
 win_bacc: 0.5914 ± 0.0572
 win_prec: 0.7201 ± 0.0378
  win_rec: 0.9252 ± 0.0774
   win_f1: 0.8069 ± 0.0277
  win_auc: 0.6851 ± 0.0273
 subj_acc: 0.7317 ± 0.0625
subj_bacc: 0.6156 ± 0.0974
subj_prec: 0.7338 ± 0.0601
 subj_rec: 0.9600 ± 0.0583
  subj_f1: 0.8286 ± 0.0342
 subj_auc: 0.7057 ± 0.0381

Done. Models run: ['eegnet', 'inception', 'eegconformer']


In [5]:
# ============================================================
# ML BASELINES — ds004584 (Cleaned FIF) — PD vs Control
#
# Models:
#   1) Linear Regression (sigmoid)
#   2) Logistic Regression
#   3) SVM (RBF, prob)
#   4) Random Forest
#   5) XGBoost (optional)
#
# Protocol:
#   ✅ Subject-wise Stratified 5-fold CV (leakage-safe)
#   ✅ EEG-only canonical channel intersection
#   ✅ Use FIRST 2 minutes (120s) from every recording
#   ✅ 10s windows, 50% overlap
#   ✅ Window-level + Subject-level metrics:
#        win_acc, win_bacc, win_prec, win_rec, win_f1, win_auc
#        subj_acc, subj_bacc, subj_prec, subj_rec, subj_f1, subj_auc
# ============================================================

import os, glob, random, warnings
import numpy as np
import mne

from tqdm.auto import tqdm
from mne.time_frequency import psd_array_welch

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------
# PATHS
# ----------------------------
ROOT_DIR  = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"   # cleaned FIF lives here
LABEL_DIR = "/home/varun/Desktop/Parkinsons_Resting_State"                  # participants.tsv lives here

# ----------------------------
# CONFIG
# ----------------------------
SEED = 42
N_SPLITS = 5

CROP_SECONDS = 120.0
WIN_SECONDS  = 10.0
OVERLAP      = 0.5

L_FREQ = 1.0
H_FREQ = 40.0
TARGET_SFREQ = 250.0

THR = 0.5  # decision threshold on P(PD)

# ----------------------------
# Reproducibility
# ----------------------------
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_all(SEED)

# ----------------------------
# Collect subject files (cleaned FIF)
# ----------------------------
def collect_subject_files(root_dir):
    """
    Expect cleaned FIF structure:
      root_dir/sub-XXX/eeg/*.fif
    If multiple FIF per subject, pick first sorted.
    Returns list of (sub_id, filepath)
    """
    fif_files = sorted(glob.glob(os.path.join(root_dir, "sub-*", "eeg", "*.fif")))
    by_sub = {}
    for fp in fif_files:
        base = os.path.basename(fp)
        sid = base.split("_")[0]
        if not sid.startswith("sub-"):
            # fallback: infer from folder
            sid = os.path.basename(os.path.dirname(os.path.dirname(fp)))
        by_sub.setdefault(sid, []).append(fp)

    items = []
    for sid in sorted(by_sub.keys()):
        items.append((sid, sorted(by_sub[sid])[0]))
    return items

# ----------------------------
# Labels
# ----------------------------
def load_labels_from_participants_tsv(label_dir):
    """
    participants.tsv columns:
      ['participant_id', 'GROUP', 'ID', 'EEG', 'AGE', 'GENDER', 'MOCA', 'UPDRS', 'TYPE']
    Returns label_map: {sub-XXX: 0/1} where 1=PD, 0=Control
    """
    import pandas as pd
    tsv = os.path.join(label_dir, "participants.tsv")
    if not os.path.exists(tsv):
        raise FileNotFoundError(f"participants.tsv not found at: {tsv}")

    df = pd.read_csv(tsv, sep="\t")

    if "participant_id" not in df.columns:
        raise ValueError(f"participants.tsv missing 'participant_id'. Columns: {list(df.columns)}")
    if "GROUP" not in df.columns:
        raise ValueError(f"participants.tsv missing 'GROUP'. Columns: {list(df.columns)}")

    def map_label(x):
        s = str(x).strip().upper()
        if s == "PD":
            return 1
        if s in {"CT", "HC", "CONTROL", "HEALTHY", "CTRL", "C"}:
            return 0
        raise ValueError(f"Unknown GROUP value: {x!r}")

    label_map = {}
    for _, row in df.iterrows():
        pid = str(row["participant_id"]).strip()
        grp = row["GROUP"]
        label_map[pid] = map_label(grp)

    n = len(label_map)
    n_pd = sum(v == 1 for v in label_map.values())
    n_ctrl = n - n_pd
    print(f"[labels] subjects={n} | PD={n_pd} | Control={n_ctrl} | label_col=GROUP")
    return label_map

# ----------------------------
# Canonical channel intersection (EEG-only)
# ----------------------------
def compute_channel_intersection(file_items):
    """
    EEG-only canonical intersection. Uses modern raw.pick(...)
    Keeps ordering from the first subject.
    """
    sid0, fp0 = file_items[0]
    raw0 = mne.io.read_raw_fif(fp0, preload=False, verbose="ERROR")
    raw0 = raw0.copy().pick("eeg")
    ch0 = [c for c in raw0.ch_names if c not in raw0.info["bads"]]
    inter = set(ch0)

    for sid, fp in tqdm(file_items, desc="Channel intersection", leave=False):
        raw = mne.io.read_raw_fif(fp, preload=False, verbose="ERROR")
        raw = raw.copy().pick("eeg")
        ch = set([c for c in raw.ch_names if c not in raw.info["bads"]])
        inter &= ch

    canonical = [c for c in ch0 if c in inter]
    if len(canonical) < 5:
        raise RuntimeError(f"Intersection too small ({len(canonical)}). Check data.")
    return canonical

# ----------------------------
# Preprocess + window: FIRST 2 MINUTES, 10s, 50% overlap
# ----------------------------
def preprocess_and_window(fp, canonical_chs):
    """
    Returns:
      Xw: [Nw, C, T] float32
      sfreq (float)
    """
    raw = mne.io.read_raw_fif(fp, preload=True, verbose="ERROR")
    raw = raw.copy().pick("eeg")           # modern API
    raw = raw.copy().pick(canonical_chs)   # modern API, ordered by canonical_chs
    raw = raw.copy().filter(L_FREQ, H_FREQ, fir_design="firwin", verbose="ERROR")
    raw = raw.copy().resample(TARGET_SFREQ, npad="auto", verbose="ERROR")

    sfreq = float(raw.info["sfreq"])
    data = raw.get_data()  # [C, Tfull]

    n_crop = int(CROP_SECONDS * sfreq)
    if data.shape[1] < n_crop:
        # should not happen per your constraint, but keep safe
        n_crop = data.shape[1]
    data = data[:, :n_crop]

    # z-score per channel (within-subject)
    data = (data - data.mean(axis=1, keepdims=True)) / (data.std(axis=1, keepdims=True) + 1e-8)

    win_len = int(WIN_SECONDS * sfreq)
    step = int(win_len * (1.0 - OVERLAP))
    if step <= 0 or data.shape[1] < win_len:
        return np.zeros((0, len(canonical_chs), win_len), dtype=np.float32), sfreq

    windows = []
    for start in range(0, data.shape[1] - win_len + 1, step):
        windows.append(data[:, start:start + win_len])

    if not windows:
        return np.zeros((0, len(canonical_chs), win_len), dtype=np.float32), sfreq

    return np.stack(windows, axis=0).astype(np.float32), sfreq

# ----------------------------
# Feature extraction (per window)
# ----------------------------
BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 40.0),
}

def window_features_bandpower(x_ct, sfreq):
    """
    x_ct: [C, T] (z-scored)
    returns 1D feature vector
    """
    C, T = x_ct.shape

    psd, freqs = psd_array_welch(
        x_ct, sfreq=sfreq, fmin=1.0, fmax=40.0,
        n_fft=min(1024, T), n_overlap=0, verbose=False
    )  # [C, F]

    feats = []
    for (f1, f2) in BANDS.values():
        idx = np.logical_and(freqs >= f1, freqs < f2)
        bp = np.trapz(psd[:, idx], freqs[idx], axis=1)  # [C]
        feats.append(bp)

    bp_all = np.stack(feats, axis=1)      # [C, nbands]
    bp_all = np.log(bp_all + 1e-12)       # stabilize
    feats_flat = bp_all.reshape(-1)       # [C*nbands]

    # simple time-domain stats per channel
    abs_mean = np.mean(np.abs(x_ct), axis=1)
    std = np.std(x_ct, axis=1)
    e2 = np.mean(x_ct**2, axis=1) + 1e-12
    e4 = np.mean(x_ct**4, axis=1)
    kurt_proxy = e4 / (e2**2)

    feats_td = np.concatenate([abs_mean, std, kurt_proxy], axis=0)  # [3C]
    return np.concatenate([feats_flat, feats_td], axis=0).astype(np.float32)

# ----------------------------
# Build subject -> windows/features cache (one-time)
# ----------------------------
def build_subject_cache(file_items, label_map, canonical_chs):
    """
    Returns:
      subj_list: list of subject ids used
      subj_y   : np.array [Nsubj]
      cache    : dict sid -> dict with:
                  'Xw_feat': [Nw, D] window features
                  'y': int label
    """
    cache = {}
    used = []

    for sid, fp in tqdm(file_items, desc="Build cache (all subjects)", leave=True):
        if sid not in label_map:
            continue

        Xw, sf = preprocess_and_window(fp, canonical_chs)
        if Xw.shape[0] == 0:
            continue

        # per-window features
        Fw = np.stack([window_features_bandpower(Xw[i], sf) for i in range(Xw.shape[0])], axis=0)  # [Nw, D]
        cache[sid] = {"Xw_feat": Fw, "y": int(label_map[sid])}
        used.append(sid)

    subj_list = sorted(used)
    subj_y = np.array([cache[s]["y"] for s in subj_list], dtype=int)
    print(f"[cache] subjects used={len(subj_list)} | PD={int(subj_y.sum())} | Control={int((subj_y==0).sum())}")
    return subj_list, subj_y, cache

# ----------------------------
# Models (ALL)
# ----------------------------
def get_models(seed=42):
    models = {}

    models["LinReg"] = Pipeline([
        ("scaler", StandardScaler()),
        ("reg", LinearRegression())
    ])

    models["LogReg"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"))
    ])

    models["SVM"] = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", probability=True))
    ])

    models["RF"] = RandomForestClassifier(
        n_estimators=600,
        random_state=seed,
        class_weight="balanced_subsample",
        n_jobs=-1,
        max_depth=None
    )

    # XGBoost optional
    try:
        import xgboost as xgb
        models["XGBoost"] = xgb.XGBClassifier(
            n_estimators=900,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1
        )
    except Exception:
        models["XGBoost"] = None

    return models

def prob_positive(model, X):
    """
    Return p(y=1) for classifiers/regressors.
    """
    # pipeline classifier
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "named_steps"):
        # pipeline with clf prob
        if "clf" in model.named_steps and hasattr(model.named_steps["clf"], "predict_proba"):
            return model.predict_proba(X)[:, 1]
        # linear regression pipeline
        if "reg" in model.named_steps:
            yhat = model.predict(X)
            return 1.0 / (1.0 + np.exp(-yhat))

    # decision function fallback
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return 1.0 / (1.0 + np.exp(-s))

    # last resort: sigmoid on prediction
    yhat = model.predict(X)
    return 1.0 / (1.0 + np.exp(-yhat))

# ----------------------------
# Metrics helper (ALL requested)
# ----------------------------
def compute_all_metrics(y_true, p_pos, thr=0.5):
    y_pred = (p_pos >= thr).astype(int)

    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1v  = f1_score(y_true, y_pred, zero_division=0)

    # AUC requires both classes present
    aucv = roc_auc_score(y_true, p_pos) if len(np.unique(y_true)) == 2 else np.nan

    return acc, bacc, prec, rec, f1v, aucv

# ----------------------------
# Run ML baselines with WINDOW + SUBJECT metrics
# ----------------------------
def run_ml_baselines(root_dir, label_dir, seed=42, n_splits=5):
    seed_all(seed)

    file_items = collect_subject_files(root_dir)
    if len(file_items) == 0:
        raise RuntimeError("No cleaned FIF files found. Check ROOT_DIR.")

    label_map = load_labels_from_participants_tsv(label_dir)

    # keep only labeled
    file_items = [(sid, fp) for sid, fp in file_items if sid in label_map]
    if len(file_items) == 0:
        raise RuntimeError("No files matched labeled subjects.")

    canonical_chs = compute_channel_intersection(file_items)
    print(f"[channels] canonical EEG intersection = {len(canonical_chs)}")

    subj_ids, subj_y, cache = build_subject_cache(file_items, label_map, canonical_chs)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = get_models(seed=seed)

    print("\nModels:")
    for k, v in models.items():
        if v is None:
            print(f"  - {k}: SKIPPED (xgboost not installed)")
        else:
            print(f"  - {k}: OK")

    # store results per model
    # each entry: list of dicts (per fold) with 12 metrics
    results = {name: [] for name, mdl in models.items() if mdl is not None}

    for fold, (tr_idx, te_idx) in enumerate(skf.split(subj_ids, subj_y), start=1):
        tr_subj = [subj_ids[i] for i in tr_idx]
        te_subj = [subj_ids[i] for i in te_idx]

        # build window-level train/test matrices (leakage-safe because split is by subject)
        Xtr_w, ytr_w = [], []
        Xte_w, yte_w, sid_te_w = [], [], []

        for sid in tr_subj:
            Fw = cache[sid]["Xw_feat"]
            y  = cache[sid]["y"]
            Xtr_w.append(Fw)
            ytr_w.append(np.full((Fw.shape[0],), y, dtype=int))

        for sid in te_subj:
            Fw = cache[sid]["Xw_feat"]
            y  = cache[sid]["y"]
            Xte_w.append(Fw)
            yte_w.append(np.full((Fw.shape[0],), y, dtype=int))
            sid_te_w.extend([sid] * Fw.shape[0])

        Xtr_w = np.concatenate(Xtr_w, axis=0)
        ytr_w = np.concatenate(ytr_w, axis=0)

        Xte_w = np.concatenate(Xte_w, axis=0)
        yte_w = np.concatenate(yte_w, axis=0)
        sid_te_w = np.array(sid_te_w)

        print(f"\n================ Fold {fold}/{n_splits} ================")
        print(f"[fold {fold}] train subjects={len(tr_subj)} | test subjects={len(te_subj)} | "
              f"train windows={len(ytr_w)} | test windows={len(yte_w)}")

        for name, model in models.items():
            if model is None:
                continue

            # fit on WINDOW-LEVEL features (more data; still safe because subject split)
            model.fit(Xtr_w, ytr_w)

            # ---------- WINDOW-LEVEL metrics ----------
            p_pos_win = prob_positive(model, Xte_w)
            win_acc, win_bacc, win_prec, win_rec, win_f1, win_auc = compute_all_metrics(yte_w, p_pos_win, thr=THR)

            # ---------- SUBJECT-LEVEL aggregation ----------
            # average probabilities across windows per subject
            subj_prob = {}
            subj_true = {}
            for y, p, sid in zip(yte_w, p_pos_win, sid_te_w):
                subj_prob.setdefault(sid, []).append(float(p))
                subj_true[sid] = int(y)

            y_true_sub = np.array([subj_true[sid] for sid in subj_prob.keys()], dtype=int)
            p_pos_sub  = np.array([np.mean(subj_prob[sid]) for sid in subj_prob.keys()], dtype=float)

            subj_acc, subj_bacc, subj_prec, subj_rec, subj_f1, subj_auc = compute_all_metrics(
                y_true_sub, p_pos_sub, thr=THR
            )

            fold_dict = dict(
                win_acc=win_acc, win_bacc=win_bacc, win_prec=win_prec, win_rec=win_rec, win_f1=win_f1, win_auc=win_auc,
                subj_acc=subj_acc, subj_bacc=subj_bacc, subj_prec=subj_prec, subj_rec=subj_rec, subj_f1=subj_f1, subj_auc=subj_auc
            )
            results[name].append(fold_dict)

            print(
                f"[{name:7s}] "
                f"WIN Acc={win_acc:.4f} BAcc={win_bacc:.4f} Prec={win_prec:.4f} Rec={win_rec:.4f} F1={win_f1:.4f} AUC={win_auc:.4f} | "
                f"SUBJ Acc={subj_acc:.4f} BAcc={subj_bacc:.4f} Prec={subj_prec:.4f} Rec={subj_rec:.4f} F1={subj_f1:.4f} AUC={subj_auc:.4f}"
            )

    # ---------- OVERALL SUMMARY (mean ± std) ----------
    metric_keys = [
        "win_acc","win_bacc","win_prec","win_rec","win_f1","win_auc",
        "subj_acc","subj_bacc","subj_prec","subj_rec","subj_f1","subj_auc"
    ]

    print("\n============================================================")
    print("=========== OVERALL 5-FOLD RESULTS (mean ± std) =============")
    print("============================================================\n")

    for name, fold_list in results.items():
        print(f"### {name}")
        for mk in metric_keys:
            vals = np.array([d[mk] for d in fold_list], dtype=float)
            print(f"{mk:>9s}: {vals.mean():.4f} ± {vals.std():.4f}")
        print()

# ----------------------------
# RUN
# ----------------------------
run_ml_baselines(ROOT_DIR, LABEL_DIR, seed=SEED, n_splits=N_SPLITS)


[labels] subjects=149 | PD=100 | Control=49 | label_col=GROUP


Channel intersection:   0%|          | 0/149 [00:00<?, ?it/s]

[channels] canonical EEG intersection = 60


Build cache (all subjects):   0%|          | 0/149 [00:00<?, ?it/s]

[cache] subjects used=149 | PD=100 | Control=49

Models:
  - LinReg: OK
  - LogReg: OK
  - SVM: OK
  - RF: OK
  - XGBoost: OK

================ Fold 1/5 ================
[fold 1] train subjects=119 | test subjects=30 | train windows=2737 | test windows=690
[LinReg ] WIN Acc=0.6522 BAcc=0.5283 Prec=0.6809 Rec=0.9000 F1=0.7753 AUC=0.6135 | SUBJ Acc=0.6333 BAcc=0.5000 Prec=0.6667 Rec=0.9000 F1=0.7660 AUC=0.6400
[LogReg ] WIN Acc=0.6855 BAcc=0.6685 Prec=0.7900 Rec=0.7196 F1=0.7531 AUC=0.7013 | SUBJ Acc=0.7000 BAcc=0.7000 Prec=0.8235 Rec=0.7000 F1=0.7568 AUC=0.7300
[SVM    ] WIN Acc=0.6246 BAcc=0.5576 Prec=0.7022 Rec=0.7587 F1=0.7294 AUC=0.6341 | SUBJ Acc=0.6333 BAcc=0.5500 Prec=0.6957 Rec=0.8000 F1=0.7442 AUC=0.6050
[RF     ] WIN Acc=0.6362 BAcc=0.5500 Prec=0.6953 Rec=0.8087 F1=0.7477 AUC=0.6317 | SUBJ Acc=0.6667 BAcc=0.5750 Prec=0.7083 Rec=0.8500 F1=0.7727 AUC=0.6200
[XGBoost] WIN Acc=0.6000 BAcc=0.5326 Prec=0.6870 Rec=0.7348 F1=0.7101 AUC=0.6478 | SUBJ Acc=0.6000 BAcc=0.5250 Prec=0.6818 

In [7]:
#!/usr/bin/env python3
# ============================================================
# Parkinsons vs Healthy (Resting EEG) — Leakage-safe 5-fold CV
# LiteParkNet v2: bias-fix + threshold tuning + median subject agg + early stop
#
# Protocol:
# ✅ Subject-wise Stratified 5-fold CV
# ✅ First 120s only
# ✅ Canonical EEG channel intersection (EEG-only)
# ✅ Windowing: 10s, 50% overlap
# ✅ Metrics per fold + overall:
#    Window: Acc/BAcc/Prec/Rec/F1/AUC
#    Subject: Acc/BAcc/Prec/Rec/F1/AUC
#
# Changes added (requested):
# 1) BCEWithLogitsLoss with pos_weight computed on TRAIN windows per fold
# 2) Validation threshold sweep; report both:
#      - thr_f1 (max F1 on val)
#      - thr_bacc (max balanced-acc on val)
#    Use thr_bacc for final reporting (you can switch)
# 3) Subject aggregation: MEDIAN of window probabilities (robust)
# 4) Early stopping patience=7 per fold (value requested)
# 5) Best checkpoint selected by validation AUC (more separability-oriented)
# ============================================================

import os
import re
import glob
import json
import math
import random
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score
)

# -------------------------
# USER PATHS
# -------------------------
ROOT_DIR = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
label_dir = "/home/varun/Desktop/Parkinsons_Resting_State"

# -------------------------
# REPRO
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# PROTOCOL SETTINGS
# -------------------------
FS = 128
DURATION_SEC = 120
WIN_SEC = 10
HOP_SEC = 5
WIN_SAMP = WIN_SEC * FS
HOP_SAMP = HOP_SEC * FS

N_FOLDS = 5

# Training hyperparams
EPOCHS = 60
BATCH = 128
LR = 2e-3
WEIGHT_DECAY = 1e-3

# Requested: early stop value = 7
PATIENCE = 7
MIN_DELTA = 1e-4

# Caching
CACHE_DIR = os.path.join(label_dir, "cache_windows_10s_50overlap_120s")
os.makedirs(CACHE_DIR, exist_ok=True)

# -------------------------
# MNE
# -------------------------
import mne


# ============================================================
# Subject id parsing
# ============================================================
def infer_subject_id(path: str) -> str:
    m = re.search(r"(sub-\d+)", path)
    if m:
        return m.group(1)
    m2 = re.search(r"(subject[_-]?\d+)", path.lower())
    if m2:
        return m2.group(1)
    parent = os.path.basename(os.path.dirname(path))
    if parent:
        return parent
    return os.path.splitext(os.path.basename(path))[0]


# ============================================================
# Labels loading
# ============================================================
def load_labels(label_dir: str, files: List[str]) -> Dict[str, int]:
    for cand in ["participants.tsv", "participants.csv", "Participants.tsv", "Participants.csv"]:
        p = os.path.join(label_dir, cand)
        if os.path.exists(p):
            df = pd.read_csv(p, sep="\t" if p.endswith(".tsv") else ",")
            df.columns = [c.lower() for c in df.columns]
            sid_col = None
            for c in ["participant_id", "subject", "sub", "subject_id", "id"]:
                if c in df.columns:
                    sid_col = c
                    break
            if sid_col is None:
                sid_col = df.columns[0]
            lab_col = None
            for c in ["group", "diagnosis", "label", "condition", "dx"]:
                if c in df.columns:
                    lab_col = c
                    break
            if lab_col is None:
                raise RuntimeError(f"Found {p} but cannot infer label column.")
            out = {}
            for _, r in df.iterrows():
                sid = str(r[sid_col])
                g = str(r[lab_col]).lower()
                if any(k in g for k in ["pd", "parkinson"]):
                    out[sid] = 1
                elif any(k in g for k in ["control", "healthy", "hc", "normal"]):
                    out[sid] = 0
                else:
                    try:
                        v = int(float(r[lab_col]))
                        out[sid] = 1 if v == 1 else 0
                    except:
                        pass
            return out

    label_candidates = glob.glob(os.path.join(label_dir, "*label*.csv")) + \
                       glob.glob(os.path.join(label_dir, "*label*.tsv")) + \
                       glob.glob(os.path.join(label_dir, "*label*.json"))
    if label_candidates:
        p = label_candidates[0]
        if p.endswith(".json"):
            data = json.load(open(p, "r"))
            return {str(k): int(v) for k, v in data.items()}
        df = pd.read_csv(p, sep="\t" if p.endswith(".tsv") else ",")
        df.columns = [c.lower() for c in df.columns]
        sid_col = df.columns[0]
        lab_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]
        return {str(r[sid_col]): int(r[lab_col]) for _, r in df.iterrows()}

    # fallback: infer from paths (weak)
    out = {}
    for f in files:
        sid = infer_subject_id(f)
        low = f.lower()
        if any(k in low for k in ["parkinson", "pd"]):
            out[sid] = 1
        elif any(k in low for k in ["control", "healthy", "hc", "normal"]):
            out[sid] = 0
    return out


def match_label(subject_id: str, label_map: Dict[str, int]) -> Optional[int]:
    if subject_id in label_map:
        return label_map[subject_id]
    for k, v in label_map.items():
        if k in subject_id or subject_id in k:
            return v
    return None


# ============================================================
# Canonical channel intersection
# ============================================================
def read_raw_any(path: str) -> mne.io.BaseRaw:
    if path.endswith(".fif") or path.endswith(".fif.gz"):
        return mne.io.read_raw_fif(path, preload=False, verbose=False)
    if path.endswith(".set"):
        return mne.io.read_raw_eeglab(path, preload=False, verbose=False)
    raise RuntimeError(f"Unsupported file: {path}")


def get_eeg_channel_names(raw: mne.io.BaseRaw) -> List[str]:
    picks = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, ecg=False, stim=False, misc=False, exclude=[])
    return [raw.ch_names[i] for i in picks]


def compute_canonical_intersection(file_list: List[str]) -> List[str]:
    inter = None
    for p in tqdm(file_list, desc="Finding canonical EEG channel intersection"):
        raw = read_raw_any(p)
        chs = set([c.upper() for c in get_eeg_channel_names(raw)])
        inter = chs if inter is None else inter.intersection(chs)
    inter = sorted(list(inter)) if inter is not None else []
    if len(inter) < 2:
        raise RuntimeError("Canonical EEG intersection too small.")
    return inter


# ============================================================
# Window extraction + caching
# ============================================================
def cache_path_for_subject(subject_id: str) -> str:
    return os.path.join(CACHE_DIR, f"{subject_id}_CACHED.npz")


def extract_windows_first120s(path: str, canonical_chs: List[str]) -> np.ndarray:
    raw = read_raw_any(path).load_data()
    sfreq = float(raw.info["sfreq"])
    if abs(sfreq - FS) > 1e-6:
        raw.resample(FS, npad="auto", verbose=False)

    picks = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, ecg=False, stim=False, misc=False, exclude=[])
    raw.pick(picks)

    ch_map = {c.upper(): c for c in raw.ch_names}
    keep = [ch_map[c] for c in canonical_chs if c in ch_map]
    raw.pick_channels(keep)

    name_to_idx = {n.upper(): i for i, n in enumerate(raw.ch_names)}
    order_idx = [name_to_idx[c] for c in canonical_chs if c in name_to_idx]
    data = raw.get_data()[order_idx].astype(np.float32)  # [C, T]

    needed = DURATION_SEC * FS
    if data.shape[1] < needed:
        return np.zeros((0, data.shape[0], WIN_SAMP), dtype=np.float32)

    data = data[:, :needed]
    C, T = data.shape
    starts = list(range(0, T - WIN_SAMP + 1, HOP_SAMP))
    X = np.zeros((len(starts), C, WIN_SAMP), dtype=np.float32)
    for i, s in enumerate(starts):
        X[i] = data[:, s:s + WIN_SAMP]
    return X


def build_or_load_subject_cache(subject_id: str, files: List[str], y: int, canonical_chs: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    cp = cache_path_for_subject(subject_id)
    if os.path.exists(cp):
        z = np.load(cp, allow_pickle=False)
        return z["X"], z["y"]

    Xs = []
    for f in files:
        X = extract_windows_first120s(f, canonical_chs)
        if X.shape[0] > 0:
            Xs.append(X)
    if len(Xs) == 0:
        Xall = np.zeros((0, len(canonical_chs), WIN_SAMP), dtype=np.float32)
        yall = np.zeros((0,), dtype=np.int64)
    else:
        Xall = np.concatenate(Xs, axis=0)
        yall = np.full((Xall.shape[0],), y, dtype=np.int64)

    np.savez_compressed(cp, X=Xall, y=yall)
    return Xall, yall


# ============================================================
# Dataset
# ============================================================
class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = X
        self.y = y

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)


# ============================================================
# Normalization (train-only stats)
# ============================================================
def compute_train_norm_stats(X_list: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    C = X_list[0].shape[1]
    total = 0
    s1 = np.zeros((C,), dtype=np.float64)
    s2 = np.zeros((C,), dtype=np.float64)

    for X in tqdm(X_list, desc="Computing train normalization stats"):
        if X.shape[0] == 0:
            continue
        total += X.shape[0] * X.shape[2]
        s1 += X.sum(axis=(0, 2))
        s2 += (X.astype(np.float64) ** 2).sum(axis=(0, 2))

    mean = (s1 / max(total, 1)).astype(np.float32)
    var = (s2 / max(total, 1) - mean.astype(np.float64) ** 2).astype(np.float32)
    var = np.maximum(var, 1e-6)
    std = np.sqrt(var).astype(np.float32)
    return mean, std


def apply_norm_inplace(X: np.ndarray, mean: np.ndarray, std: np.ndarray):
    X -= mean[None, :, None]
    X /= std[None, :, None]


# ============================================================
# Metrics
# ============================================================
def bin_metrics(y_true: np.ndarray, y_prob: np.ndarray, thresh: float) -> Dict[str, float]:
    y_pred = (y_prob >= thresh).astype(np.int64)
    out = {}
    out["acc"] = float(accuracy_score(y_true, y_pred))
    out["bacc"] = float(balanced_accuracy_score(y_true, y_pred))
    out["prec"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["rec"] = float(recall_score(y_true, y_pred, zero_division=0))
    out["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    try:
        out["auc"] = float(roc_auc_score(y_true, y_prob))
    except Exception:
        out["auc"] = float("nan")
    return out


def best_thresholds_from_val(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    ths = np.linspace(0.05, 0.95, 181)
    best_f1 = (-1.0, 0.5)
    best_bacc = (-1.0, 0.5)

    for t in ths:
        yp = (y_prob >= t).astype(np.int64)
        f1 = f1_score(y_true, yp, zero_division=0)
        bacc = balanced_accuracy_score(y_true, yp)
        if f1 > best_f1[0]:
            best_f1 = (f1, t)
        if bacc > best_bacc[0]:
            best_bacc = (bacc, t)

    return {"thr_f1": float(best_f1[1]), "thr_bacc": float(best_bacc[1])}


def aggregate_subject_probs_median(subject_ids: List[str], y_true_win: np.ndarray, y_prob_win: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    subj_to_probs = {}
    subj_to_true = {}
    for sid, yt, yp in zip(subject_ids, y_true_win, y_prob_win):
        subj_to_probs.setdefault(sid, []).append(float(yp))
        subj_to_true[sid] = int(yt)

    subjs = sorted(subj_to_probs.keys())
    y_true_subj = np.array([subj_to_true[s] for s in subjs], dtype=np.int64)
    y_prob_subj = np.array([np.median(subj_to_probs[s]) for s in subjs], dtype=np.float32)
    return y_true_subj, y_prob_subj


# ============================================================
# Model: LiteParkNet
# ============================================================
class DepthwiseSeparableConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, k, dilation=1, dropout=0.0):
        super().__init__()
        pad = ((k - 1) // 2) * dilation
        self.dw = nn.Conv1d(in_ch, in_ch, k, padding=pad, dilation=dilation, groups=in_ch, bias=False)
        self.pw = nn.Conv1d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dw(x)
        x = self.pw(x)
        x = self.bn(x)
        x = F.gelu(x)
        x = self.drop(x)
        return x


class SE1d(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hidden = max(4, ch // r)
        self.fc1 = nn.Conv1d(ch, hidden, 1)
        self.fc2 = nn.Conv1d(hidden, ch, 1)

    def forward(self, x):
        s = x.mean(dim=-1, keepdim=True)
        s = F.gelu(self.fc1(s))
        s = torch.sigmoid(self.fc2(s))
        return x * s


class TCNBlock(nn.Module):
    def __init__(self, ch, k=7, dilation=1, dropout=0.1, se=True):
        super().__init__()
        self.conv = DepthwiseSeparableConv1d(ch, ch, k=k, dilation=dilation, dropout=dropout)
        self.se = SE1d(ch) if se else nn.Identity()

    def forward(self, x):
        return x + self.se(self.conv(x))


class AttentiveStatsPool(nn.Module):
    def __init__(self, ch, attn_hidden=64):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(ch, attn_hidden, 1),
            nn.GELU(),
            nn.Conv1d(attn_hidden, 1, 1),
        )

    def forward(self, x):
        a = self.attn(x)  # [B, 1, T]
        a = torch.softmax(a, dim=-1)
        mean = (x * a).sum(dim=-1)  # [B, C]
        var = (a * (x - mean.unsqueeze(-1)) ** 2).sum(dim=-1) + 1e-6
        std = torch.sqrt(var)
        return torch.cat([mean, std], dim=1)


class LiteParkNet(nn.Module):
    def __init__(self, n_ch: int, emb: int = 64, dropout=0.15):
        super().__init__()
        fb = 3
        self.fb1 = nn.Conv1d(n_ch, n_ch * fb, kernel_size=31, padding=15, groups=n_ch, bias=False)
        self.fb2 = nn.Conv1d(n_ch, n_ch * fb, kernel_size=63, padding=31, groups=n_ch, bias=False)
        self.fb3 = nn.Conv1d(n_ch, n_ch * fb, kernel_size=125, padding=62, groups=n_ch, bias=False)

        self.mix = nn.Sequential(
            nn.Conv1d(n_ch * fb * 3, emb, 1, bias=False),
            nn.BatchNorm1d(emb),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.tcn = nn.Sequential(
            TCNBlock(emb, k=7, dilation=1, dropout=dropout, se=True),
            TCNBlock(emb, k=7, dilation=2, dropout=dropout, se=True),
            TCNBlock(emb, k=7, dilation=4, dropout=dropout, se=True),
            TCNBlock(emb, k=7, dilation=8, dropout=dropout, se=True),
            TCNBlock(emb, k=7, dilation=16, dropout=dropout, se=True),
        )

        self.pool = AttentiveStatsPool(emb, attn_hidden=64)

        self.head = nn.Sequential(
            nn.Linear(emb * 2, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        z = torch.cat([self.fb1(x), self.fb2(x), self.fb3(x)], dim=1)
        z = self.mix(z)
        z = self.tcn(z)
        p = self.pool(z)
        return self.head(p).squeeze(-1)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# Train / Eval
# ============================================================
@torch.no_grad()
def eval_loader(model, loader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs, ys = [], []
    for x, y in tqdm(loader, desc="Eval", leave=False):
        x = x.to(DEVICE)
        logit = model(x)
        prob = torch.sigmoid(logit).cpu().numpy()
        probs.append(prob)
        ys.append(y.numpy())
    return np.concatenate(ys).astype(np.int64), np.concatenate(probs).astype(np.float32)


def train_one_fold(model, train_loader, val_loader, pos_weight: float):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    # pos_weight to reduce PD-bias (computed from TRAIN windows)
    pw = torch.tensor([pos_weight], device=DEVICE, dtype=torch.float32)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)

    best_score = -1e9
    best_state = None
    bad = 0

    for ep in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in tqdm(train_loader, desc=f"Train ep{ep:02d}", leave=False):
            x = x.to(DEVICE)
            y = y.to(DEVICE).float()
            opt.zero_grad(set_to_none=True)
            logit = model(x)
            loss = crit(logit, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        sched.step()

        # Validation
        yv_true, yv_prob = eval_loader(model, val_loader)
        try:
            val_auc = float(roc_auc_score(yv_true, yv_prob))
        except Exception:
            val_auc = float("nan")
        val_thr = best_thresholds_from_val(yv_true, yv_prob)
        m_bacc = bin_metrics(yv_true, yv_prob, thresh=val_thr["thr_bacc"])
        m_f1 = bin_metrics(yv_true, yv_prob, thresh=val_thr["thr_f1"])

        # Select best checkpoint by AUC (primary), fallback to bAcc if AUC nan
        score = val_auc if not np.isnan(val_auc) else m_bacc["bacc"]

        print(
            f"[ep {ep:02d}] loss={np.mean(losses):.4f} | "
            f"val_auc={val_auc:.4f} | "
            f"val_bacc@thr={m_bacc['bacc']:.4f} (thr={val_thr['thr_bacc']:.3f}) | "
            f"val_f1@thr={m_f1['f1']:.4f} (thr={val_thr['thr_f1']:.3f})"
        )

        if score > best_score + MIN_DELTA:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"Early stopping at epoch {ep} (best_score={best_score:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


# ============================================================
# Main
# ============================================================
def main():
    print(f"DEVICE: {DEVICE}")
    files = []
    for ext in ["*.fif", "*.fif.gz", "*.set"]:
        files.extend(glob.glob(os.path.join(ROOT_DIR, "**", ext), recursive=True))
    files = sorted(files)
    if len(files) == 0:
        raise RuntimeError(f"No EEG files found in {ROOT_DIR} (.fif/.set expected).")

    print(f"Found {len(files)} EEG file(s).")

    label_map = load_labels(label_dir, files)

    subj_to_files: Dict[str, List[str]] = {}
    subj_to_label: Dict[str, int] = {}

    for f in files:
        sid = infer_subject_id(f)
        y = match_label(sid, label_map)
        if y is None:
            low = f.lower()
            if any(k in low for k in ["parkinson", "pd"]):
                y = 1
            elif any(k in low for k in ["control", "healthy", "hc", "normal"]):
                y = 0
        if y is None:
            continue
        subj_to_files.setdefault(sid, []).append(f)
        subj_to_label[sid] = int(y)

    subjects = sorted(subj_to_files.keys())
    y_subj = np.array([subj_to_label[s] for s in subjects], dtype=np.int64)
    print(f"Subjects with labels: {len(subjects)} | PD={int(y_subj.sum())} | Control={int((1-y_subj).sum())}")
    if len(subjects) < 10:
        raise RuntimeError("Too few labeled subjects after label matching.")

    repr_files = [subj_to_files[s][0] for s in subjects]
    canonical_chs = compute_canonical_intersection(repr_files)
    print(f"Canonical EEG intersection channels: {len(canonical_chs)}")

    print("Caching windows (subject-level cache)...")
    subj_cache = {}
    usable_subjects = []
    usable_labels = []

    for sid in tqdm(subjects, desc="Caching"):
        X, y = build_or_load_subject_cache(sid, subj_to_files[sid], subj_to_label[sid], canonical_chs)
        if X.shape[0] == 0:
            continue
        subj_cache[sid] = (X, y)
        usable_subjects.append(sid)
        usable_labels.append(subj_to_label[sid])

    subjects = usable_subjects
    y_subj = np.array(usable_labels, dtype=np.int64)
    print(f"Usable subjects after 120s+windowing: {len(subjects)}")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_window_metrics = []
    fold_subject_metrics = []

    all_win_true, all_win_prob = [], []
    all_sub_true, all_sub_prob = [], []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(subjects)), y_subj), start=1):
        tr_sids = [subjects[i] for i in tr_idx]
        te_sids = [subjects[i] for i in te_idx]

        # fold-local stratified train/val split (simple, deterministic)
        val_frac = 0.15
        tr0 = [s for s in tr_sids if subj_to_label[s] == 0]
        tr1 = [s for s in tr_sids if subj_to_label[s] == 1]
        rng = np.random.RandomState(SEED + fold)
        rng.shuffle(tr0); rng.shuffle(tr1)
        v0 = tr0[: max(1, int(len(tr0) * val_frac))]
        v1 = tr1[: max(1, int(len(tr1) * val_frac))]
        val_sids = sorted(list(set(v0 + v1)))
        train_sids = [s for s in tr_sids if s not in set(val_sids)]
        if len(val_sids) < 2:
            val_sids = tr_sids[:2]
            train_sids = tr_sids[2:]

        print("\n" + "=" * 60)
        print(f"FOLD {fold}/{N_FOLDS}")
        print(f"Train subjects: {len(train_sids)} | Val subjects: {len(val_sids)} | Test subjects: {len(te_sids)}")

        # train stats
        X_train_list = [subj_cache[s][0] for s in train_sids]
        mean, std = compute_train_norm_stats(X_train_list)

        def collect_windows(sids: List[str]) -> Tuple[np.ndarray, np.ndarray, List[str]]:
            Xs, ys, sid_list = [], [], []
            for s in sids:
                X, y = subj_cache[s]
                Xc = X.copy()
                apply_norm_inplace(Xc, mean, std)
                Xs.append(Xc)
                ys.append(y)
                sid_list.extend([s] * Xc.shape[0])
            return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0), sid_list

        Xtr, ytr, _ = collect_windows(train_sids)
        Xva, yva, _ = collect_windows(val_sids)
        Xte, yte, te_sid_per_win = collect_windows(te_sids)

        # window-level class counts for pos_weight
        n_pos = int((ytr == 1).sum())
        n_neg = int((ytr == 0).sum())
        pos_weight = (n_neg / max(n_pos, 1))
        print(f"Train windows: {Xtr.shape[0]} | pos={n_pos} neg={n_neg} | pos_weight={pos_weight:.4f}")

        print(f"Windows: train={Xtr.shape[0]} val={Xva.shape[0]} test={Xte.shape[0]}")
        n_ch = Xtr.shape[1]

        train_loader = DataLoader(WindowDataset(Xtr, ytr), batch_size=BATCH, shuffle=True, num_workers=4, pin_memory=True)
        val_loader   = DataLoader(WindowDataset(Xva, yva), batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
        test_loader  = DataLoader(WindowDataset(Xte, yte), batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

        model = LiteParkNet(n_ch=n_ch, emb=64, dropout=0.15).to(DEVICE)
        n_params = count_params(model)
        print(f"LiteParkNet params: {n_params} (~{n_params/1e3:.1f}k)")
        if n_params > 150_000:
            raise RuntimeError("Model exceeds 150k parameters. Reduce emb or head width.")

        model = train_one_fold(model, train_loader, val_loader, pos_weight=pos_weight)

        # --- Determine thresholds from VAL (leakage-safe), then apply to TEST
        yv_true, yv_prob = eval_loader(model, val_loader)
        thrs = best_thresholds_from_val(yv_true, yv_prob)
        thr_use = thrs["thr_bacc"]  # choose thr_bacc for final reporting (more balanced)
        print(f"Chosen thresholds from VAL: thr_bacc={thrs['thr_bacc']:.3f} | thr_f1={thrs['thr_f1']:.3f} | USING thr_bacc")

        # test inference
        y_true_win, y_prob_win = eval_loader(model, test_loader)

        # window metrics @ chosen threshold
        win_m = bin_metrics(y_true_win, y_prob_win, thresh=thr_use)

        # subject aggregation (median)
        y_true_sub, y_prob_sub = aggregate_subject_probs_median(te_sid_per_win, y_true_win, y_prob_win)
        sub_m = bin_metrics(y_true_sub, y_prob_sub, thresh=thr_use)

        fold_window_metrics.append(win_m)
        fold_subject_metrics.append(sub_m)

        all_win_true.append(y_true_win)
        all_win_prob.append(y_prob_win)
        all_sub_true.append(y_true_sub)
        all_sub_prob.append(y_prob_sub)

        print("---- Fold results (TEST) ----")
        print(f"Window: acc={win_m['acc']:.4f} bacc={win_m['bacc']:.4f} prec={win_m['prec']:.4f} rec={win_m['rec']:.4f} f1={win_m['f1']:.4f} auc={win_m['auc']:.4f}")
        print(f"Subj  : acc={sub_m['acc']:.4f} bacc={sub_m['bacc']:.4f} prec={sub_m['prec']:.4f} rec={sub_m['rec']:.4f} f1={sub_m['f1']:.4f} auc={sub_m['auc']:.4f}")

    def mean_std(metric_list: List[Dict[str, float]], key: str) -> Tuple[float, float]:
        vals = np.array([m[key] for m in metric_list], dtype=np.float64)
        return float(np.nanmean(vals)), float(np.nanstd(vals))

    print("\n" + "=" * 70)
    print("=============== OVERALL (mean ± std across folds) ===============")

    for k in ["acc", "bacc", "prec", "rec", "f1", "auc"]:
        m, s = mean_std(fold_window_metrics, k)
        print(f"win_{k:>4s}: {m:.4f} ± {s:.4f}")

    for k in ["acc", "bacc", "prec", "rec", "f1", "auc"]:
        m, s = mean_std(fold_subject_metrics, k)
        print(f"subj_{k:>4s}: {m:.4f} ± {s:.4f}")

    yW = np.concatenate(all_win_true, axis=0)
    pW = np.concatenate(all_win_prob, axis=0)
    yS = np.concatenate(all_sub_true, axis=0)
    pS = np.concatenate(all_sub_prob, axis=0)

    try:
        aucW = roc_auc_score(yW, pW)
        aucS = roc_auc_score(yS, pS)
        print("\nPooled ROC-AUC (all folds concatenated):")
        print(f"  Window AUC: {aucW:.4f}")
        print(f"  Subj   AUC: {aucS:.4f}")
    except Exception:
        pass


if __name__ == "__main__":
    main()


DEVICE: cuda
Found 149 EEG file(s).
Subjects with labels: 149 | PD=100 | Control=49


Finding canonical EEG channel intersection:   0%|          | 0/149 [00:00<?, ?it/s]

Canonical EEG intersection channels: 60
Caching windows (subject-level cache)...


Caching:   0%|          | 0/149 [00:00<?, ?it/s]

Usable subjects after 120s+windowing: 149

FOLD 1/5
Train subjects: 102 | Val subjects: 17 | Test subjects: 30


...

[ep 20] loss=0.0537 | val_auc=0.7785 | val_bacc@thr=0.7609 (thr=0.155) | val_f1@thr=0.8105 (thr=0.050)
Early stopping at epoch 20 (best_score=0.7968)


Eval:   0%|          | 0/4 [00:00<?, ?it/s]

Chosen thresholds from VAL: thr_bacc=0.180 | thr_f1=0.150 | USING thr_bacc


Eval:   0%|          | 0/6 [00:00<?, ?it/s]

---- Fold results (TEST) ----
Window: acc=0.7586 bacc=0.7732 prec=0.8966 rec=0.7348 f1=0.8076 auc=0.8243
Subj  : acc=0.8276 bacc=0.8444 prec=0.9412 rec=0.8000 f1=0.8649 auc=0.8944

=============== OVERALL (mean ± std across folds) ===============
win_ acc: 0.6909 ± 0.0583
win_bacc: 0.6762 ± 0.1042
win_prec: 0.8184 ± 0.0881
win_ rec: 0.7213 ± 0.1054
win_  f1: 0.7562 ± 0.0479
win_ auc: 0.7253 ± 0.0983
subj_ acc: 0.7189 ± 0.0761
subj_bacc: 0.7039 ± 0.1352
subj_prec: 0.8426 ± 0.1077
subj_ rec: 0.7500 ± 0.1000
subj_  f1: 0.7819 ± 0.0497
subj_ auc: 0.7569 ± 0.1168

Pooled ROC-AUC (all folds concatenated):
  Window AUC: 0.6571
  Subj   AUC: 0.6647


In [13]:
!pip install mamba-ssm


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
  Created wheel for mamba-ssm: filename=mamba_ssm-2.2.6.post3-cp310-cp310-linux_x86_64.whl size=367849466 sha256=21913c90ee3fc5cad4dbf8c98d78149b90f967e8a73b4b0ad5941216f54163ef
  Stored in directory: /home/varun/.cache/pip/wheels/06/22/95/e8e06a9238bb80283b83f91bafb56589cee954483992b5de8c
Successfully built mamba-ssm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mamba-ssm]━━━━━━━━ 2/3 [mamba-ssm]


In [17]:
#!/usr/bin/env python3
# ============================================================
# Parkinsons vs Healthy (Resting EEG) — Leakage-safe 5-fold CV
# LiteParkSSM v2 (FULL-LENGTH): uses ENTIRE duration per recording
#
# Protocol:
# ✅ Subject-wise Stratified 5-fold CV
# ✅ Uses FULL recording duration (variable length per subject/recording)
# ✅ Canonical EEG channel intersection (EEG-only)
# ✅ Windowing: 10s, 50% overlap (slide across full length)
# ✅ Metrics per fold + overall:
#    Window: Acc/BAcc/Prec/Rec/F1/AUC
#    Subject: Acc/BAcc/Prec/Rec/F1/AUC
#
# Training:
# 1) BCEWithLogitsLoss with pos_weight computed on TRAIN windows per fold
# 2) Validation threshold sweep; report both thr_f1 and thr_bacc; use thr_bacc
# 3) Subject aggregation: MEDIAN of window probabilities (robust)
# 4) Early stopping patience=7 per fold
# 5) Best checkpoint selected by validation AUC (fallback to bAcc if AUC NaN)
#
# Notes:
# - If you install "mamba-ssm", the backbone uses Mamba SSM.
# - If not installed, it falls back to a lightweight EMA-based linear SSM.
# - FULL-LENGTH can create many windows for long recordings; adjust BATCH if OOM.
# ============================================================

import os
import re
import glob
import json
import random
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score
)

# -------------------------
# USER PATHS
# -------------------------
ROOT_DIR = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
label_dir = "/home/varun/Desktop/Parkinsons_Resting_State"

# -------------------------
# REPRO
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# PROTOCOL SETTINGS
# -------------------------
FS = 128
WIN_SEC = 10
HOP_SEC = 5
WIN_SAMP = WIN_SEC * FS
HOP_SAMP = HOP_SEC * FS

N_FOLDS = 5

# Training hyperparams
EPOCHS = 60
BATCH = 128
LR = 2e-3
WEIGHT_DECAY = 1e-3

# Requested: early stop value = 7
PATIENCE = 7
MIN_DELTA = 1e-4

# Caching (FULL LENGTH)
CACHE_DIR = os.path.join(label_dir, "cache_windows_10s_50overlap_FULLLEN")
os.makedirs(CACHE_DIR, exist_ok=True)

# -------------------------
# MNE
# -------------------------
import mne


# ============================================================
# Subject id parsing
# ============================================================
def infer_subject_id(path: str) -> str:
    m = re.search(r"(sub-\d+)", path)
    if m:
        return m.group(1)
    m2 = re.search(r"(subject[_-]?\d+)", path.lower())
    if m2:
        return m2.group(1)
    parent = os.path.basename(os.path.dirname(path))
    if parent:
        return parent
    return os.path.splitext(os.path.basename(path))[0]


# ============================================================
# Labels loading
# ============================================================
def load_labels(label_dir: str, files: List[str]) -> Dict[str, int]:
    for cand in ["participants.tsv", "participants.csv", "Participants.tsv", "Participants.csv"]:
        p = os.path.join(label_dir, cand)
        if os.path.exists(p):
            df = pd.read_csv(p, sep="\t" if p.endswith(".tsv") else ",")
            df.columns = [c.lower() for c in df.columns]
            sid_col = None
            for c in ["participant_id", "subject", "sub", "subject_id", "id"]:
                if c in df.columns:
                    sid_col = c
                    break
            if sid_col is None:
                sid_col = df.columns[0]
            lab_col = None
            for c in ["group", "diagnosis", "label", "condition", "dx"]:
                if c in df.columns:
                    lab_col = c
                    break
            if lab_col is None:
                raise RuntimeError(f"Found {p} but cannot infer label column.")
            out = {}
            for _, r in df.iterrows():
                sid = str(r[sid_col])
                g = str(r[lab_col]).lower()
                if any(k in g for k in ["pd", "parkinson"]):
                    out[sid] = 1
                elif any(k in g for k in ["control", "healthy", "hc", "normal"]):
                    out[sid] = 0
                else:
                    try:
                        v = int(float(r[lab_col]))
                        out[sid] = 1 if v == 1 else 0
                    except:
                        pass
            return out

    label_candidates = glob.glob(os.path.join(label_dir, "*label*.csv")) + \
                       glob.glob(os.path.join(label_dir, "*label*.tsv")) + \
                       glob.glob(os.path.join(label_dir, "*label*.json"))
    if label_candidates:
        p = label_candidates[0]
        if p.endswith(".json"):
            data = json.load(open(p, "r"))
            return {str(k): int(v) for k, v in data.items()}
        df = pd.read_csv(p, sep="\t" if p.endswith(".tsv") else ",")
        df.columns = [c.lower() for c in df.columns]
        sid_col = df.columns[0]
        lab_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]
        return {str(r[sid_col]): int(r[lab_col]) for _, r in df.iterrows()}

    # fallback: infer from paths (weak)
    out = {}
    for f in files:
        sid = infer_subject_id(f)
        low = f.lower()
        if any(k in low for k in ["parkinson", "pd"]):
            out[sid] = 1
        elif any(k in low for k in ["control", "healthy", "hc", "normal"]):
            out[sid] = 0
    return out


def match_label(subject_id: str, label_map: Dict[str, int]) -> Optional[int]:
    if subject_id in label_map:
        return label_map[subject_id]
    for k, v in label_map.items():
        if k in subject_id or subject_id in k:
            return v
    return None


# ============================================================
# Canonical channel intersection
# ============================================================
def read_raw_any(path: str) -> mne.io.BaseRaw:
    if path.endswith(".fif") or path.endswith(".fif.gz"):
        return mne.io.read_raw_fif(path, preload=False, verbose=False)
    if path.endswith(".set"):
        return mne.io.read_raw_eeglab(path, preload=False, verbose=False)
    raise RuntimeError(f"Unsupported file: {path}")


def get_eeg_channel_names(raw: mne.io.BaseRaw) -> List[str]:
    picks = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, ecg=False, stim=False, misc=False, exclude=[])
    return [raw.ch_names[i] for i in picks]


def compute_canonical_intersection(file_list: List[str]) -> List[str]:
    inter = None
    for p in tqdm(file_list, desc="Finding canonical EEG channel intersection"):
        raw = read_raw_any(p)
        chs = set([c.upper() for c in get_eeg_channel_names(raw)])
        inter = chs if inter is None else inter.intersection(chs)
    inter = sorted(list(inter)) if inter is not None else []
    if len(inter) < 2:
        raise RuntimeError("Canonical EEG intersection too small.")
    return inter


# ============================================================
# Window extraction + caching (FULL LENGTH)
# ============================================================
def cache_path_for_subject(subject_id: str) -> str:
    return os.path.join(CACHE_DIR, f"{subject_id}_CACHED_FULL.npz")


def extract_windows_full_length(path: str, canonical_chs: List[str]) -> np.ndarray:
    """
    Extract 10s windows with 5s hop across the ENTIRE recording.
    Returns X: [Nw, C, WIN_SAMP]
    """
    raw = read_raw_any(path).load_data()
    sfreq = float(raw.info["sfreq"])
    if abs(sfreq - FS) > 1e-6:
        raw.resample(FS, npad="auto", verbose=False)

    picks = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, ecg=False, stim=False, misc=False, exclude=[])
    raw.pick(picks)

    # keep only canonical channels (case-insensitive)
    ch_map = {c.upper(): c for c in raw.ch_names}
    keep = [ch_map[c] for c in canonical_chs if c in ch_map]
    raw.pick_channels(keep)

    # enforce canonical ordering
    name_to_idx = {n.upper(): i for i, n in enumerate(raw.ch_names)}
    order_idx = [name_to_idx[c] for c in canonical_chs if c in name_to_idx]
    data = raw.get_data()[order_idx].astype(np.float32)  # [C, T]

    if data.shape[1] < WIN_SAMP:
        return np.zeros((0, data.shape[0], WIN_SAMP), dtype=np.float32)

    C, T = data.shape
    starts = list(range(0, T - WIN_SAMP + 1, HOP_SAMP))
    X = np.zeros((len(starts), C, WIN_SAMP), dtype=np.float32)
    for i, s in enumerate(starts):
        X[i] = data[:, s:s + WIN_SAMP]
    return X


def build_or_load_subject_cache(subject_id: str, files: List[str], y: int, canonical_chs: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    cp = cache_path_for_subject(subject_id)
    if os.path.exists(cp):
        z = np.load(cp, allow_pickle=False)
        return z["X"], z["y"]

    Xs = []
    for f in files:
        X = extract_windows_full_length(f, canonical_chs)
        if X.shape[0] > 0:
            Xs.append(X)

    if len(Xs) == 0:
        Xall = np.zeros((0, len(canonical_chs), WIN_SAMP), dtype=np.float32)
        yall = np.zeros((0,), dtype=np.int64)
    else:
        Xall = np.concatenate(Xs, axis=0)
        yall = np.full((Xall.shape[0],), y, dtype=np.int64)

    np.savez_compressed(cp, X=Xall, y=yall)
    return Xall, yall


# ============================================================
# Dataset
# ============================================================
class WindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = X
        self.y = y

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)


# ============================================================
# Normalization (train-only stats)
# ============================================================
def compute_train_norm_stats(X_list: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    C = X_list[0].shape[1]
    total = 0
    s1 = np.zeros((C,), dtype=np.float64)
    s2 = np.zeros((C,), dtype=np.float64)

    for X in tqdm(X_list, desc="Computing train normalization stats"):
        if X.shape[0] == 0:
            continue
        total += X.shape[0] * X.shape[2]
        s1 += X.sum(axis=(0, 2))
        s2 += (X.astype(np.float64) ** 2).sum(axis=(0, 2))

    mean = (s1 / max(total, 1)).astype(np.float32)
    var = (s2 / max(total, 1) - mean.astype(np.float64) ** 2).astype(np.float32)
    var = np.maximum(var, 1e-6)
    std = np.sqrt(var).astype(np.float32)
    return mean, std


def apply_norm_inplace(X: np.ndarray, mean: np.ndarray, std: np.ndarray):
    X -= mean[None, :, None]
    X /= std[None, :, None]


# ============================================================
# Metrics
# ============================================================
def bin_metrics(y_true: np.ndarray, y_prob: np.ndarray, thresh: float) -> Dict[str, float]:
    y_pred = (y_prob >= thresh).astype(np.int64)
    out = {}
    out["acc"] = float(accuracy_score(y_true, y_pred))
    out["bacc"] = float(balanced_accuracy_score(y_true, y_pred))
    out["prec"] = float(precision_score(y_true, y_pred, zero_division=0))
    out["rec"] = float(recall_score(y_true, y_pred, zero_division=0))
    out["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    try:
        out["auc"] = float(roc_auc_score(y_true, y_prob))
    except Exception:
        out["auc"] = float("nan")
    return out


def best_thresholds_from_val(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    ths = np.linspace(0.05, 0.95, 181)
    best_f1 = (-1.0, 0.5)
    best_bacc = (-1.0, 0.5)
    for t in ths:
        yp = (y_prob >= t).astype(np.int64)
        f1v = f1_score(y_true, yp, zero_division=0)
        baccv = balanced_accuracy_score(y_true, yp)
        if f1v > best_f1[0]:
            best_f1 = (f1v, t)
        if baccv > best_bacc[0]:
            best_bacc = (baccv, t)
    return {"thr_f1": float(best_f1[1]), "thr_bacc": float(best_bacc[1])}


def aggregate_subject_probs_median(subject_ids: List[str], y_true_win: np.ndarray, y_prob_win: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    subj_to_probs = {}
    subj_to_true = {}
    for sid, yt, yp in zip(subject_ids, y_true_win, y_prob_win):
        subj_to_probs.setdefault(sid, []).append(float(yp))
        subj_to_true[sid] = int(yt)

    subjs = sorted(subj_to_probs.keys())
    y_true_subj = np.array([subj_to_true[s] for s in subjs], dtype=np.int64)
    y_prob_subj = np.array([np.median(subj_to_probs[s]) for s in subjs], dtype=np.float32)
    return y_true_subj, y_prob_subj


# ============================================================
# Model: LiteParkSSM (Mamba if available; EMA-SSM fallback)
# ============================================================
def _try_import_mamba():
    try:
        from mamba_ssm import Mamba  # type: ignore
        return Mamba
    except Exception:
        try:
            from mamba_ssm.modules.mamba_simple import Mamba  # type: ignore
            return Mamba
        except Exception:
            return None


MambaLayer = _try_import_mamba()


class DropPath(nn.Module):
    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = float(p)

    def forward(self, x):
        if self.p == 0.0 or (not self.training):
            return x
        keep = 1.0 - self.p
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(shape).bernoulli_(keep)
        return x * mask / keep


class EMAStateSpace(nn.Module):
    """
    Simple diagonal linear SSM:
      h_t = a*h_{t-1} + (1-a)*x_t, with learned a in (0,1) per feature dim.
    """
    def __init__(self, d_model: int):
        super().__init__()
        self.logit_a = nn.Parameter(torch.zeros(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B,T,D]
        a = torch.sigmoid(self.logit_a).view(1, 1, -1)  # [1,1,D]
        B, T, D = x.shape
        h = torch.zeros((B, D), device=x.device, dtype=x.dtype)
        y = torch.empty_like(x)
        for t in range(T):
            h = a[:, 0, :] * h + (1.0 - a[:, 0, :]) * x[:, t, :]
            y[:, t, :] = h
        return y


class SSMBlock(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.15, drop_path: float = 0.0, use_mamba: bool = True):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.dp = DropPath(drop_path)

        self.use_mamba = bool(use_mamba and (MambaLayer is not None))
        if self.use_mamba:
            self.ssm = MambaLayer(d_model=d_model, d_state=16, d_conv=4, expand=2)
        else:
            self.ssm = EMAStateSpace(d_model)

        mlp_hidden = int(2 * d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.norm1(x)
        z = self.ssm(z)
        x = x + self.dp(self.drop(z))

        z2 = self.norm2(x)
        z2 = self.mlp(z2)
        x = x + self.dp(self.drop(z2))
        return x


class AttentiveStatsPool(nn.Module):
    def __init__(self, d_model: int, attn_hidden: int = 64):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(d_model, attn_hidden, 1),
            nn.GELU(),
            nn.Conv1d(attn_hidden, 1, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B,D,T]
        a = self.attn(x)  # [B,1,T]
        a = torch.softmax(a, dim=-1)
        mean = (x * a).sum(dim=-1)  # [B,D]
        var = (a * (x - mean.unsqueeze(-1)) ** 2).sum(dim=-1) + 1e-6
        std = torch.sqrt(var)
        return torch.cat([mean, std], dim=1)  # [B,2D]


class LiteParkSSM(nn.Module):
    def __init__(
        self,
        n_ch: int,
        d_model: int = 64,
        n_layers: int = 4,
        dropout: float = 0.15,
        drop_path: float = 0.0,
        use_mamba: bool = True,
    ):
        super().__init__()

        self.prefilt = nn.Conv1d(n_ch, n_ch, kernel_size=31, padding=15, groups=n_ch, bias=False)

        self.proj = nn.Sequential(
            nn.Conv1d(n_ch, d_model, kernel_size=1, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        blocks = []
        for i in range(n_layers):
            dp_i = drop_path * (i / max(n_layers - 1, 1))
            blocks.append(SSMBlock(d_model, dropout=dropout, drop_path=dp_i, use_mamba=use_mamba))
        self.blocks = nn.ModuleList(blocks)

        self.pool = AttentiveStatsPool(d_model, attn_hidden=64)

        self.head = nn.Sequential(
            nn.Linear(d_model * 2, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B,C,T]
        x = self.prefilt(x)
        x = self.proj(x)           # [B,D,T]
        x = x.transpose(1, 2)      # [B,T,D]
        for blk in self.blocks:
            x = blk(x)
        x = x.transpose(1, 2)      # [B,D,T]
        p = self.pool(x)           # [B,2D]
        return self.head(p).squeeze(-1)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# Train / Eval
# ============================================================
@torch.no_grad()
def eval_loader(model, loader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs, ys = [], []
    for x, y in tqdm(loader, desc="Eval", leave=False):
        x = x.to(DEVICE)
        logit = model(x)
        prob = torch.sigmoid(logit).cpu().numpy()
        probs.append(prob)
        ys.append(y.numpy())
    return np.concatenate(ys).astype(np.int64), np.concatenate(probs).astype(np.float32)


def train_one_fold(model, train_loader, val_loader, pos_weight: float):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    pw = torch.tensor([pos_weight], device=DEVICE, dtype=torch.float32)
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)

    best_score = -1e9
    best_state = None
    bad = 0

    for ep in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for x, y in tqdm(train_loader, desc=f"Train ep{ep:02d}", leave=False):
            x = x.to(DEVICE)
            y = y.to(DEVICE).float()
            opt.zero_grad(set_to_none=True)
            logit = model(x)
            loss = crit(logit, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        sched.step()

        yv_true, yv_prob = eval_loader(model, val_loader)
        try:
            val_auc = float(roc_auc_score(yv_true, yv_prob))
        except Exception:
            val_auc = float("nan")

        val_thr = best_thresholds_from_val(yv_true, yv_prob)
        m_bacc = bin_metrics(yv_true, yv_prob, thresh=val_thr["thr_bacc"])
        m_f1 = bin_metrics(yv_true, yv_prob, thresh=val_thr["thr_f1"])

        score = val_auc if not np.isnan(val_auc) else m_bacc["bacc"]

        print(
            f"[ep {ep:02d}] loss={np.mean(losses):.4f} | "
            f"val_auc={val_auc:.4f} | "
            f"val_bacc@thr={m_bacc['bacc']:.4f} (thr={val_thr['thr_bacc']:.3f}) | "
            f"val_f1@thr={m_f1['f1']:.4f} (thr={val_thr['thr_f1']:.3f})"
        )

        if score > best_score + MIN_DELTA:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"Early stopping at epoch {ep} (best_score={best_score:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


# ============================================================
# Main
# ============================================================
def main():
    print(f"DEVICE: {DEVICE}")
    print(f"SSM backend: {'Mamba' if (MambaLayer is not None) else 'EMA-SSM fallback (install mamba-ssm for best speed/quality)'}")
    print(f"Windowing: WIN_SEC={WIN_SEC}s, HOP_SEC={HOP_SEC}s, FS={FS}Hz (FULL LENGTH per file)")

    files = []
    for ext in ["*.fif", "*.fif.gz", "*.set"]:
        files.extend(glob.glob(os.path.join(ROOT_DIR, "**", ext), recursive=True))
    files = sorted(files)
    if len(files) == 0:
        raise RuntimeError(f"No EEG files found in {ROOT_DIR} (.fif/.set expected).")
    print(f"Found {len(files)} EEG file(s).")

    label_map = load_labels(label_dir, files)

    subj_to_files: Dict[str, List[str]] = {}
    subj_to_label: Dict[str, int] = {}

    for f in files:
        sid = infer_subject_id(f)
        y = match_label(sid, label_map)
        if y is None:
            low = f.lower()
            if any(k in low for k in ["parkinson", "pd"]):
                y = 1
            elif any(k in low for k in ["control", "healthy", "hc", "normal"]):
                y = 0
        if y is None:
            continue
        subj_to_files.setdefault(sid, []).append(f)
        subj_to_label[sid] = int(y)

    subjects = sorted(subj_to_files.keys())
    y_subj = np.array([subj_to_label[s] for s in subjects], dtype=np.int64)
    print(f"Subjects with labels: {len(subjects)} | PD={int(y_subj.sum())} | Control={int((1-y_subj).sum())}")
    if len(subjects) < 10:
        raise RuntimeError("Too few labeled subjects after label matching.")

    # canonical channels computed from representative file per subject
    repr_files = [subj_to_files[s][0] for s in subjects]
    canonical_chs = compute_canonical_intersection(repr_files)
    print(f"Canonical EEG intersection channels: {len(canonical_chs)}")

    # cache full-length windows per subject
    print("Caching windows (subject-level cache, FULL LENGTH)...")
    subj_cache = {}
    usable_subjects = []
    usable_labels = []

    # (optional) print rough window counts to sanity check imbalance
    total_windows = 0
    for sid in tqdm(subjects, desc="Caching"):
        X, y = build_or_load_subject_cache(sid, subj_to_files[sid], subj_to_label[sid], canonical_chs)
        if X.shape[0] == 0:
            continue
        subj_cache[sid] = (X, y)
        usable_subjects.append(sid)
        usable_labels.append(subj_to_label[sid])
        total_windows += int(X.shape[0])

    subjects = usable_subjects
    y_subj = np.array(usable_labels, dtype=np.int64)
    print(f"Usable subjects after FULLLEN+windowing: {len(subjects)} | total cached windows={total_windows}")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    fold_window_metrics = []
    fold_subject_metrics = []

    all_win_true, all_win_prob = [], []
    all_sub_true, all_sub_prob = [], []

    for fold, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(subjects)), y_subj), start=1):
        tr_sids = [subjects[i] for i in tr_idx]
        te_sids = [subjects[i] for i in te_idx]

        # fold-local stratified train/val split
        val_frac = 0.15
        tr0 = [s for s in tr_sids if subj_to_label[s] == 0]
        tr1 = [s for s in tr_sids if subj_to_label[s] == 1]
        rng = np.random.RandomState(SEED + fold)
        rng.shuffle(tr0); rng.shuffle(tr1)
        v0 = tr0[: max(1, int(len(tr0) * val_frac))]
        v1 = tr1[: max(1, int(len(tr1) * val_frac))]
        val_sids = sorted(list(set(v0 + v1)))
        train_sids = [s for s in tr_sids if s not in set(val_sids)]
        if len(val_sids) < 2:
            val_sids = tr_sids[:2]
            train_sids = tr_sids[2:]

        print("\n" + "=" * 60)
        print(f"FOLD {fold}/{N_FOLDS}")
        print(f"Train subjects: {len(train_sids)} | Val subjects: {len(val_sids)} | Test subjects: {len(te_sids)}")

        # train stats (train windows only)
        X_train_list = [subj_cache[s][0] for s in train_sids]
        mean, std = compute_train_norm_stats(X_train_list)

        def collect_windows(sids: List[str]) -> Tuple[np.ndarray, np.ndarray, List[str]]:
            Xs, ys, sid_list = [], [], []
            for s in sids:
                X, y = subj_cache[s]
                Xc = X.copy()
                apply_norm_inplace(Xc, mean, std)
                Xs.append(Xc)
                ys.append(y)
                sid_list.extend([s] * Xc.shape[0])
            return np.concatenate(Xs, axis=0), np.concatenate(ys, axis=0), sid_list

        Xtr, ytr, _ = collect_windows(train_sids)
        Xva, yva, _ = collect_windows(val_sids)
        Xte, yte, te_sid_per_win = collect_windows(te_sids)

        # pos_weight from TRAIN windows only
        n_pos = int((ytr == 1).sum())
        n_neg = int((ytr == 0).sum())
        pos_weight = (n_neg / max(n_pos, 1))
        print(f"Train windows: {Xtr.shape[0]} | pos={n_pos} neg={n_neg} | pos_weight={pos_weight:.4f}")
        print(f"Windows: train={Xtr.shape[0]} val={Xva.shape[0]} test={Xte.shape[0]}")

        n_ch = Xtr.shape[1]
        train_loader = DataLoader(WindowDataset(Xtr, ytr), batch_size=BATCH, shuffle=True,  num_workers=4, pin_memory=True)
        val_loader   = DataLoader(WindowDataset(Xva, yva), batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
        test_loader  = DataLoader(WindowDataset(Xte, yte), batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

        model = LiteParkSSM(
            n_ch=n_ch,
            d_model=64,
            n_layers=4,
            dropout=0.15,
            drop_path=0.0,
            use_mamba=True,  # auto-fallback if not installed
        ).to(DEVICE)

        n_params = count_params(model)
        print(f"LiteParkSSM params: {n_params} (~{n_params/1e3:.1f}k)")

        model = train_one_fold(model, train_loader, val_loader, pos_weight=pos_weight)

        # thresholds from VAL only
        yv_true, yv_prob = eval_loader(model, val_loader)
        thrs = best_thresholds_from_val(yv_true, yv_prob)
        thr_use = thrs["thr_bacc"]
        print(f"Chosen thresholds from VAL: thr_bacc={thrs['thr_bacc']:.3f} | thr_f1={thrs['thr_f1']:.3f} | USING thr_bacc")

        # test inference
        y_true_win, y_prob_win = eval_loader(model, test_loader)

        win_m = bin_metrics(y_true_win, y_prob_win, thresh=thr_use)

        # subject aggregation (median)
        y_true_sub, y_prob_sub = aggregate_subject_probs_median(te_sid_per_win, y_true_win, y_prob_win)
        sub_m = bin_metrics(y_true_sub, y_prob_sub, thresh=thr_use)

        fold_window_metrics.append(win_m)
        fold_subject_metrics.append(sub_m)

        all_win_true.append(y_true_win)
        all_win_prob.append(y_prob_win)
        all_sub_true.append(y_true_sub)
        all_sub_prob.append(y_prob_sub)

        print("---- Fold results (TEST) ----")
        print(f"Window: acc={win_m['acc']:.4f} bacc={win_m['bacc']:.4f} prec={win_m['prec']:.4f} rec={win_m['rec']:.4f} f1={win_m['f1']:.4f} auc={win_m['auc']:.4f}")
        print(f"Subj  : acc={sub_m['acc']:.4f} bacc={sub_m['bacc']:.4f} prec={sub_m['prec']:.4f} rec={sub_m['rec']:.4f} f1={sub_m['f1']:.4f} auc={sub_m['auc']:.4f}")

    def mean_std(metric_list: List[Dict[str, float]], key: str) -> Tuple[float, float]:
        vals = np.array([m[key] for m in metric_list], dtype=np.float64)
        return float(np.nanmean(vals)), float(np.nanstd(vals))

    print("\n" + "=" * 70)
    print("=============== OVERALL (mean ± std across folds) ===============")

    for k in ["acc", "bacc", "prec", "rec", "f1", "auc"]:
        m, s = mean_std(fold_window_metrics, k)
        print(f"win_{k:>4s}: {m:.4f} ± {s:.4f}")

    for k in ["acc", "bacc", "prec", "rec", "f1", "auc"]:
        m, s = mean_std(fold_subject_metrics, k)
        print(f"subj_{k:>4s}: {m:.4f} ± {s:.4f}")

    yW = np.concatenate(all_win_true, axis=0)
    pW = np.concatenate(all_win_prob, axis=0)
    yS = np.concatenate(all_sub_true, axis=0)
    pS = np.concatenate(all_sub_prob, axis=0)

    try:
        aucW = roc_auc_score(yW, pW)
        aucS = roc_auc_score(yS, pS)
        print("\nPooled ROC-AUC (all folds concatenated):")
        print(f"  Window AUC: {aucW:.4f}")
        print(f"  Subj   AUC: {aucS:.4f}")
    except Exception:
        pass


if __name__ == "__main__":
    main()


DEVICE: cuda
SSM backend: Mamba
Windowing: WIN_SEC=10s, HOP_SEC=5s, FS=128Hz (FULL LENGTH per file)
Found 149 EEG file(s).
Subjects with labels: 149 | PD=100 | Control=49


Finding canonical EEG channel intersection:   0%|          | 0/149 [00:00<?, ?it/s]

Canonical EEG intersection channels: 60
Caching windows (subject-level cache, FULL LENGTH)...


Caching:   0%|          | 0/149 [00:00<?, ?it/s]

Usable subjects after FULLLEN+windowing: 149 | total cached windows=4570

FOLD 1/5
Train subjects: 102 | Val subjects: 17 | Test subjects: 30


...

[ep 10] loss=0.1056 | val_auc=0.7652 | val_bacc@thr=0.7520 (thr=0.230) | val_f1@thr=0.8781 (thr=0.110)
Early stopping at epoch 10 (best_score=0.8442)


Eval:   0%|          | 0/5 [00:00<?, ?it/s]

Chosen thresholds from VAL: thr_bacc=0.265 | thr_f1=0.100 | USING thr_bacc


Eval:   0%|          | 0/7 [00:00<?, ?it/s]

---- Fold results (TEST) ----
Window: acc=0.6315 bacc=0.6625 prec=0.8346 rec=0.5808 f1=0.6849 auc=0.7466
Subj  : acc=0.6552 bacc=0.6583 prec=0.8125 rec=0.6500 f1=0.7222 auc=0.7833

=============== OVERALL (mean ± std across folds) ===============
win_ acc: 0.6875 ± 0.0795
win_bacc: 0.6547 ± 0.0776
win_prec: 0.7709 ± 0.0781
win_ rec: 0.7840 ± 0.2007
win_  f1: 0.7544 ± 0.0879
win_ auc: 0.7712 ± 0.0535
subj_ acc: 0.6977 ± 0.0904
subj_bacc: 0.6467 ± 0.0873
subj_prec: 0.7754 ± 0.0631
subj_ rec: 0.8000 ± 0.1975
subj_  f1: 0.7691 ± 0.0947
subj_ auc: 0.7807 ± 0.0489

Pooled ROC-AUC (all folds concatenated):
  Window AUC: 0.6763
  Subj   AUC: 0.6751


In [12]:
#!/usr/bin/env python3
# ============================================================
# Parkinson's Resting-State EEG — Subject-wise Stratified 5-Fold CV
# Protocol:
# ✅ Subject-wise Stratified 5-fold CV (StratifiedGroupKFold)
# ✅ FULL variable-length recordings (no truncation)
# ✅ Canonical EEG channel intersection (EEG-only)
# ✅ Windowing: 10s, 50% overlap across full length
# ✅ Metrics per fold + overall:
#    Window: Acc/BAcc/Prec/Rec/F1/AUC
#    Subject: Acc/BAcc/Prec/Rec/F1/AUC
#
# New architecture (<=250k params): NeuroConvFormer-Lite
# ============================================================

import os, re, glob, math, json, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, roc_auc_score
)

import mne

# -----------------------
# USER PATHS
# -----------------------
ROOT_DIR  = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
label_dir = "/home/varun/Desktop/Parkinsons_Resting_State"

# -----------------------
# GLOBALS
# -----------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# ============================================================
# 1) LABEL LOADING (robust)
# ============================================================
def find_label_file(label_dir):
    cands = []
    for fn in ["participants.tsv", "labels.csv", "labels.tsv", "participants.csv"]:
        p = os.path.join(label_dir, fn)
        if os.path.exists(p):
            cands.append(p)
    if len(cands) > 0:
        return cands[0]
    # fallback: any csv/tsv with "label" or "group"
    for ext in ["*.csv", "*.tsv"]:
        for p in glob.glob(os.path.join(label_dir, ext)):
            cands.append(p)
    return cands[0] if len(cands) else None

def load_labels(label_dir):
    """
    Expected columns (any of these patterns):
      - participant_id / subject / sub / id
      - label / group / diagnosis / class
    Binary mapping is inferred if text labels exist.
    """
    lf = find_label_file(label_dir)
    if lf is None:
        raise FileNotFoundError(
            f"No label file found in {label_dir}. "
            "Place participants.tsv or labels.csv/tsv there."
        )

    sep = "\t" if lf.endswith(".tsv") else ","
    df = pd.read_csv(lf, sep=sep)

    # normalize colnames
    cols = {c.lower().strip(): c for c in df.columns}
    def pick_col(possibles):
        for k in possibles:
            for c_low, c_orig in cols.items():
                if c_low == k or k in c_low:
                    return c_orig
        return None

    id_col = pick_col(["participant_id", "subject", "sub", "id"])
    y_col  = pick_col(["label", "group", "diagnosis", "class"])

    if id_col is None or y_col is None:
        raise ValueError(
            f"Could not infer id/label columns from {lf}. "
            f"Columns present: {list(df.columns)}"
        )

    df = df[[id_col, y_col]].copy()
    df.rename(columns={id_col: "subject_id", y_col: "label_raw"}, inplace=True)

    # clean subject ids
    df["subject_id"] = df["subject_id"].astype(str).str.strip()

    # map labels to 0/1
    raw = df["label_raw"]
    if raw.dtype == object:
        # heuristics for PD vs HC
        low = raw.astype(str).str.lower()
        # PD positive class = 1
        pd_mask = low.str.contains("pd|parkinson|patient|case|disease", regex=True)
        hc_mask = low.str.contains("hc|control|healthy|normal", regex=True)

        # if both ambiguous, fallback to factorize
        if pd_mask.any() and (hc_mask.any() or (~pd_mask).any()):
            y = np.where(pd_mask, 1, 0).astype(int)
        else:
            y = pd.factorize(raw)[0].astype(int)
            # ensure binary
            if len(np.unique(y)) != 2:
                raise ValueError(f"Non-binary labels detected: {raw.unique()}")
    else:
        y = raw.values.astype(int)
        if len(np.unique(y)) != 2:
            raise ValueError(f"Non-binary labels detected: {np.unique(y)}")

    df["y"] = y
    return df[["subject_id", "y"]]

labels_df = load_labels(label_dir)

# ============================================================
# 2) DATA DISCOVERY + SUBJECT ID PARSING
# ============================================================
SUPPORTED_EXTS = [".set", ".edf", ".bdf", ".fif", ".vhdr"]

def discover_eeg_files(root_dir):
    files = []
    for ext in SUPPORTED_EXTS:
        files.extend(glob.glob(os.path.join(root_dir, f"**/*{ext}"), recursive=True))
    files = sorted(list(set(files)))
    if len(files) == 0:
        raise FileNotFoundError(
            f"No EEG files found in {root_dir} with extensions {SUPPORTED_EXTS}"
        )
    return files

def infer_subject_id_from_path(path):
    """
    Tries common patterns: sub-XXX, subjectXXX, or folder name match.
    """
    s = path.replace("\\", "/")
    m = re.search(r"(sub-[a-zA-Z0-9]+)", s)
    if m:
        return m.group(1)
    m = re.search(r"(subject[_-]?[a-zA-Z0-9]+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1)
    # fallback: parent folder
    return os.path.basename(os.path.dirname(path))

files = discover_eeg_files(ROOT_DIR)

# align files with labels
file_rows = []
label_map = dict(zip(labels_df["subject_id"], labels_df["y"]))

# allow minor normalization of subject keys
def normalize_sid(sid):
    sid = str(sid).strip()
    sid = sid.replace(" ", "")
    return sid

label_map_norm = {normalize_sid(k): v for k, v in label_map.items()}

for fp in files:
    sid = infer_subject_id_from_path(fp)
    y = label_map_norm.get(normalize_sid(sid), None)
    if y is None:
        # sometimes labels use just numeric id, try extracting trailing digits
        digits = re.findall(r"\d+", sid)
        if len(digits):
            y = label_map_norm.get(normalize_sid(digits[-1]), None)
    if y is None:
        continue
    file_rows.append((fp, sid, int(y)))

if len(file_rows) == 0:
    raise RuntimeError(
        "No files matched with labels. Check subject_id mapping between "
        "label file and file naming."
    )

meta_df = pd.DataFrame(file_rows, columns=["path", "subject_id", "y"])
print(f"[INFO] Matched recordings: {len(meta_df)} | Subjects: {meta_df['subject_id'].nunique()}")

# ============================================================
# 3) EEG LOADING (EEG-only) + CANONICAL INTERSECTION
# ============================================================
def load_raw(fp):
    ext = os.path.splitext(fp)[1].lower()
    if ext == ".set":
        raw = mne.io.read_raw_eeglab(fp, preload=True, verbose=False)
    elif ext == ".edf":
        raw = mne.io.read_raw_edf(fp, preload=True, verbose=False)
    elif ext == ".bdf":
        raw = mne.io.read_raw_bdf(fp, preload=True, verbose=False)
    elif ext == ".fif":
        raw = mne.io.read_raw_fif(fp, preload=True, verbose=False)
    elif ext == ".vhdr":
        raw = mne.io.read_raw_brainvision(fp, preload=True, verbose=False)
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    return raw

def eeg_only_pick(raw):
    picks = mne.pick_types(raw.info, eeg=True, eog=False, ecg=False, emg=False, stim=False, misc=False)
    if len(picks) == 0:
        raise RuntimeError("No EEG channels found in file.")
    return picks

# compute canonical EEG channel intersection
all_ch_sets = []
for fp in tqdm(meta_df["path"].tolist(), desc="Scanning channels"):
    raw = load_raw(fp)
    picks = eeg_only_pick(raw)
    chs = [raw.ch_names[i] for i in picks]
    # normalize channel names
    chs = [c.strip().upper() for c in chs]
    all_ch_sets.append(set(chs))

canon = set.intersection(*all_ch_sets)
canon = sorted(list(canon))
if len(canon) < 4:
    raise RuntimeError(f"Canonical EEG intersection too small: {len(canon)} channels: {canon}")

print(f"[INFO] Canonical EEG channels (intersection): {len(canon)}")

# ============================================================
# 4) WINDOWING (10s, 50% overlap) OVER FULL DURATION
# ============================================================
WIN_SEC = 10.0
OVERLAP = 0.50

def extract_windows_from_raw(raw, canon_chs, win_sec=WIN_SEC, overlap=OVERLAP):
    """
    Returns: X_windows (nW, C, T), sfreq
    Uses the full duration; drops the last partial window.
    """
    sfreq = float(raw.info["sfreq"])
    # pick canonical channels (case-insensitive)
    name_to_idx = {c.strip().upper(): i for i, c in enumerate(raw.ch_names)}
    picks = [name_to_idx[c] for c in canon_chs if c in name_to_idx]
    if len(picks) != len(canon_chs):
        # should not happen if intersection computed correctly, but guard anyway
        raise RuntimeError("Canonical channel mismatch during extraction.")

    data = raw.get_data(picks=picks)  # (C, N)
    C, N = data.shape

    win_len = int(round(win_sec * sfreq))
    hop = int(round(win_len * (1.0 - overlap)))
    hop = max(1, hop)

    if N < win_len:
        return np.zeros((0, C, win_len), dtype=np.float32), sfreq

    starts = np.arange(0, N - win_len + 1, hop, dtype=int)
    X = np.stack([data[:, s:s+win_len] for s in starts], axis=0).astype(np.float32)  # (W,C,T)
    return X, sfreq

# ============================================================
# 5) AUGMENTATIONS (window-level, safe + effective)
# ============================================================
class Augment:
    def __init__(self, p_noise=0.3, p_shift=0.3, p_chdrop=0.2, p_banddrop=0.2,
                 noise_std=0.01, max_shift_frac=0.1, chdrop_frac=0.1):
        self.p_noise = p_noise
        self.p_shift = p_shift
        self.p_chdrop = p_chdrop
        self.p_banddrop = p_banddrop
        self.noise_std = noise_std
        self.max_shift_frac = max_shift_frac
        self.chdrop_frac = chdrop_frac

    def __call__(self, x):
        # x: (C,T) torch
        C, T = x.shape

        if random.random() < self.p_shift:
            max_shift = int(self.max_shift_frac * T)
            if max_shift > 0:
                s = random.randint(-max_shift, max_shift)
                x = torch.roll(x, shifts=s, dims=1)

        if random.random() < self.p_noise:
            x = x + self.noise_std * torch.randn_like(x)

        if random.random() < self.p_chdrop:
            k = max(1, int(self.chdrop_frac * C))
            idx = torch.randperm(C)[:k]
            x[idx] = 0.0

        # simple "band dropout" proxy via random temporal smoothing / differencing
        # (cheap, stable, helps robustness)
        if random.random() < self.p_banddrop:
            if random.random() < 0.5:
                # smooth
                k = random.choice([3,5,7])
                pad = k//2
                x = F.avg_pool1d(x.unsqueeze(0), kernel_size=k, stride=1, padding=pad).squeeze(0)
            else:
                # high-pass-ish
                x = x - F.avg_pool1d(x.unsqueeze(0), kernel_size=7, stride=1, padding=3).squeeze(0)

        return x

# ============================================================
# 6) DATASET (cache windows per recording)
# ============================================================
class WindowDataset(torch.utils.data.Dataset):
    def __init__(self, records, canon_chs, training=False, augment=None):
        """
        records: list of dict {path, subject_id, y}
        Creates a flat list of windows with subject_id for later subject aggregation.
        """
        self.training = training
        self.augment = augment

        self.items = []  # each: (window_np, y, subject_id)
        self.sfreqs = []
        for r in tqdm(records, desc="Loading & windowing"):
            raw = load_raw(r["path"])
            Xw, sf = extract_windows_from_raw(raw, canon_chs)
            self.sfreqs.append(sf)
            if Xw.shape[0] == 0:
                continue
            for i in range(Xw.shape[0]):
                self.items.append((Xw[i], r["y"], r["subject_id"]))

        if len(self.items) == 0:
            raise RuntimeError("No windows extracted. Check data durations and sfreq.")

        # sanity: ensure consistent sampling rate
        if np.std(self.sfreqs) > 1e-6:
            print("[WARN] Sampling rate varies across recordings. Model still works (fixed window samples per file).")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        x, y, sid = self.items[idx]
        x = torch.from_numpy(x)  # (C,T)
        # per-window robust normalization (channelwise z-score)
        x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)

        if self.training and self.augment is not None:
            x = self.augment(x)

        return x, torch.tensor(y, dtype=torch.long), sid

def make_records(df):
    recs = []
    for _, row in df.iterrows():
        recs.append({"path": row["path"], "subject_id": row["subject_id"], "y": int(row["y"])})
    return recs

# ============================================================
# 7) MODEL: NeuroConvFormer-Lite (<=250k params)
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hid = max(4, ch // r)
        self.fc1 = nn.Linear(ch, hid)
        self.fc2 = nn.Linear(hid, ch)

    def forward(self, x):
        # x: (B, C, T)
        s = x.mean(dim=2)              # (B,C)
        s = F.relu(self.fc1(s))
        s = torch.sigmoid(self.fc2(s)) # (B,C)
        return x * s.unsqueeze(-1)

class NeuroConvFormerLite(nn.Module):
    """
    Input: (B, C, T)
    Output: logits (B,2)
    """
    def __init__(self, n_ch, n_time, d_model=64, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()

        # Multi-scale depthwise temporal conv
        # (depthwise conv per channel, then pointwise mixing)
        ks = [7, 15, 31]
        self.dw_convs = nn.ModuleList([
            nn.Conv1d(n_ch, n_ch, kernel_size=k, padding=k//2, groups=n_ch, bias=False)
            for k in ks
        ])
        self.pw_mix = nn.Conv1d(n_ch * len(ks), d_model, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm1d(d_model)

        self.se = SEBlock(d_model, r=8)

        # Temporal downsampling to tokens
        self.ds = nn.Conv1d(d_model, d_model, kernel_size=5, stride=2, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=2*d_model,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True
        )
        self.tr = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # attentive pooling
        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.GELU(),
            nn.Linear(d_model//2, 1)
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

        # param sanity print
        self._n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, x):
        # x: (B,C,T)
        feats = []
        for dw in self.dw_convs:
            feats.append(dw(x))
        x = torch.cat(feats, dim=1)          # (B, C*3, T)
        x = self.pw_mix(x)                   # (B, d_model, T)
        x = self.bn1(x)
        x = F.gelu(x)

        x = self.se(x)

        x = self.ds(x)                       # (B, d_model, T/2)
        x = self.bn2(x)
        x = F.gelu(x)

        # transformer expects (B, S, E)
        x = x.transpose(1, 2)                # (B, S, d_model)
        x = self.tr(x)

        # attentive pooling
        w = self.attn(x)                     # (B,S,1)
        w = torch.softmax(w, dim=1)
        z = (x * w).sum(dim=1)               # (B,d_model)

        logits = self.head(z)                # (B,2)
        return logits

# ============================================================
# 8) METRICS (window + subject)
# ============================================================
def binary_metrics(y_true, y_prob, thr=0.5):
    """
    y_true: (N,)
    y_prob: (N,) probability for class=1
    """
    y_pred = (y_prob >= thr).astype(int)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    # AUC needs both classes present
    auc = np.nan
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_prob)

    return dict(acc=acc, bacc=bacc, prec=prec, rec=rec, f1=f1, auc=auc)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_y = []
    all_p = []
    all_sid = []

    for xb, yb, sid in tqdm(loader, desc="Eval", leave=False):
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.numpy().astype(int)

        logits = model(xb)
        prob1 = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

        all_y.append(yb)
        all_p.append(prob1)
        all_sid.extend(list(sid))

    y = np.concatenate(all_y, axis=0)
    p = np.concatenate(all_p, axis=0)

    # window-level
    win = binary_metrics(y, p)

    # subject-level aggregation: log-mean (geometric mean) of probabilities
    # stable: clip to avoid log(0)
    eps = 1e-6
    df = pd.DataFrame({"sid": all_sid, "y": y, "p": p})
    subj_rows = []
    for sid, g in df.groupby("sid"):
        yt = int(g["y"].iloc[0])
        pp = np.clip(g["p"].values, eps, 1 - eps)
        # log-mean
        lp = np.mean(np.log(pp))
        p_agg = float(np.exp(lp))
        subj_rows.append((sid, yt, p_agg))

    sids, ys, ps = zip(*subj_rows)
    subj = binary_metrics(np.array(ys), np.array(ps))

    return win, subj

# ============================================================
# 9) TRAINING
# ============================================================
def train_one_fold(fold, train_df, val_df, canon_chs,
                   epochs=60, batch_size=64, lr=2e-3, weight_decay=1e-3,
                   patience=12):

    aug = Augment(
        p_noise=0.35, p_shift=0.35, p_chdrop=0.20, p_banddrop=0.25,
        noise_std=0.015, max_shift_frac=0.08, chdrop_frac=0.10
    )

    train_set = WindowDataset(make_records(train_df), canon_chs, training=True, augment=aug)
    val_set   = WindowDataset(make_records(val_df), canon_chs, training=False, augment=None)

    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=4, pin_memory=True, drop_last=True
    )
    val_loader = torch.utils.data.DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True
    )

    # infer shapes
    x0, _, _ = train_set[0]
    n_ch, n_time = x0.shape

    model = NeuroConvFormerLite(n_ch=n_ch, n_time=n_time, d_model=64, n_heads=4, n_layers=2, dropout=0.25).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[fold{fold:02d}] Model params: {n_params}")
    if n_params > 250_000:
        raise RuntimeError(f"Param budget exceeded: {n_params} > 250k")

    # class balance for pos_weight
    y_train = train_df["y"].values.astype(int)
    n0 = (y_train == 0).sum()
    n1 = (y_train == 1).sum()
    pos_weight = torch.tensor([max(1.0, n0 / max(1, n1))], device=DEVICE)

    crit = nn.CrossEntropyLoss()  # stable for 2-class with softmax

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # cosine with warmup
    total_steps = epochs * len(train_loader)
    warmup_steps = max(1, int(0.1 * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_bacc = -1
    best_state = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        losses = []

        pbar = tqdm(train_loader, desc=f"[fold{fold:02d}] Train ep{ep:03d}", leave=False)
        for xb, yb, _ in pbar:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(xb)
                loss = crit(logits, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()

            losses.append(loss.item())
            pbar.set_postfix(loss=float(np.mean(losses)), lr=opt.param_groups[0]["lr"])

        # eval each epoch (you want visibility)
        win_m, subj_m = evaluate(model, val_loader)

        print(f"[fold{fold:02d}] ep{ep:03d} | "
              f"VAL(win) acc={win_m['acc']:.4f} bacc={win_m['bacc']:.4f} f1={win_m['f1']:.4f} auc={win_m['auc'] if not np.isnan(win_m['auc']) else -1:.4f} | "
              f"VAL(subj) acc={subj_m['acc']:.4f} bacc={subj_m['bacc']:.4f} f1={subj_m['f1']:.4f} auc={subj_m['auc'] if not np.isnan(subj_m['auc']) else -1:.4f}")

        # early stop on subject bacc (more meaningful)
        if subj_m["bacc"] > best_bacc + 1e-4:
            best_bacc = subj_m["bacc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"[fold{fold:02d}] Early stop at ep{ep:03d} (best subj_bacc={best_bacc:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # final fold eval (best checkpoint)
    win_m, subj_m = evaluate(model, val_loader)
    return model, win_m, subj_m

# ============================================================
# 10) SUBJECT-WISE STRATIFIED 5-FOLD CV
# ============================================================
# one label per subject for stratification
subj_df = meta_df.groupby("subject_id").agg(
    y=("y", "first")
).reset_index()

X_dummy = np.zeros((len(subj_df), 1))
y_subj = subj_df["y"].values.astype(int)
groups = subj_df["subject_id"].values.astype(str)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_results = []
all_win_y = []
all_win_p = []
all_subj_y = []
all_subj_p = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_dummy, y_subj, groups), start=1):
    tr_sids = set(subj_df.iloc[tr_idx]["subject_id"].tolist())
    va_sids = set(subj_df.iloc[va_idx]["subject_id"].tolist())

    train_df = meta_df[meta_df["subject_id"].isin(tr_sids)].reset_index(drop=True)
    val_df   = meta_df[meta_df["subject_id"].isin(va_sids)].reset_index(drop=True)

    print("\n" + "="*70)
    print(f"[fold{fold:02d}] Train subjects: {len(tr_sids)} | Val subjects: {len(va_sids)}")
    print(f"[fold{fold:02d}] Train recs: {len(train_df)} | Val recs: {len(val_df)}")
    print("="*70)

    model, win_m, subj_m = train_one_fold(fold, train_df, val_df, canon)

    fold_results.append((win_m, subj_m))

# report fold-wise + overall mean/std
def summarize(results, keyset=("acc","bacc","prec","rec","f1","auc")):
    arr = {k: [] for k in keyset}
    for m in results:
        for k in keyset:
            arr[k].append(m[k] if not (isinstance(m[k], float) and np.isnan(m[k])) else np.nan)
    out = {}
    for k, v in arr.items():
        vv = np.array(v, dtype=float)
        out[k] = (np.nanmean(vv), np.nanstd(vv))
    return out

win_folds  = [r[0] for r in fold_results]
subj_folds = [r[1] for r in fold_results]

win_sum  = summarize(win_folds)
subj_sum = summarize(subj_folds)

print("\n" + "#"*72)
print("FOLD-WISE RESULTS:")
for i, (w, s) in enumerate(fold_results, start=1):
    print(f"[fold{i:02d}] WIN  acc={w['acc']:.4f} bacc={w['bacc']:.4f} prec={w['prec']:.4f} rec={w['rec']:.4f} f1={w['f1']:.4f} auc={w['auc'] if not np.isnan(w['auc']) else -1:.4f}")
    print(f"         SUBJ acc={s['acc']:.4f} bacc={s['bacc']:.4f} prec={s['prec']:.4f} rec={s['rec']:.4f} f1={s['f1']:.4f} auc={s['auc'] if not np.isnan(s['auc']) else -1:.4f}")

print("\n" + "#"*72)
print("OVERALL (mean ± std across folds):")
print(f"WIN : acc={win_sum['acc'][0]:.4f}±{win_sum['acc'][1]:.4f} | "
      f"bacc={win_sum['bacc'][0]:.4f}±{win_sum['bacc'][1]:.4f} | "
      f"prec={win_sum['prec'][0]:.4f}±{win_sum['prec'][1]:.4f} | "
      f"rec={win_sum['rec'][0]:.4f}±{win_sum['rec'][1]:.4f} | "
      f"f1={win_sum['f1'][0]:.4f}±{win_sum['f1'][1]:.4f} | "
      f"auc={win_sum['auc'][0]:.4f}±{win_sum['auc'][1]:.4f}")
print(f"SUBJ: acc={subj_sum['acc'][0]:.4f}±{subj_sum['acc'][1]:.4f} | "
      f"bacc={subj_sum['bacc'][0]:.4f}±{subj_sum['bacc'][1]:.4f} | "
      f"prec={subj_sum['prec'][0]:.4f}±{subj_sum['prec'][1]:.4f} | "
      f"rec={subj_sum['rec'][0]:.4f}±{subj_sum['rec'][1]:.4f} | "
      f"f1={subj_sum['f1'][0]:.4f}±{subj_sum['f1'][1]:.4f} | "
      f"auc={subj_sum['auc'][0]:.4f}±{subj_sum['auc'][1]:.4f}")
print("#"*72)


[INFO] Matched recordings: 149 | Subjects: 149


Scanning channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [00:02<00:00, 58.34it/s]


[INFO] Canonical EEG channels (intersection): 60

[fold01] Train subjects: 119 | Val subjects: 30
[fold01] Train recs: 119 | Val recs: 30


Loading & windowing: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 46.42it/s]


[fold01] Model params: 110007


...

[fold05] ep027 | VAL(win) acc=0.5062 bacc=0.4933 f1=0.5851 auc=0.5362 | VAL(subj) acc=0.4828 bacc=0.5333 f1=0.5161 auc=0.5278


[fold05] ep028 | VAL(win) acc=0.5351 bacc=0.4772 f1=0.6522 auc=0.5700 | VAL(subj) acc=0.5517 bacc=0.5528 f1=0.6286 auc=0.5556


[fold05] ep029 | VAL(win) acc=0.5331 bacc=0.4816 f1=0.6452 auc=0.5824 | VAL(subj) acc=0.5862 bacc=0.5778 f1=0.6667 auc=0.5889


[fold05] ep030 | VAL(win) acc=0.5372 bacc=0.5266 f1=0.6118 auc=0.5745 | VAL(subj) acc=0.5517 bacc=0.6139 f1=0.5806 auc=0.5500
[fold05] Early stop at ep030 (best subj_bacc=0.6889)



########################################################################
FOLD-WISE RESULTS:
[fold01] WIN  acc=0.7348 bacc=0.7539 prec=0.8714 rec=0.6835 f1=0.7661 auc=0.8434
         SUBJ acc=0.7333 bacc=0.7750 prec=0.9286 rec=0.6500 f1=0.7647 auc=0.8450
[fold02] WIN  acc=0.6563 bacc=0.7021 prec=0.8757 rec=0.5456 f1=0.6723 auc=0.7328
         SUBJ acc=0.6333 bacc=0.7250 prec=1.0000 rec=0.4500 f1=0.6207 auc=0.7250
[fold03] WIN  acc=0.8013 bacc=0.7502 prec=0.8023 rec=0.9210 f1=0.8576 auc=0.8268
         SUBJ acc=0.8333 bacc=0.8000 prec=0.8571 rec=0.9000 f1=0.8780 auc=0.8100
[fold04] WIN  acc=0.7789 bacc=0.7080 prec=0.7912 rec=0.9125 f1=0.8475 auc=0.7516
         SUBJ acc=0.8000 bacc=0.7500 prec=0.8182 rec=0.9000 f1=0.8571 auc=0.7600
[fold05] WIN  acc=0.5754 bacc=0.5554 prec=0.6920 rec=0.6226 f1=0.6555 auc=0.6164
         SUBJ acc=0.6552 bacc=0.6889 prec=0.8571 rec=0.6000 f1=0.7059 auc=0.6167

########################################################################
OVERALL (mean ± std acr

In [13]:
#!/usr/bin/env python3
# ============================================================
# Parkinson's Resting-State EEG — Subject-wise Stratified 5-Fold CV
# Protocol:
# ✅ Subject-wise Stratified 5-fold CV (StratifiedGroupKFold)
# ✅ FULL variable-length recordings (no truncation)
# ✅ Canonical EEG channel intersection (EEG-only)
# ✅ Windowing: 10s, 50% overlap across full length
# ✅ Metrics per fold + overall:
#    Window: Acc/BAcc/Prec/Rec/F1/AUC
#    Subject: Acc/BAcc/Prec/Rec/F1/AUC
#
# New architecture (<=250k params): NeuroConvFormer-Lite
#
# UPGRADE (requested): MIL subject-level training
# - Bag sampling (N windows/subject per step)
# - Attention pooling over windows -> subject logits
# - Focal / weighted CE for stability
# - Confidence-trimmed inference for subject aggregation
# ============================================================

import os, re, glob, math, json, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, roc_auc_score
)

import mne

# -----------------------
# USER PATHS
# -----------------------
ROOT_DIR  = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
label_dir = "/home/varun/Desktop/Parkinsons_Resting_State"

# -----------------------
# GLOBALS
# -----------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# ============================================================
# 1) LABEL LOADING (robust)
# ============================================================
def find_label_file(label_dir):
    cands = []
    for fn in ["participants.tsv", "labels.csv", "labels.tsv", "participants.csv"]:
        p = os.path.join(label_dir, fn)
        if os.path.exists(p):
            cands.append(p)
    if len(cands) > 0:
        return cands[0]
    # fallback: any csv/tsv with "label" or "group"
    for ext in ["*.csv", "*.tsv"]:
        for p in glob.glob(os.path.join(label_dir, ext)):
            cands.append(p)
    return cands[0] if len(cands) else None

def load_labels(label_dir):
    """
    Expected columns (any of these patterns):
      - participant_id / subject / sub / id
      - label / group / diagnosis / class
    Binary mapping is inferred if text labels exist.
    """
    lf = find_label_file(label_dir)
    if lf is None:
        raise FileNotFoundError(
            f"No label file found in {label_dir}. "
            "Place participants.tsv or labels.csv/tsv there."
        )

    sep = "\t" if lf.endswith(".tsv") else ","
    df = pd.read_csv(lf, sep=sep)

    # normalize colnames
    cols = {c.lower().strip(): c for c in df.columns}
    def pick_col(possibles):
        for k in possibles:
            for c_low, c_orig in cols.items():
                if c_low == k or k in c_low:
                    return c_orig
        return None

    id_col = pick_col(["participant_id", "subject", "sub", "id"])
    y_col  = pick_col(["label", "group", "diagnosis", "class"])

    if id_col is None or y_col is None:
        raise ValueError(
            f"Could not infer id/label columns from {lf}. "
            f"Columns present: {list(df.columns)}"
        )

    df = df[[id_col, y_col]].copy()
    df.rename(columns={id_col: "subject_id", y_col: "label_raw"}, inplace=True)

    # clean subject ids
    df["subject_id"] = df["subject_id"].astype(str).str.strip()

    # map labels to 0/1
    raw = df["label_raw"]
    if raw.dtype == object:
        # heuristics for PD vs HC
        low = raw.astype(str).str.lower()
        # PD positive class = 1
        pd_mask = low.str.contains("pd|parkinson|patient|case|disease", regex=True)
        hc_mask = low.str.contains("hc|control|healthy|normal", regex=True)

        # if both ambiguous, fallback to factorize
        if pd_mask.any() and (hc_mask.any() or (~pd_mask).any()):
            y = np.where(pd_mask, 1, 0).astype(int)
        else:
            y = pd.factorize(raw)[0].astype(int)
            # ensure binary
            if len(np.unique(y)) != 2:
                raise ValueError(f"Non-binary labels detected: {raw.unique()}")
    else:
        y = raw.values.astype(int)
        if len(np.unique(y)) != 2:
            raise ValueError(f"Non-binary labels detected: {np.unique(y)}")

    df["y"] = y
    return df[["subject_id", "y"]]

labels_df = load_labels(label_dir)

# ============================================================
# 2) DATA DISCOVERY + SUBJECT ID PARSING
# ============================================================
SUPPORTED_EXTS = [".set", ".edf", ".bdf", ".fif", ".vhdr"]

def discover_eeg_files(root_dir):
    files = []
    for ext in SUPPORTED_EXTS:
        files.extend(glob.glob(os.path.join(root_dir, f"**/*{ext}"), recursive=True))
    files = sorted(list(set(files)))
    if len(files) == 0:
        raise FileNotFoundError(
            f"No EEG files found in {root_dir} with extensions {SUPPORTED_EXTS}"
        )
    return files

def infer_subject_id_from_path(path):
    """
    Tries common patterns: sub-XXX, subjectXXX, or folder name match.
    """
    s = path.replace("\\", "/")
    m = re.search(r"(sub-[a-zA-Z0-9]+)", s)
    if m:
        return m.group(1)
    m = re.search(r"(subject[_-]?[a-zA-Z0-9]+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1)
    # fallback: parent folder
    return os.path.basename(os.path.dirname(path))

files = discover_eeg_files(ROOT_DIR)

# align files with labels
file_rows = []
label_map = dict(zip(labels_df["subject_id"], labels_df["y"]))

# allow minor normalization of subject keys
def normalize_sid(sid):
    sid = str(sid).strip()
    sid = sid.replace(" ", "")
    return sid

label_map_norm = {normalize_sid(k): v for k, v in label_map.items()}

for fp in files:
    sid = infer_subject_id_from_path(fp)
    y = label_map_norm.get(normalize_sid(sid), None)
    if y is None:
        # sometimes labels use just numeric id, try extracting trailing digits
        digits = re.findall(r"\d+", sid)
        if len(digits):
            y = label_map_norm.get(normalize_sid(digits[-1]), None)
    if y is None:
        continue
    file_rows.append((fp, sid, int(y)))

if len(file_rows) == 0:
    raise RuntimeError(
        "No files matched with labels. Check subject_id mapping between "
        "label file and file naming."
    )

meta_df = pd.DataFrame(file_rows, columns=["path", "subject_id", "y"])
print(f"[INFO] Matched recordings: {len(meta_df)} | Subjects: {meta_df['subject_id'].nunique()}")

# ============================================================
# 3) EEG LOADING (EEG-only) + CANONICAL INTERSECTION
# ============================================================
def load_raw(fp):
    ext = os.path.splitext(fp)[1].lower()
    if ext == ".set":
        raw = mne.io.read_raw_eeglab(fp, preload=True, verbose=False)
    elif ext == ".edf":
        raw = mne.io.read_raw_edf(fp, preload=True, verbose=False)
    elif ext == ".bdf":
        raw = mne.io.read_raw_bdf(fp, preload=True, verbose=False)
    elif ext == ".fif":
        raw = mne.io.read_raw_fif(fp, preload=True, verbose=False)
    elif ext == ".vhdr":
        raw = mne.io.read_raw_brainvision(fp, preload=True, verbose=False)
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    return raw

def eeg_only_pick(raw):
    picks = mne.pick_types(raw.info, eeg=True, eog=False, ecg=False, emg=False, stim=False, misc=False)
    if len(picks) == 0:
        raise RuntimeError("No EEG channels found in file.")
    return picks

# compute canonical EEG channel intersection
all_ch_sets = []
for fp in tqdm(meta_df["path"].tolist(), desc="Scanning channels"):
    raw = load_raw(fp)
    picks = eeg_only_pick(raw)
    chs = [raw.ch_names[i] for i in picks]
    # normalize channel names
    chs = [c.strip().upper() for c in chs]
    all_ch_sets.append(set(chs))

canon = set.intersection(*all_ch_sets)
canon = sorted(list(canon))
if len(canon) < 4:
    raise RuntimeError(f"Canonical EEG intersection too small: {len(canon)} channels: {canon}")

print(f"[INFO] Canonical EEG channels (intersection): {len(canon)}")

# ============================================================
# 4) WINDOWING (10s, 50% overlap) OVER FULL DURATION
# ============================================================
WIN_SEC = 10.0
OVERLAP = 0.50

def extract_windows_from_raw(raw, canon_chs, win_sec=WIN_SEC, overlap=OVERLAP):
    """
    Returns: X_windows (nW, C, T), sfreq
    Uses the full duration; drops the last partial window.
    """
    sfreq = float(raw.info["sfreq"])
    # pick canonical channels (case-insensitive)
    name_to_idx = {c.strip().upper(): i for i, c in enumerate(raw.ch_names)}
    picks = [name_to_idx[c] for c in canon_chs if c in name_to_idx]
    if len(picks) != len(canon_chs):
        # should not happen if intersection computed correctly, but guard anyway
        raise RuntimeError("Canonical channel mismatch during extraction.")

    data = raw.get_data(picks=picks)  # (C, N)
    C, N = data.shape

    win_len = int(round(win_sec * sfreq))
    hop = int(round(win_len * (1.0 - overlap)))
    hop = max(1, hop)

    if N < win_len:
        return np.zeros((0, C, win_len), dtype=np.float32), sfreq

    starts = np.arange(0, N - win_len + 1, hop, dtype=int)
    X = np.stack([data[:, s:s+win_len] for s in starts], axis=0).astype(np.float32)  # (W,C,T)
    return X, sfreq

# ============================================================
# 5) AUGMENTATIONS (window-level, safe + effective)
# ============================================================
class Augment:
    def __init__(self, p_noise=0.3, p_shift=0.3, p_chdrop=0.2, p_banddrop=0.2,
                 noise_std=0.01, max_shift_frac=0.1, chdrop_frac=0.1):
        self.p_noise = p_noise
        self.p_shift = p_shift
        self.p_chdrop = p_chdrop
        self.p_banddrop = p_banddrop
        self.noise_std = noise_std
        self.max_shift_frac = max_shift_frac
        self.chdrop_frac = chdrop_frac

    def __call__(self, x):
        # x: (C,T) torch
        C, T = x.shape

        if random.random() < self.p_shift:
            max_shift = int(self.max_shift_frac * T)
            if max_shift > 0:
                s = random.randint(-max_shift, max_shift)
                x = torch.roll(x, shifts=s, dims=1)

        if random.random() < self.p_noise:
            x = x + self.noise_std * torch.randn_like(x)

        if random.random() < self.p_chdrop:
            k = max(1, int(self.chdrop_frac * C))
            idx = torch.randperm(C)[:k]
            x[idx] = 0.0

        # simple "band dropout" proxy via random temporal smoothing / differencing
        # (cheap, stable, helps robustness)
        if random.random() < self.p_banddrop:
            if random.random() < 0.5:
                # smooth
                k = random.choice([3,5,7])
                pad = k//2
                x = F.avg_pool1d(x.unsqueeze(0), kernel_size=k, stride=1, padding=pad).squeeze(0)
            else:
                # high-pass-ish
                x = x - F.avg_pool1d(x.unsqueeze(0), kernel_size=7, stride=1, padding=3).squeeze(0)

        return x

# ============================================================
# 6) DATASET (cache windows per recording)
# ============================================================
class WindowDataset(torch.utils.data.Dataset):
    def __init__(self, records, canon_chs, training=False, augment=None):
        """
        records: list of dict {path, subject_id, y}
        Creates a flat list of windows with subject_id for later subject aggregation.
        """
        self.training = training
        self.augment = augment

        self.items = []  # each: (window_np, y, subject_id)
        self.sfreqs = []
        for r in tqdm(records, desc="Loading & windowing"):
            raw = load_raw(r["path"])
            Xw, sf = extract_windows_from_raw(raw, canon_chs)
            self.sfreqs.append(sf)
            if Xw.shape[0] == 0:
                continue
            for i in range(Xw.shape[0]):
                self.items.append((Xw[i], r["y"], r["subject_id"]))

        if len(self.items) == 0:
            raise RuntimeError("No windows extracted. Check data durations and sfreq.")

        # sanity: ensure consistent sampling rate
        if np.std(self.sfreqs) > 1e-6:
            print("[WARN] Sampling rate varies across recordings. Model still works (fixed window samples per file).")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        x, y, sid = self.items[idx]
        x = torch.from_numpy(x)  # (C,T)
        # per-window robust normalization (channelwise z-score)
        x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)

        if self.training and self.augment is not None:
            x = self.augment(x)

        return x, torch.tensor(y, dtype=torch.long), sid

class SubjectBagDataset(torch.utils.data.Dataset):
    def __init__(self, records, canon_chs, bag_size=16, training=False, augment=None):
        """
        records: list of dict {path, subject_id, y}
        Returns per-subject bags of windows: (bag_size, C, T), y, subject_id
        """
        self.training = training
        self.augment = augment
        self.bag_size = int(bag_size)

        # group paths by subject (one label per subject assumed)
        subj_map = {}
        for r in records:
            sid = r["subject_id"]
            if sid not in subj_map:
                subj_map[sid] = {"y": int(r["y"]), "paths": []}
            subj_map[sid]["paths"].append(r["path"])

        self.subject_ids = sorted(list(subj_map.keys()))
        self.subj_y = {sid: subj_map[sid]["y"] for sid in self.subject_ids}

        # cache windows per subject
        self.subj_windows = {}
        self.sfreqs = []
        for sid in tqdm(self.subject_ids, desc="Loading & windowing (MIL bags)"):
            all_w = []
            for fp in subj_map[sid]["paths"]:
                raw = load_raw(fp)
                Xw, sf = extract_windows_from_raw(raw, canon_chs)
                self.sfreqs.append(sf)
                if Xw.shape[0] > 0:
                    all_w.append(Xw)
            if len(all_w) == 0:
                continue
            Xcat = np.concatenate(all_w, axis=0).astype(np.float32)  # (W,C,T)
            self.subj_windows[sid] = Xcat

        # filter subjects with no windows
        self.subject_ids = [sid for sid in self.subject_ids if sid in self.subj_windows]
        if len(self.subject_ids) == 0:
            raise RuntimeError("No subject bags created. Check data durations and sfreq.")

        if np.std(self.sfreqs) > 1e-6:
            print("[WARN] Sampling rate varies across recordings. Model still works (fixed window samples per file).")

    def __len__(self):
        return len(self.subject_ids)

    def __getitem__(self, idx):
        sid = self.subject_ids[idx]
        y = self.subj_y[sid]
        X = self.subj_windows[sid]  # (W,C,T)
        W = X.shape[0]

        # sample bag_size windows (with replacement if needed)
        if W >= self.bag_size:
            sel = np.random.choice(W, size=self.bag_size, replace=False)
        else:
            sel = np.random.choice(W, size=self.bag_size, replace=True)

        Xb = X[sel]  # (BAG,C,T)
        xb = torch.from_numpy(Xb)  # float32

        # per-window normalization + optional augmentation
        # xb: (BAG,C,T)
        xb = (xb - xb.mean(dim=2, keepdim=True)) / (xb.std(dim=2, keepdim=True) + 1e-6)

        if self.training and self.augment is not None:
            # apply per-window
            out = []
            for i in range(xb.shape[0]):
                out.append(self.augment(xb[i]))
            xb = torch.stack(out, dim=0)

        return xb, torch.tensor(y, dtype=torch.long), sid

def make_records(df):
    recs = []
    for _, row in df.iterrows():
        recs.append({"path": row["path"], "subject_id": row["subject_id"], "y": int(row["y"])})
    return recs

# ============================================================
# 7) MODEL: NeuroConvFormer-Lite (<=250k params)
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hid = max(4, ch // r)
        self.fc1 = nn.Linear(ch, hid)
        self.fc2 = nn.Linear(hid, ch)

    def forward(self, x):
        # x: (B, C, T)
        s = x.mean(dim=2)              # (B,C)
        s = F.relu(self.fc1(s))
        s = torch.sigmoid(self.fc2(s)) # (B,C)
        return x * s.unsqueeze(-1)

class NeuroConvFormerLite(nn.Module):
    """
    Input: (B, C, T)
    Output: logits (B,2)

    Also exposes forward_features() to support MIL subject pooling.
    """
    def __init__(self, n_ch, n_time, d_model=64, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()

        ks = [7, 15, 31]
        self.dw_convs = nn.ModuleList([
            nn.Conv1d(n_ch, n_ch, kernel_size=k, padding=k//2, groups=n_ch, bias=False)
            for k in ks
        ])
        self.pw_mix = nn.Conv1d(n_ch * len(ks), d_model, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm1d(d_model)

        self.se = SEBlock(d_model, r=8)

        self.ds = nn.Conv1d(d_model, d_model, kernel_size=5, stride=2, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=2*d_model,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True
        )
        self.tr = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.GELU(),
            nn.Linear(d_model//2, 1)
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

        self._n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward_features(self, x):
        # x: (B,C,T) -> z: (B, d_model)
        feats = []
        for dw in self.dw_convs:
            feats.append(dw(x))
        x = torch.cat(feats, dim=1)          # (B, C*3, T)
        x = self.pw_mix(x)                   # (B, d_model, T)
        x = self.bn1(x)
        x = F.gelu(x)

        x = self.se(x)

        x = self.ds(x)                       # (B, d_model, T/2)
        x = self.bn2(x)
        x = F.gelu(x)

        x = x.transpose(1, 2)                # (B, S, d_model)
        x = self.tr(x)

        w = self.attn(x)                     # (B,S,1)
        w = torch.softmax(w, dim=1)
        z = (x * w).sum(dim=1)               # (B,d_model)
        return z

    def forward(self, x):
        z = self.forward_features(x)
        logits = self.head(z)
        return logits

class MILPool(nn.Module):
    """
    Attention pooling across windows in a subject bag.
    Input:  (B, W, D)
    Output: (B, D)
    """
    def __init__(self, d_model=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.GELU(),
            nn.Linear(d_model//2, 1)
        )

    def forward(self, z):
        a = self.net(z)          # (B,W,1)
        a = torch.softmax(a, dim=1)
        z_subj = (z * a).sum(dim=1)
        return z_subj

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = float(gamma)

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()

# ============================================================
# 8) METRICS (window + subject)
# ============================================================
def binary_metrics(y_true, y_prob, thr=0.5):
    """
    y_true: (N,)
    y_prob: (N,) probability for class=1
    """
    y_pred = (y_prob >= thr).astype(int)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    auc = np.nan
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_prob)

    return dict(acc=acc, bacc=bacc, prec=prec, rec=rec, f1=f1, auc=auc)

@torch.no_grad()
def evaluate(model, loader, keep_frac=0.60):
    """
    keep_frac: confidence trimming for subject aggregation (top fraction by confidence).
              keep_frac=1.0 recovers the original "use all windows" behavior.
    """
    model.eval()
    all_y = []
    all_p = []
    all_sid = []

    for xb, yb, sid in tqdm(loader, desc="Eval", leave=False):
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.numpy().astype(int)

        logits = model(xb)
        prob1 = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

        all_y.append(yb)
        all_p.append(prob1)
        all_sid.extend(list(sid))

    y = np.concatenate(all_y, axis=0)
    p = np.concatenate(all_p, axis=0)

    # window-level
    win = binary_metrics(y, p)

    # subject-level aggregation: confidence-trimmed log-mean (geometric mean)
    eps = 1e-6
    df = pd.DataFrame({"sid": all_sid, "y": y, "p": p})
    subj_rows = []
    for sid, g in df.groupby("sid"):
        yt = int(g["y"].iloc[0])
        pp = np.clip(g["p"].values, eps, 1 - eps)

        # confidence = distance from 0.5
        conf = np.abs(pp - 0.5)
        k = max(1, int(round(float(keep_frac) * len(pp))))
        idx = np.argsort(-conf)[:k]
        pp = pp[idx]

        lp = np.mean(np.log(pp))
        p_agg = float(np.exp(lp))
        subj_rows.append((sid, yt, p_agg))

    sids, ys, ps = zip(*subj_rows)
    subj = binary_metrics(np.array(ys), np.array(ps))

    return win, subj

# ============================================================
# 9) TRAINING
# ============================================================
def train_one_fold(fold, train_df, val_df, canon_chs,
                   epochs=60, batch_size=64, lr=2e-3, weight_decay=1e-3,
                   patience=12):

    aug = Augment(
        p_noise=0.35, p_shift=0.35, p_chdrop=0.20, p_banddrop=0.25,
        noise_std=0.015, max_shift_frac=0.08, chdrop_frac=0.10
    )

    # window datasets (for eval metrics exactly as before)
    train_set_win = WindowDataset(make_records(train_df), canon_chs, training=True, augment=aug)
    val_set       = WindowDataset(make_records(val_df), canon_chs, training=False, augment=None)

    # MIL subject bag dataset (for training objective aligned to subject metrics)
    # NOTE: batch_size in this MIL loader is "subjects per batch"
    BAG_SIZE = 16
    train_set_bag = SubjectBagDataset(make_records(train_df), canon_chs, bag_size=BAG_SIZE, training=True, augment=aug)

    # MIL loader: subjects per batch (reduce if CUDA OOM)
    mil_batch_size = max(1, min(16, batch_size // 4))

    train_loader_bag = torch.utils.data.DataLoader(
        train_set_bag, batch_size=mil_batch_size, shuffle=True,
        num_workers=4, pin_memory=True, drop_last=True
    )

    # Eval loader stays window-level
    val_loader = torch.utils.data.DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True
    )

    # infer shapes
    x0, _, _ = train_set_win[0]
    n_ch, n_time = x0.shape

    model = NeuroConvFormerLite(n_ch=n_ch, n_time=n_time, d_model=64, n_heads=4, n_layers=2, dropout=0.25).to(DEVICE)
    pool = MILPool(d_model=64).to(DEVICE)

    n_params = sum(p.numel() for p in list(model.parameters()) + list(pool.parameters()) if p.requires_grad)
    print(f"[fold{fold:02d}] Model params: {n_params}")
    if n_params > 250_000:
        raise RuntimeError(f"Param budget exceeded: {n_params} > 250k")

    # class balance (use subject labels for weighting)
    subj_train = train_df.groupby("subject_id").agg(y=("y", "first")).reset_index()
    y_train = subj_train["y"].values.astype(int)
    n0 = (y_train == 0).sum()
    n1 = (y_train == 1).sum()
    pos_weight = torch.tensor([max(1.0, n0 / max(1, n1))], device=DEVICE)

    # class weights for CE/Focal (normalized)
    w0 = (n0 + n1) / max(1, 2 * n0)
    w1 = (n0 + n1) / max(1, 2 * n1)
    class_w = torch.tensor([w0, w1], dtype=torch.float32, device=DEVICE)

    # Focal loss (weighted)
    crit = FocalLoss(weight=class_w, gamma=2.0)

    opt = torch.optim.AdamW(
        list(model.parameters()) + list(pool.parameters()),
        lr=lr, weight_decay=weight_decay
    )

    total_steps = epochs * len(train_loader_bag)
    warmup_steps = max(1, int(0.1 * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_bacc = -1
    best_state = None
    best_state_pool = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        pool.train()
        losses = []

        pbar = tqdm(train_loader_bag, desc=f"[fold{fold:02d}] MIL Train ep{ep:03d}", leave=False)
        for xb_bag, yb, _ in pbar:
            # xb_bag: (Bsubj, BAG, C, T)
            xb_bag = xb_bag.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            Bsubj, BAG, C, T = xb_bag.shape
            xb_flat = xb_bag.view(Bsubj * BAG, C, T)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                z = model.forward_features(xb_flat)                 # (Bsubj*BAG, D)
                z = z.view(Bsubj, BAG, -1)                          # (Bsubj, BAG, D)
                z_subj = pool(z)                                     # (Bsubj, D)
                logits_subj = model.head(z_subj)                     # (Bsubj, 2)
                loss = crit(logits_subj, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(pool.parameters()), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()

            losses.append(loss.item())
            pbar.set_postfix(loss=float(np.mean(losses)), lr=opt.param_groups[0]["lr"])

        # eval each epoch (window + subject metrics)
        win_m, subj_m = evaluate(model, val_loader, keep_frac=0.60)

        print(f"[fold{fold:02d}] ep{ep:03d} | "
              f"VAL(win) acc={win_m['acc']:.4f} bacc={win_m['bacc']:.4f} f1={win_m['f1']:.4f} auc={win_m['auc'] if not np.isnan(win_m['auc']) else -1:.4f} | "
              f"VAL(subj) acc={subj_m['acc']:.4f} bacc={subj_m['bacc']:.4f} f1={subj_m['f1']:.4f} auc={subj_m['auc'] if not np.isnan(subj_m['auc']) else -1:.4f}")

        # early stop on subject bacc
        if subj_m["bacc"] > best_bacc + 1e-4:
            best_bacc = subj_m["bacc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_state_pool = {k: v.detach().cpu().clone() for k, v in pool.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"[fold{fold:02d}] Early stop at ep{ep:03d} (best subj_bacc={best_bacc:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    if best_state_pool is not None:
        pool.load_state_dict(best_state_pool)

    # final fold eval (best checkpoint)
    win_m, subj_m = evaluate(model, val_loader, keep_frac=0.60)
    return model, win_m, subj_m

# ============================================================
# 10) SUBJECT-WISE STRATIFIED 5-FOLD CV
# ============================================================
subj_df = meta_df.groupby("subject_id").agg(
    y=("y", "first")
).reset_index()

X_dummy = np.zeros((len(subj_df), 1))
y_subj = subj_df["y"].values.astype(int)
groups = subj_df["subject_id"].values.astype(str)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_results = []
all_win_y = []
all_win_p = []
all_subj_y = []
all_subj_p = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_dummy, y_subj, groups), start=1):
    tr_sids = set(subj_df.iloc[tr_idx]["subject_id"].tolist())
    va_sids = set(subj_df.iloc[va_idx]["subject_id"].tolist())

    train_df = meta_df[meta_df["subject_id"].isin(tr_sids)].reset_index(drop=True)
    val_df   = meta_df[meta_df["subject_id"].isin(va_sids)].reset_index(drop=True)

    print("\n" + "="*70)
    print(f"[fold{fold:02d}] Train subjects: {len(tr_sids)} | Val subjects: {len(va_sids)}")
    print(f"[fold{fold:02d}] Train recs: {len(train_df)} | Val recs: {len(val_df)}")
    print("="*70)

    model, win_m, subj_m = train_one_fold(fold, train_df, val_df, canon)

    fold_results.append((win_m, subj_m))

def summarize(results, keyset=("acc","bacc","prec","rec","f1","auc")):
    arr = {k: [] for k in keyset}
    for m in results:
        for k in keyset:
            arr[k].append(m[k] if not (isinstance(m[k], float) and np.isnan(m[k])) else np.nan)
    out = {}
    for k, v in arr.items():
        vv = np.array(v, dtype=float)
        out[k] = (np.nanmean(vv), np.nanstd(vv))
    return out

win_folds  = [r[0] for r in fold_results]
subj_folds = [r[1] for r in fold_results]

win_sum  = summarize(win_folds)
subj_sum = summarize(subj_folds)

print("\n" + "#"*72)
print("FOLD-WISE RESULTS:")
for i, (w, s) in enumerate(fold_results, start=1):
    print(f"[fold{i:02d}] WIN  acc={w['acc']:.4f} bacc={w['bacc']:.4f} prec={w['prec']:.4f} rec={w['rec']:.4f} f1={w['f1']:.4f} auc={w['auc'] if not np.isnan(w['auc']) else -1:.4f}")
    print(f"         SUBJ acc={s['acc']:.4f} bacc={s['bacc']:.4f} prec={s['prec']:.4f} rec={s['rec']:.4f} f1={s['f1']:.4f} auc={s['auc'] if not np.isnan(s['auc']) else -1:.4f}")

print("\n" + "#"*72)
print("OVERALL (mean ± std across folds):")
print(f"WIN : acc={win_sum['acc'][0]:.4f}±{win_sum['acc'][1]:.4f} | "
      f"bacc={win_sum['bacc'][0]:.4f}±{win_sum['bacc'][1]:.4f} | "
      f"prec={win_sum['prec'][0]:.4f}±{win_sum['prec'][1]:.4f} | "
      f"rec={win_sum['rec'][0]:.4f}±{win_sum['rec'][1]:.4f} | "
      f"f1={win_sum['f1'][0]:.4f}±{win_sum['f1'][1]:.4f} | "
      f"auc={win_sum['auc'][0]:.4f}±{win_sum['auc'][1]:.4f}")
print(f"SUBJ: acc={subj_sum['acc'][0]:.4f}±{subj_sum['acc'][1]:.4f} | "
      f"bacc={subj_sum['bacc'][0]:.4f}±{subj_sum['bacc'][1]:.4f} | "
      f"prec={subj_sum['prec'][0]:.4f}±{subj_sum['prec'][1]:.4f} | "
      f"rec={subj_sum['rec'][0]:.4f}±{subj_sum['rec'][1]:.4f} | "
      f"f1={subj_sum['f1'][0]:.4f}±{subj_sum['f1'][1]:.4f} | "
      f"auc={subj_sum['auc'][0]:.4f}±{subj_sum['auc'][1]:.4f}")
print("#"*72)


[INFO] Matched recordings: 149 | Subjects: 149


Scanning channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [00:02<00:00, 57.89it/s]


[INFO] Canonical EEG channels (intersection): 60

[fold01] Train subjects: 119 | Val subjects: 30
[fold01] Train recs: 119 | Val recs: 30


Loading & windowing (MIL bags): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 119/119 [00:03<00:00, 38.78it/s]


[fold01] Model params: 112120


...

[fold05] ep016 | VAL(win) acc=0.5475 bacc=0.5764 f1=0.5788 auc=0.6498 | VAL(subj) acc=0.5172 bacc=0.6500 f1=0.4615 auc=0.6167


[fold05] ep017 | VAL(win) acc=0.5899 bacc=0.6435 f1=0.5945 auc=0.7113 | VAL(subj) acc=0.5172 bacc=0.6500 f1=0.4615 auc=0.7500


[fold05] ep018 | VAL(win) acc=0.6498 bacc=0.5554 f1=0.7638 auc=0.6588 | VAL(subj) acc=0.6897 bacc=0.5611 f1=0.8000 auc=0.6778


[fold05] ep019 | VAL(win) acc=0.6219 bacc=0.5629 f1=0.7231 auc=0.6619 | VAL(subj) acc=0.6897 bacc=0.6222 f1=0.7805 auc=0.6611
[fold05] Early stop at ep019 (best subj_bacc=0.6944)



########################################################################
FOLD-WISE RESULTS:
[fold01] WIN  acc=0.8210 bacc=0.8236 prec=0.8948 rec=0.8139 f1=0.8525 auc=0.8358
         SUBJ acc=0.8333 bacc=0.8500 prec=0.9412 rec=0.8000 f1=0.8649 auc=0.8400
[fold02] WIN  acc=0.6218 bacc=0.6626 prec=0.8283 rec=0.5232 f1=0.6414 auc=0.7247
         SUBJ acc=0.6667 bacc=0.7500 prec=1.0000 rec=0.5000 f1=0.6667 auc=0.7650
[fold03] WIN  acc=0.8144 bacc=0.7868 prec=0.8422 rec=0.8790 f1=0.8602 auc=0.8707
         SUBJ acc=0.8667 bacc=0.8500 prec=0.9000 rec=0.9000 f1=0.9000 auc=0.8950
[fold04] WIN  acc=0.6837 bacc=0.6802 prec=0.8119 rec=0.6902 f1=0.7461 auc=0.7304
         SUBJ acc=0.7000 bacc=0.7000 prec=0.8235 rec=0.7000 f1=0.7568 auc=0.7350
[fold05] WIN  acc=0.6271 bacc=0.6539 prec=0.8027 rec=0.5637 f1=0.6623 auc=0.7296
         SUBJ acc=0.6207 bacc=0.6944 prec=0.9091 rec=0.5000 f1=0.6452 auc=0.7611

########################################################################
OVERALL (mean ± std acr

In [14]:
#!/usr/bin/env python3
# ============================================================
# Parkinson's Resting-State EEG — Subject-wise Stratified 5-Fold CV
# Protocol:
# ✅ Subject-wise Stratified 5-fold CV (StratifiedGroupKFold)
# ✅ FULL variable-length recordings (no truncation)
# ✅ Canonical EEG channel intersection (EEG-only)
# ✅ Windowing: 10s, 50% overlap across full length
# ✅ Metrics per fold + overall:
#    Window: Acc/BAcc/Prec/Rec/F1/AUC
#    Subject: Acc/BAcc/Prec/Rec/F1/AUC
#
# New architecture (<=250k params): NeuroConvFormer-Lite
#
# UPGRADE (requested): MIL subject-level training
# - Bag sampling (N windows/subject per step)
# - Attention pooling over windows -> subject logits
# - Focal / weighted CE for stability
# - Confidence-trimmed inference for subject aggregation
#
# NEW CHANGES (requested):
# (1) Per-fold subject threshold tuning (train-only) for subject BAcc
# (2) Use MILPool at inference for subject prediction
# (3) Logit-mean aggregation (implemented as attention-weighted mean of per-window log-odds)
# (4) Balanced bag sampler (WeightedRandomSampler) to lift recall
# ============================================================

import os, re, glob, math, json, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, roc_auc_score
)

import mne

# -----------------------
# USER PATHS
# -----------------------
ROOT_DIR  = "/home/varun/Desktop/Parkinsons_Resting_State/Cleaned dataset"
label_dir = "/home/varun/Desktop/Parkinsons_Resting_State"

# -----------------------
# GLOBALS
# -----------------------
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# ============================================================
# 1) LABEL LOADING (robust)
# ============================================================
def find_label_file(label_dir):
    cands = []
    for fn in ["participants.tsv", "labels.csv", "labels.tsv", "participants.csv"]:
        p = os.path.join(label_dir, fn)
        if os.path.exists(p):
            cands.append(p)
    if len(cands) > 0:
        return cands[0]
    # fallback: any csv/tsv with "label" or "group"
    for ext in ["*.csv", "*.tsv"]:
        for p in glob.glob(os.path.join(label_dir, ext)):
            cands.append(p)
    return cands[0] if len(cands) else None

def load_labels(label_dir):
    """
    Expected columns (any of these patterns):
      - participant_id / subject / sub / id
      - label / group / diagnosis / class
    Binary mapping is inferred if text labels exist.
    """
    lf = find_label_file(label_dir)
    if lf is None:
        raise FileNotFoundError(
            f"No label file found in {label_dir}. "
            "Place participants.tsv or labels.csv/tsv there."
        )

    sep = "\t" if lf.endswith(".tsv") else ","
    df = pd.read_csv(lf, sep=sep)

    # normalize colnames
    cols = {c.lower().strip(): c for c in df.columns}
    def pick_col(possibles):
        for k in possibles:
            for c_low, c_orig in cols.items():
                if c_low == k or k in c_low:
                    return c_orig
        return None

    id_col = pick_col(["participant_id", "subject", "sub", "id"])
    y_col  = pick_col(["label", "group", "diagnosis", "class"])

    if id_col is None or y_col is None:
        raise ValueError(
            f"Could not infer id/label columns from {lf}. "
            f"Columns present: {list(df.columns)}"
        )

    df = df[[id_col, y_col]].copy()
    df.rename(columns={id_col: "subject_id", y_col: "label_raw"}, inplace=True)

    # clean subject ids
    df["subject_id"] = df["subject_id"].astype(str).str.strip()

    # map labels to 0/1
    raw = df["label_raw"]
    if raw.dtype == object:
        # heuristics for PD vs HC
        low = raw.astype(str).str.lower()
        # PD positive class = 1
        pd_mask = low.str.contains("pd|parkinson|patient|case|disease", regex=True)
        hc_mask = low.str.contains("hc|control|healthy|normal", regex=True)

        # if both ambiguous, fallback to factorize
        if pd_mask.any() and (hc_mask.any() or (~pd_mask).any()):
            y = np.where(pd_mask, 1, 0).astype(int)
        else:
            y = pd.factorize(raw)[0].astype(int)
            # ensure binary
            if len(np.unique(y)) != 2:
                raise ValueError(f"Non-binary labels detected: {raw.unique()}")
    else:
        y = raw.values.astype(int)
        if len(np.unique(y)) != 2:
            raise ValueError(f"Non-binary labels detected: {np.unique(y)}")

    df["y"] = y
    return df[["subject_id", "y"]]

labels_df = load_labels(label_dir)

# ============================================================
# 2) DATA DISCOVERY + SUBJECT ID PARSING
# ============================================================
SUPPORTED_EXTS = [".set", ".edf", ".bdf", ".fif", ".vhdr"]

def discover_eeg_files(root_dir):
    files = []
    for ext in SUPPORTED_EXTS:
        files.extend(glob.glob(os.path.join(root_dir, f"**/*{ext}"), recursive=True))
    files = sorted(list(set(files)))
    if len(files) == 0:
        raise FileNotFoundError(
            f"No EEG files found in {root_dir} with extensions {SUPPORTED_EXTS}"
        )
    return files

def infer_subject_id_from_path(path):
    """
    Tries common patterns: sub-XXX, subjectXXX, or folder name match.
    """
    s = path.replace("\\", "/")
    m = re.search(r"(sub-[a-zA-Z0-9]+)", s)
    if m:
        return m.group(1)
    m = re.search(r"(subject[_-]?[a-zA-Z0-9]+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1)
    # fallback: parent folder
    return os.path.basename(os.path.dirname(path))

files = discover_eeg_files(ROOT_DIR)

# align files with labels
file_rows = []
label_map = dict(zip(labels_df["subject_id"], labels_df["y"]))

# allow minor normalization of subject keys
def normalize_sid(sid):
    sid = str(sid).strip()
    sid = sid.replace(" ", "")
    return sid

label_map_norm = {normalize_sid(k): v for k, v in label_map.items()}

for fp in files:
    sid = infer_subject_id_from_path(fp)
    y = label_map_norm.get(normalize_sid(sid), None)
    if y is None:
        # sometimes labels use just numeric id, try extracting trailing digits
        digits = re.findall(r"\d+", sid)
        if len(digits):
            y = label_map_norm.get(normalize_sid(digits[-1]), None)
    if y is None:
        continue
    file_rows.append((fp, sid, int(y)))

if len(file_rows) == 0:
    raise RuntimeError(
        "No files matched with labels. Check subject_id mapping between "
        "label file and file naming."
    )

meta_df = pd.DataFrame(file_rows, columns=["path", "subject_id", "y"])
print(f"[INFO] Matched recordings: {len(meta_df)} | Subjects: {meta_df['subject_id'].nunique()}")

# ============================================================
# 3) EEG LOADING (EEG-only) + CANONICAL INTERSECTION
# ============================================================
def load_raw(fp):
    ext = os.path.splitext(fp)[1].lower()
    if ext == ".set":
        raw = mne.io.read_raw_eeglab(fp, preload=True, verbose=False)
    elif ext == ".edf":
        raw = mne.io.read_raw_edf(fp, preload=True, verbose=False)
    elif ext == ".bdf":
        raw = mne.io.read_raw_bdf(fp, preload=True, verbose=False)
    elif ext == ".fif":
        raw = mne.io.read_raw_fif(fp, preload=True, verbose=False)
    elif ext == ".vhdr":
        raw = mne.io.read_raw_brainvision(fp, preload=True, verbose=False)
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    return raw

def eeg_only_pick(raw):
    picks = mne.pick_types(raw.info, eeg=True, eog=False, ecg=False, emg=False, stim=False, misc=False)
    if len(picks) == 0:
        raise RuntimeError("No EEG channels found in file.")
    return picks

# compute canonical EEG channel intersection
all_ch_sets = []
for fp in tqdm(meta_df["path"].tolist(), desc="Scanning channels"):
    raw = load_raw(fp)
    picks = eeg_only_pick(raw)
    chs = [raw.ch_names[i] for i in picks]
    # normalize channel names
    chs = [c.strip().upper() for c in chs]
    all_ch_sets.append(set(chs))

canon = set.intersection(*all_ch_sets)
canon = sorted(list(canon))
if len(canon) < 4:
    raise RuntimeError(f"Canonical EEG intersection too small: {len(canon)} channels: {canon}")

print(f"[INFO] Canonical EEG channels (intersection): {len(canon)}")

# ============================================================
# 4) WINDOWING (10s, 50% overlap) OVER FULL DURATION
# ============================================================
WIN_SEC = 10.0
OVERLAP = 0.50

def extract_windows_from_raw(raw, canon_chs, win_sec=WIN_SEC, overlap=OVERLAP):
    """
    Returns: X_windows (nW, C, T), sfreq
    Uses the full duration; drops the last partial window.
    """
    sfreq = float(raw.info["sfreq"])
    # pick canonical channels (case-insensitive)
    name_to_idx = {c.strip().upper(): i for i, c in enumerate(raw.ch_names)}
    picks = [name_to_idx[c] for c in canon_chs if c in name_to_idx]
    if len(picks) != len(canon_chs):
        # should not happen if intersection computed correctly, but guard anyway
        raise RuntimeError("Canonical channel mismatch during extraction.")

    data = raw.get_data(picks=picks)  # (C, N)
    C, N = data.shape

    win_len = int(round(win_sec * sfreq))
    hop = int(round(win_len * (1.0 - overlap)))
    hop = max(1, hop)

    if N < win_len:
        return np.zeros((0, C, win_len), dtype=np.float32), sfreq

    starts = np.arange(0, N - win_len + 1, hop, dtype=int)
    X = np.stack([data[:, s:s+win_len] for s in starts], axis=0).astype(np.float32)  # (W,C,T)
    return X, sfreq

# ============================================================
# 5) AUGMENTATIONS (window-level, safe + effective)
# ============================================================
class Augment:
    def __init__(self, p_noise=0.3, p_shift=0.3, p_chdrop=0.2, p_banddrop=0.2,
                 noise_std=0.01, max_shift_frac=0.1, chdrop_frac=0.1):
        self.p_noise = p_noise
        self.p_shift = p_shift
        self.p_chdrop = p_chdrop
        self.p_banddrop = p_banddrop
        self.noise_std = noise_std
        self.max_shift_frac = max_shift_frac
        self.chdrop_frac = chdrop_frac

    def __call__(self, x):
        # x: (C,T) torch
        C, T = x.shape

        if random.random() < self.p_shift:
            max_shift = int(self.max_shift_frac * T)
            if max_shift > 0:
                s = random.randint(-max_shift, max_shift)
                x = torch.roll(x, shifts=s, dims=1)

        if random.random() < self.p_noise:
            x = x + self.noise_std * torch.randn_like(x)

        if random.random() < self.p_chdrop:
            k = max(1, int(self.chdrop_frac * C))
            idx = torch.randperm(C)[:k]
            x[idx] = 0.0

        # simple "band dropout" proxy via random temporal smoothing / differencing
        # (cheap, stable, helps robustness)
        if random.random() < self.p_banddrop:
            if random.random() < 0.5:
                # smooth
                k = random.choice([3,5,7])
                pad = k//2
                x = F.avg_pool1d(x.unsqueeze(0), kernel_size=k, stride=1, padding=pad).squeeze(0)
            else:
                # high-pass-ish
                x = x - F.avg_pool1d(x.unsqueeze(0), kernel_size=7, stride=1, padding=3).squeeze(0)

        return x

# ============================================================
# 6) DATASET (cache windows per recording)
# ============================================================
class WindowDataset(torch.utils.data.Dataset):
    def __init__(self, records, canon_chs, training=False, augment=None):
        """
        records: list of dict {path, subject_id, y}
        Creates a flat list of windows with subject_id for later subject aggregation.
        """
        self.training = training
        self.augment = augment

        self.items = []  # each: (window_np, y, subject_id)
        self.sfreqs = []
        for r in tqdm(records, desc="Loading & windowing"):
            raw = load_raw(r["path"])
            Xw, sf = extract_windows_from_raw(raw, canon_chs)
            self.sfreqs.append(sf)
            if Xw.shape[0] == 0:
                continue
            for i in range(Xw.shape[0]):
                self.items.append((Xw[i], r["y"], r["subject_id"]))

        if len(self.items) == 0:
            raise RuntimeError("No windows extracted. Check data durations and sfreq.")

        # sanity: ensure consistent sampling rate
        if np.std(self.sfreqs) > 1e-6:
            print("[WARN] Sampling rate varies across recordings. Model still works (fixed window samples per file).")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        x, y, sid = self.items[idx]
        x = torch.from_numpy(x)  # (C,T)
        # per-window robust normalization (channelwise z-score)
        x = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)

        if self.training and self.augment is not None:
            x = self.augment(x)

        return x, torch.tensor(y, dtype=torch.long), sid

class SubjectBagDataset(torch.utils.data.Dataset):
    def __init__(self, records, canon_chs, bag_size=16, training=False, augment=None):
        """
        records: list of dict {path, subject_id, y}
        Returns per-subject bags of windows: (bag_size, C, T), y, subject_id
        """
        self.training = training
        self.augment = augment
        self.bag_size = int(bag_size)

        # group paths by subject (one label per subject assumed)
        subj_map = {}
        for r in records:
            sid = r["subject_id"]
            if sid not in subj_map:
                subj_map[sid] = {"y": int(r["y"]), "paths": []}
            subj_map[sid]["paths"].append(r["path"])

        self.subject_ids = sorted(list(subj_map.keys()))
        self.subj_y = {sid: subj_map[sid]["y"] for sid in self.subject_ids}

        # cache windows per subject
        self.subj_windows = {}
        self.sfreqs = []
        for sid in tqdm(self.subject_ids, desc="Loading & windowing (MIL bags)"):
            all_w = []
            for fp in subj_map[sid]["paths"]:
                raw = load_raw(fp)
                Xw, sf = extract_windows_from_raw(raw, canon_chs)
                self.sfreqs.append(sf)
                if Xw.shape[0] > 0:
                    all_w.append(Xw)
            if len(all_w) == 0:
                continue
            Xcat = np.concatenate(all_w, axis=0).astype(np.float32)  # (W,C,T)
            self.subj_windows[sid] = Xcat

        # filter subjects with no windows
        self.subject_ids = [sid for sid in self.subject_ids if sid in self.subj_windows]
        if len(self.subject_ids) == 0:
            raise RuntimeError("No subject bags created. Check data durations and sfreq.")

        if np.std(self.sfreqs) > 1e-6:
            print("[WARN] Sampling rate varies across recordings. Model still works (fixed window samples per file).")

    def __len__(self):
        return len(self.subject_ids)

    def __getitem__(self, idx):
        sid = self.subject_ids[idx]
        y = self.subj_y[sid]
        X = self.subj_windows[sid]  # (W,C,T)
        W = X.shape[0]

        # sample bag_size windows (with replacement if needed)
        if W >= self.bag_size:
            sel = np.random.choice(W, size=self.bag_size, replace=False)
        else:
            sel = np.random.choice(W, size=self.bag_size, replace=True)

        Xb = X[sel]  # (BAG,C,T)
        xb = torch.from_numpy(Xb)  # float32

        # per-window normalization + optional augmentation
        # xb: (BAG,C,T)
        xb = (xb - xb.mean(dim=2, keepdim=True)) / (xb.std(dim=2, keepdim=True) + 1e-6)

        if self.training and self.augment is not None:
            # apply per-window
            out = []
            for i in range(xb.shape[0]):
                out.append(self.augment(xb[i]))
            xb = torch.stack(out, dim=0)

        return xb, torch.tensor(y, dtype=torch.long), sid

def make_records(df):
    recs = []
    for _, row in df.iterrows():
        recs.append({"path": row["path"], "subject_id": row["subject_id"], "y": int(row["y"])})
    return recs

# ============================================================
# 7) MODEL: NeuroConvFormer-Lite (<=250k params)
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hid = max(4, ch // r)
        self.fc1 = nn.Linear(ch, hid)
        self.fc2 = nn.Linear(hid, ch)

    def forward(self, x):
        # x: (B, C, T)
        s = x.mean(dim=2)              # (B,C)
        s = F.relu(self.fc1(s))
        s = torch.sigmoid(self.fc2(s)) # (B,C)
        return x * s.unsqueeze(-1)

class NeuroConvFormerLite(nn.Module):
    """
    Input: (B, C, T)
    Output: logits (B,2)

    Also exposes forward_features() to support MIL subject pooling.
    """
    def __init__(self, n_ch, n_time, d_model=64, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()

        ks = [7, 15, 31]
        self.dw_convs = nn.ModuleList([
            nn.Conv1d(n_ch, n_ch, kernel_size=k, padding=k//2, groups=n_ch, bias=False)
            for k in ks
        ])
        self.pw_mix = nn.Conv1d(n_ch * len(ks), d_model, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm1d(d_model)

        self.se = SEBlock(d_model, r=8)

        self.ds = nn.Conv1d(d_model, d_model, kernel_size=5, stride=2, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=2*d_model,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True
        )
        self.tr = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.GELU(),
            nn.Linear(d_model//2, 1)
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

        self._n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward_features(self, x):
        # x: (B,C,T) -> z: (B, d_model)
        feats = []
        for dw in self.dw_convs:
            feats.append(dw(x))
        x = torch.cat(feats, dim=1)          # (B, C*3, T)
        x = self.pw_mix(x)                   # (B, d_model, T)
        x = self.bn1(x)
        x = F.gelu(x)

        x = self.se(x)

        x = self.ds(x)                       # (B, d_model, T/2)
        x = self.bn2(x)
        x = F.gelu(x)

        x = x.transpose(1, 2)                # (B, S, d_model)
        x = self.tr(x)

        w = self.attn(x)                     # (B,S,1)
        w = torch.softmax(w, dim=1)
        z = (x * w).sum(dim=1)               # (B,d_model)
        return z

    def forward(self, x):
        z = self.forward_features(x)
        logits = self.head(z)
        return logits

class MILPool(nn.Module):
    """
    Attention pooling across windows in a subject bag.
    Input:  (B, W, D)
    Output: (B, D)
    """
    def __init__(self, d_model=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.GELU(),
            nn.Linear(d_model//2, 1)
        )

    def forward(self, z, return_attn=False):
        a = self.net(z)          # (B,W,1)
        a = torch.softmax(a, dim=1)
        z_subj = (z * a).sum(dim=1)
        if return_attn:
            return z_subj, a
        return z_subj

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = float(gamma)

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()

# ============================================================
# 8) METRICS (window + subject)
# ============================================================
def binary_metrics(y_true, y_prob, thr=0.5):
    """
    y_true: (N,)
    y_prob: (N,) probability for class=1
    """
    y_pred = (y_prob >= thr).astype(int)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    auc = np.nan
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_prob)

    return dict(acc=acc, bacc=bacc, prec=prec, rec=rec, f1=f1, auc=auc)

@torch.no_grad()
def _collect_window_outputs(model, loader):
    """
    Collects window-level:
      - y (N,)
      - p (N,) probability for class1 from window logits
      - logodds (N,) = logits1 - logits0
      - z (N,D) window embeddings from forward_features
      - sid list length N
    """
    model.eval()
    all_y = []
    all_p = []
    all_sid = []
    all_z = []
    all_logodds = []

    for xb, yb, sid in tqdm(loader, desc="Eval", leave=False):
        xb = xb.to(DEVICE, non_blocking=True)
        yb_np = yb.numpy().astype(int)

        z = model.forward_features(xb)                 # (B,D)
        logits = model.head(z)                         # (B,2)
        prob1 = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

        # log-odds for binary = logits1 - logits0
        lo = (logits[:, 1] - logits[:, 0]).detach().cpu().numpy()

        all_y.append(yb_np)
        all_p.append(prob1)
        all_logodds.append(lo)
        all_z.append(z.detach().cpu().numpy())
        all_sid.extend(list(sid))

    y = np.concatenate(all_y, axis=0)
    p = np.concatenate(all_p, axis=0)
    logodds = np.concatenate(all_logodds, axis=0)
    z = np.concatenate(all_z, axis=0)
    return y, p, logodds, z, all_sid

@torch.no_grad()
def evaluate(model, pool, loader, keep_frac=0.60, thr=0.5):
    """
    keep_frac: confidence trimming for subject aggregation (top fraction by confidence).
              keep_frac=1.0 uses all windows.
    thr: classification threshold used for both WIN and SUBJ metrics.
    """
    y, p, logodds, z, sids = _collect_window_outputs(model, loader)

    # window-level metrics (exactly as before, just threshold is configurable)
    win = binary_metrics(y, p, thr=thr)

    # subject-level (use MILPool at inference) + logit-mean aggregation
    # Implemented as: attention-weighted mean of per-window log-odds (logit domain),
    # optionally confidence-trimmed first.
    eps = 1e-6
    df = pd.DataFrame({"sid": sids, "y": y, "p": p})
    subj_rows = []

    # To compute attention weights, we need per-subject window embeddings as torch tensors.
    # We'll rebuild per-subject arrays from z/logodds using indices.
    sid_to_indices = {}
    for i, sid in enumerate(sids):
        if sid not in sid_to_indices:
            sid_to_indices[sid] = []
        sid_to_indices[sid].append(i)

    pool.eval()

    for sid, idxs in sid_to_indices.items():
        yt = int(df[df["sid"] == sid]["y"].iloc[0])

        pp = np.clip(p[idxs], eps, 1 - eps)
        conf = np.abs(pp - 0.5)
        k = max(1, int(round(float(keep_frac) * len(pp))))
        sel_local = np.argsort(-conf)[:k]
        sel = np.array(idxs, dtype=int)[sel_local]

        z_sel = torch.from_numpy(z[sel]).to(DEVICE)              # (W,D)
        lo_sel = torch.from_numpy(logodds[sel]).to(DEVICE)       # (W,)

        z_sel = z_sel.unsqueeze(0)                               # (1,W,D)
        _, a = pool(z_sel, return_attn=True)                     # a: (1,W,1)
        a = a.squeeze(0).squeeze(-1)                             # (W,)

        # attention-weighted mean of log-odds (logit-mean)
        lo_agg = torch.sum(a * lo_sel)                           # scalar
        p_agg = torch.sigmoid(lo_agg).item()

        subj_rows.append((sid, yt, float(p_agg)))

    _, ys, ps = zip(*subj_rows)
    subj = binary_metrics(np.array(ys), np.array(ps), thr=thr)

    return win, subj

def tune_threshold_subject_bacc(model, pool, loader, keep_frac=0.60):
    """
    Train-only threshold tuning:
      - compute subject probabilities using MILPool inference (same as evaluate)
      - scan thresholds and pick the one maximizing subject BAcc
    Returns: best_thr, dict(best_metrics)
    """
    y, p, logodds, z, sids = _collect_window_outputs(model, loader)

    eps = 1e-6
    df = pd.DataFrame({"sid": sids, "y": y, "p": p})

    sid_to_indices = {}
    for i, sid in enumerate(sids):
        if sid not in sid_to_indices:
            sid_to_indices[sid] = []
        sid_to_indices[sid].append(i)

    pool.eval()

    subj_true = []
    subj_prob = []

    for sid, idxs in sid_to_indices.items():
        yt = int(df[df["sid"] == sid]["y"].iloc[0])

        pp = np.clip(p[idxs], eps, 1 - eps)
        conf = np.abs(pp - 0.5)
        k = max(1, int(round(float(keep_frac) * len(pp))))
        sel_local = np.argsort(-conf)[:k]
        sel = np.array(idxs, dtype=int)[sel_local]

        z_sel = torch.from_numpy(z[sel]).to(DEVICE)              # (W,D)
        lo_sel = torch.from_numpy(logodds[sel]).to(DEVICE)       # (W,)

        z_sel = z_sel.unsqueeze(0)                               # (1,W,D)
        _, a = pool(z_sel, return_attn=True)                     # a: (1,W,1)
        a = a.squeeze(0).squeeze(-1)                             # (W,)

        lo_agg = torch.sum(a * lo_sel)                           # scalar
        p_agg = torch.sigmoid(lo_agg).item()

        subj_true.append(yt)
        subj_prob.append(float(p_agg))

    subj_true = np.array(subj_true, dtype=int)
    subj_prob = np.array(subj_prob, dtype=float)

    best_thr = 0.5
    best_m = None
    best_bacc = -1.0

    # dense scan
    for thr in np.linspace(0.05, 0.95, 181):
        m = binary_metrics(subj_true, subj_prob, thr=float(thr))
        if m["bacc"] > best_bacc + 1e-12:
            best_bacc = m["bacc"]
            best_thr = float(thr)
            best_m = m

    return best_thr, best_m

# ============================================================
# 9) TRAINING
# ============================================================
def train_one_fold(fold, train_df, val_df, canon_chs,
                   epochs=60, batch_size=64, lr=2e-3, weight_decay=1e-3,
                   patience=12):

    aug = Augment(
        p_noise=0.35, p_shift=0.35, p_chdrop=0.20, p_banddrop=0.25,
        noise_std=0.015, max_shift_frac=0.08, chdrop_frac=0.10
    )

    # window datasets (for eval metrics exactly as before)
    train_set_win = WindowDataset(make_records(train_df), canon_chs, training=True, augment=aug)
    val_set       = WindowDataset(make_records(val_df), canon_chs, training=False, augment=None)

    # train-only calibration loader for threshold tuning (no augmentation)
    train_set_cal = WindowDataset(make_records(train_df), canon_chs, training=False, augment=None)

    # MIL subject bag dataset (for training objective aligned to subject metrics)
    # NOTE: batch_size in this MIL loader is "subjects per batch"
    BAG_SIZE = 16
    train_set_bag = SubjectBagDataset(make_records(train_df), canon_chs, bag_size=BAG_SIZE, training=True, augment=aug)

    # MIL loader: subjects per batch (reduce if CUDA OOM)
    mil_batch_size = max(1, min(16, batch_size // 4))

    # Balanced bag sampler (subject-balanced sampling)
    # Build per-subject weights aligned with train_set_bag.subject_ids order.
    subj_labels = np.array([train_set_bag.subj_y[sid] for sid in train_set_bag.subject_ids], dtype=int)
    n0 = int((subj_labels == 0).sum())
    n1 = int((subj_labels == 1).sum())
    w0 = 1.0 / max(1, n0)
    w1 = 1.0 / max(1, n1)
    sample_weights = torch.tensor([w1 if y == 1 else w0 for y in subj_labels], dtype=torch.double)
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_set_bag),
        replacement=True
    )

    train_loader_bag = torch.utils.data.DataLoader(
        train_set_bag, batch_size=mil_batch_size, sampler=sampler,
        num_workers=4, pin_memory=True, drop_last=True
    )

    # Eval loader stays window-level
    val_loader = torch.utils.data.DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True
    )

    train_loader_cal = torch.utils.data.DataLoader(
        train_set_cal, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True
    )

    # infer shapes
    x0, _, _ = train_set_win[0]
    n_ch, n_time = x0.shape

    model = NeuroConvFormerLite(n_ch=n_ch, n_time=n_time, d_model=64, n_heads=4, n_layers=2, dropout=0.25).to(DEVICE)
    pool = MILPool(d_model=64).to(DEVICE)

    n_params = sum(p.numel() for p in list(model.parameters()) + list(pool.parameters()) if p.requires_grad)
    print(f"[fold{fold:02d}] Model params: {n_params}")
    if n_params > 250_000:
        raise RuntimeError(f"Param budget exceeded: {n_params} > 250k")

    # class balance (use subject labels for weighting)
    subj_train = train_df.groupby("subject_id").agg(y=("y", "first")).reset_index()
    y_train = subj_train["y"].values.astype(int)
    n0_ce = (y_train == 0).sum()
    n1_ce = (y_train == 1).sum()
    pos_weight = torch.tensor([max(1.0, n0_ce / max(1, n1_ce))], device=DEVICE)

    # class weights for CE/Focal (normalized)
    w0_ce = (n0_ce + n1_ce) / max(1, 2 * n0_ce)
    w1_ce = (n0_ce + n1_ce) / max(1, 2 * n1_ce)
    class_w = torch.tensor([w0_ce, w1_ce], dtype=torch.float32, device=DEVICE)

    # Focal loss (weighted)
    crit = FocalLoss(weight=class_w, gamma=2.0)

    opt = torch.optim.AdamW(
        list(model.parameters()) + list(pool.parameters()),
        lr=lr, weight_decay=weight_decay
    )

    total_steps = epochs * len(train_loader_bag)
    warmup_steps = max(1, int(0.1 * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_bacc = -1
    best_state = None
    best_state_pool = None
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        pool.train()
        losses = []

        pbar = tqdm(train_loader_bag, desc=f"[fold{fold:02d}] MIL Train ep{ep:03d}", leave=False)
        for xb_bag, yb, _ in pbar:
            # xb_bag: (Bsubj, BAG, C, T)
            xb_bag = xb_bag.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            Bsubj, BAG, C, T = xb_bag.shape
            xb_flat = xb_bag.view(Bsubj * BAG, C, T)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                z = model.forward_features(xb_flat)                 # (Bsubj*BAG, D)
                z = z.view(Bsubj, BAG, -1)                          # (Bsubj, BAG, D)
                z_subj = pool(z)                                    # (Bsubj, D)
                logits_subj = model.head(z_subj)                    # (Bsubj, 2)
                loss = crit(logits_subj, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(pool.parameters()), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()

            losses.append(loss.item())
            pbar.set_postfix(loss=float(np.mean(losses)), lr=opt.param_groups[0]["lr"])

        # eval each epoch (window + subject metrics) with default thr=0.5 (no tuning on VAL)
        win_m, subj_m = evaluate(model, pool, val_loader, keep_frac=0.60, thr=0.5)

        print(f"[fold{fold:02d}] ep{ep:03d} | "
              f"VAL(win) acc={win_m['acc']:.4f} bacc={win_m['bacc']:.4f} f1={win_m['f1']:.4f} auc={win_m['auc'] if not np.isnan(win_m['auc']) else -1:.4f} | "
              f"VAL(subj) acc={subj_m['acc']:.4f} bacc={subj_m['bacc']:.4f} f1={subj_m['f1']:.4f} auc={subj_m['auc'] if not np.isnan(subj_m['auc']) else -1:.4f}")

        # early stop on subject bacc (thr=0.5 only)
        if subj_m["bacc"] > best_bacc + 1e-4:
            best_bacc = subj_m["bacc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_state_pool = {k: v.detach().cpu().clone() for k, v in pool.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"[fold{fold:02d}] Early stop at ep{ep:03d} (best subj_bacc={best_bacc:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    if best_state_pool is not None:
        pool.load_state_dict(best_state_pool)

    # per-fold threshold tuning (train-only)
    best_thr, best_train_subj_m = tune_threshold_subject_bacc(model, pool, train_loader_cal, keep_frac=0.60)
    print(f"[fold{fold:02d}] Train-only tuned subject threshold: thr={best_thr:.3f} | "
          f"train_subj_bacc={best_train_subj_m['bacc']:.4f} f1={best_train_subj_m['f1']:.4f} "
          f"prec={best_train_subj_m['prec']:.4f} rec={best_train_subj_m['rec']:.4f}")

    # final fold eval (best checkpoint) using tuned threshold
    win_m, subj_m = evaluate(model, pool, val_loader, keep_frac=0.60, thr=best_thr)
    return model, win_m, subj_m

# ============================================================
# 10) SUBJECT-WISE STRATIFIED 5-FOLD CV
# ============================================================
subj_df = meta_df.groupby("subject_id").agg(
    y=("y", "first")
).reset_index()

X_dummy = np.zeros((len(subj_df), 1))
y_subj = subj_df["y"].values.astype(int)
groups = subj_df["subject_id"].values.astype(str)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_results = []
all_win_y = []
all_win_p = []
all_subj_y = []
all_subj_p = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_dummy, y_subj, groups), start=1):
    tr_sids = set(subj_df.iloc[tr_idx]["subject_id"].tolist())
    va_sids = set(subj_df.iloc[va_idx]["subject_id"].tolist())

    train_df = meta_df[meta_df["subject_id"].isin(tr_sids)].reset_index(drop=True)
    val_df   = meta_df[meta_df["subject_id"].isin(va_sids)].reset_index(drop=True)

    print("\n" + "="*70)
    print(f"[fold{fold:02d}] Train subjects: {len(tr_sids)} | Val subjects: {len(va_sids)}")
    print(f"[fold{fold:02d}] Train recs: {len(train_df)} | Val recs: {len(val_df)}")
    print("="*70)

    model, win_m, subj_m = train_one_fold(fold, train_df, val_df, canon)

    fold_results.append((win_m, subj_m))

def summarize(results, keyset=("acc","bacc","prec","rec","f1","auc")):
    arr = {k: [] for k in keyset}
    for m in results:
        for k in keyset:
            arr[k].append(m[k] if not (isinstance(m[k], float) and np.isnan(m[k])) else np.nan)
    out = {}
    for k, v in arr.items():
        vv = np.array(v, dtype=float)
        out[k] = (np.nanmean(vv), np.nanstd(vv))
    return out

win_folds  = [r[0] for r in fold_results]
subj_folds = [r[1] for r in fold_results]

win_sum  = summarize(win_folds)
subj_sum = summarize(subj_folds)

print("\n" + "#"*72)
print("FOLD-WISE RESULTS:")
for i, (w, s) in enumerate(fold_results, start=1):
    print(f"[fold{i:02d}] WIN  acc={w['acc']:.4f} bacc={w['bacc']:.4f} prec={w['prec']:.4f} rec={w['rec']:.4f} f1={w['f1']:.4f} auc={w['auc'] if not np.isnan(w['auc']) else -1:.4f}")
    print(f"         SUBJ acc={s['acc']:.4f} bacc={s['bacc']:.4f} prec={s['prec']:.4f} rec={s['rec']:.4f} f1={s['f1']:.4f} auc={s['auc'] if not np.isnan(s['auc']) else -1:.4f}")

print("\n" + "#"*72)
print("OVERALL (mean ± std across folds):")
print(f"WIN : acc={win_sum['acc'][0]:.4f}±{win_sum['acc'][1]:.4f} | "
      f"bacc={win_sum['bacc'][0]:.4f}±{win_sum['bacc'][1]:.4f} | "
      f"prec={win_sum['prec'][0]:.4f}±{win_sum['prec'][1]:.4f} | "
      f"rec={win_sum['rec'][0]:.4f}±{win_sum['rec'][1]:.4f} | "
      f"f1={win_sum['f1'][0]:.4f}±{win_sum['f1'][1]:.4f} | "
      f"auc={win_sum['auc'][0]:.4f}±{win_sum['auc'][1]:.4f}")
print(f"SUBJ: acc={subj_sum['acc'][0]:.4f}±{subj_sum['acc'][1]:.4f} | "
      f"bacc={subj_sum['bacc'][0]:.4f}±{subj_sum['bacc'][1]:.4f} | "
      f"prec={subj_sum['prec'][0]:.4f}±{subj_sum['prec'][1]:.4f} | "
      f"rec={subj_sum['rec'][0]:.4f}±{subj_sum['rec'][1]:.4f} | "
      f"f1={subj_sum['f1'][0]:.4f}±{subj_sum['f1'][1]:.4f} | "
      f"auc={subj_sum['auc'][0]:.4f}±{subj_sum['auc'][1]:.4f}")
print("#"*72)


[INFO] Matched recordings: 149 | Subjects: 149


Scanning channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 149/149 [00:02<00:00, 57.74it/s]


[INFO] Canonical EEG channels (intersection): 60

[fold01] Train subjects: 119 | Val subjects: 30
[fold01] Train recs: 119 | Val recs: 30


Loading & windowing (MIL bags): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 119/119 [00:03<00:00, 32.28it/s]


[fold01] Model params: 112120


...

[fold05] ep013 | VAL(win) acc=0.5537 bacc=0.5178 f1=0.6499 auc=0.5601 | VAL(subj) acc=0.5172 bacc=0.4667 f1=0.6316 auc=0.5111


[fold05] ep014 | VAL(win) acc=0.5733 bacc=0.5349 f1=0.6688 auc=0.5503 | VAL(subj) acc=0.5862 bacc=0.5167 f1=0.7000 auc=0.5500


[fold05] ep015 | VAL(win) acc=0.5723 bacc=0.5679 f1=0.6387 auc=0.6155 | VAL(subj) acc=0.5862 bacc=0.5778 f1=0.6667 auc=0.6167
[fold05] Early stop at ep015 (best subj_bacc=0.6139)


[fold05] Train-only tuned subject threshold: thr=0.470 | train_subj_bacc=0.7625 f1=0.8537 prec=0.8333 rec=0.8750



########################################################################
FOLD-WISE RESULTS:
[fold01] WIN  acc=0.7083 bacc=0.7382 prec=0.8783 rec=0.6278 f1=0.7323 auc=0.8079
         SUBJ acc=0.7667 bacc=0.8000 prec=0.9333 rec=0.7000 f1=0.8000 auc=0.8550
[fold02] WIN  acc=0.6607 bacc=0.5752 prec=0.6885 rec=0.8675 f1=0.7677 auc=0.7732
         SUBJ acc=0.6333 bacc=0.5500 prec=0.6957 rec=0.8000 f1=0.7442 auc=0.7750
[fold03] WIN  acc=0.7456 bacc=0.7282 prec=0.8153 rec=0.7866 f1=0.8007 auc=0.7159
         SUBJ acc=0.8000 bacc=0.7750 prec=0.8500 rec=0.8500 f1=0.8500 auc=0.7300
[fold04] WIN  acc=0.6043 bacc=0.6579 prec=0.8470 rec=0.5034 f1=0.6315 auc=0.7503
         SUBJ acc=0.6000 bacc=0.6500 prec=0.8333 rec=0.5000 f1=0.6250 auc=0.7450
[fold05] WIN  acc=0.6436 bacc=0.5608 prec=0.6835 rec=0.8392 f1=0.7534 auc=0.6683
         SUBJ acc=0.6207 bacc=0.5417 prec=0.7143 rec=0.7500 f1=0.7317 auc=0.6667

########################################################################
OVERALL (mean ± std acr